# Trying to measure the inner core radius: the Fiji experiment

Research notebook accompanying Andrei Akimov's article **"Trying to measure the inner core radius"** (October 6, 2026 revision).

**Question:** Do absolute PKIKP and PKiKP arrival times and their same-station differences favor the same inner-core radius? How strongly do these quantities respond to a change in radius?

The experiment uses the November 12, 2022 Fiji M7.0 earthquake, ISC event **625210364**. The ISC source is fixed at latitude -20.1146 degrees, longitude -178.2086 degrees, depth 601.2 km, and origin time 07:09:14.55 UTC. Trial radii span 1080-1300 km in 2 km steps, with the ak135 reference radius of 1217.5 km included.

PKIKP passes through the inner core; PKiKP reflects from its boundary. Subtracting their arrival times at one station removes origin time. It does not remove errors in source location, depth, phase identification, or Earth structure.

This notebook implements the article's corrected continuous-model Fiji scan and sensitivity analysis. These are catalog-pick fits, not waveform measurements or a test of temporal changes in the core.


In [ ]:
# Run this only if packages are missing, then restart the kernel.
# %pip install numpy matplotlib obspy ellipticipy pyproj


## 1. Model and search parameters

The source remains fixed while radius varies. There is no joint inversion for origin time, hypocenter, or core velocities. Radius grid spacing describes numerical sampling, not measurement precision.


In [ ]:
import csv
import hashlib
import json
import time
from collections import Counter, defaultdict
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import obspy
from ellipticipy import ellipticity_correction
from obspy import UTCDateTime
from obspy.taup import TauPyModel
from obspy.taup.taup_create import build_taup_model
from pyproj import Geod

EARTH_RADIUS_KM = 6371.0
CMB_DEPTH_KM = 2891.5
REFERENCE_ICB_DEPTH_KM = 5153.5
REFERENCE_RADIUS_KM = 1217.5
RESIDUAL_LIMIT_S = 1.0
PHASES = ("PKIKP", "PKiKP")
MODEL_METHOD = "ak135-core-coordinate-remap-v1"

RADII_KM = np.unique(np.r_[np.arange(1080.0, 1300.1, 2.0), REFERENCE_RADIUS_KM])
if len(RADII_KM) < 3 or np.any(np.diff(RADII_KM) <= 0):
    raise ValueError("Use at least three distinct, increasing radii.")

OUTPUT_DIR = Path.cwd() / "Fiji_2022_continuous_core_radius_numpy_v1"
MODEL_DIR = OUTPUT_DIR / "radius_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
AK135_PATH = Path(obspy.__file__).resolve().parent / "taup" / "data" / "ak135.tvel"

print(f"Output folder: {OUTPUT_DIR}")
print(f"Number of trial radii: {len(RADII_KM)}")


## 2. Export and display of research results


In [ ]:
def write_csv(path, rows, columns=None):
    """Write a list of row dictionaries to a UTF-8 CSV file.
    
    Parameters
    ----------
    path : str or Path
        Destination file; its parent directory must already exist.
    rows : iterable of dict
        One dictionary per CSV row. Keys are column names.
    columns : sequence of str or None
        Column names in output order. Supply this for an empty table so
        the file still has a header. None infers names from the rows.
    
    Returns
    -------
    None
        Writes the file. NaN and infinite numeric values become blank fields.
    """
    rows = list(rows)
    if columns is None:
        columns = list(dict.fromkeys(key for row in rows for key in row))

    def csv_value(value):
        """Convert one table value to its CSV representation.
        
        value : object
            Field value. Non-finite floating numbers are written as blanks.
        Returns the original value, or an empty string for a non-finite float.
        """
        if isinstance(value, (float, np.floating)) and not np.isfinite(value):
            return ""
        return value

    with Path(path).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow({key: csv_value(value) for key, value in row.items()})


def write_matrix(path, values, column_names):
    """Export a numeric array with radius as the first CSV column.
    
    Parameters
    ----------
    path : str or Path
        Destination CSV filename.
    values : numpy.ndarray, shape (number_of_radii, number_of_columns)
        Values to export. Rows must follow the global RADII_KM order.
        Units depend on the quantity, for example seconds or seconds/km.
    column_names : sequence of str
        Station or quantity names in the same order as array columns.
    
    Returns
    -------
    None
        Writes a CSV whose first column is radius_km.
    """
    rows = []
    for radius, values_at_radius in zip(RADII_KM, values):
        row = {"radius_km": float(radius)}
        row.update(zip(column_names, values_at_radius))
        rows.append(row)
    write_csv(path, rows, ["radius_km", *column_names])


def print_table(rows, columns=None):
    """Print dictionaries as an aligned text table.
    
    Parameters
    ----------
    rows : iterable of dict
        Records to display; each dictionary represents one table row.
    columns : sequence of str or None
        Column order. None uses the first row's keys.
    
    Returns
    -------
    None
        Prints values with up to six significant digits for floats.
    """
    rows = list(rows)
    if not rows:
        print("No rows.")
        return
    columns = columns or list(rows[0])
    text_rows = []
    for row in rows:
        values = []
        for key in columns:
            value = row.get(key, "")
            values.append(f"{value:.6g}" if isinstance(value, (float, np.floating)) else str(value))
        text_rows.append(values)
    widths = [max(len(key), *(len(row[i]) for row in text_rows)) for i, key in enumerate(columns)]
    print("  ".join(key.ljust(width) for key, width in zip(columns, widths)))
    for row in text_rows:
        print("  ".join(value.ljust(width) for value, width in zip(row, widths)))


def numeric_or_nan(value):
    """Convert a scalar to float without failing on missing data.
    
    value : object
        A number, numeric string, None, or invalid text.
    Returns a float; None and unparseable values become numpy.nan.
    """
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")


## 3. ISC catalog observations

The article identifies the input as ISC Bulletin event 625210364, downloaded October 5, 2026. Reports retain the ISC phase label, agency-reported label, station coordinates, arrival time, and catalog residual for audit.


In [ ]:
RAW_RECORDS = [
    {
        "station": "KLR",
        "network": "IR",
        "latitude": 49.23,
        "longitude": 131.75,
        "channel": "???",
        "isc_distance_deg": 82.1,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:25:50.01Z",
        "isc_residual_s": "0.5",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ILAR",
        "network": "IR",
        "latitude": 64.7714,
        "longitude": -146.8866,
        "channel": "???",
        "isc_distance_deg": 87.98,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:25:58.60Z",
        "isc_residual_s": "-0.1",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HILR",
        "network": "IR",
        "latitude": 49.5562,
        "longitude": 119.7508,
        "channel": "???",
        "isc_distance_deg": 88.42,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:00.75Z",
        "isc_residual_s": "0.8",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SONM",
        "network": "IR",
        "latitude": 47.8347,
        "longitude": 106.395,
        "channel": "???",
        "isc_distance_deg": 95.34,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:11.20Z",
        "isc_residual_s": "-0.8",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "YKA",
        "network": "IR",
        "latitude": 62.4932,
        "longitude": -114.6053,
        "channel": "???",
        "isc_distance_deg": 96.25,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:13.92Z",
        "isc_residual_s": "1.0",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PDYAR",
        "network": "IR",
        "latitude": 59.6553,
        "longitude": 112.4408,
        "channel": "???",
        "isc_distance_deg": 97.25,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:14.72Z",
        "isc_residual_s": "-0.1",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ULM",
        "network": "IR",
        "latitude": 50.2497,
        "longitude": -95.875,
        "channel": "???",
        "isc_distance_deg": 100.47,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:21.55Z",
        "isc_residual_s": "0.7",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LPAZ",
        "network": "IR",
        "latitude": -16.2879,
        "longitude": -68.1307,
        "channel": "???",
        "isc_distance_deg": 102.39,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:28.12Z",
        "isc_residual_s": "2.0",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CPUP",
        "network": "IR",
        "latitude": -26.3306,
        "longitude": -57.3292,
        "channel": "???",
        "isc_distance_deg": 106.38,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:32.25Z",
        "isc_residual_s": "-0.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HYB",
        "network": "IR",
        "latitude": 17.4169,
        "longitude": 78.5531,
        "channel": "B?Z",
        "isc_distance_deg": 107.89,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:35.53Z",
        "isc_residual_s": "0.0",
        "reporter": "HYB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SIV",
        "network": "IR",
        "latitude": -15.9913,
        "longitude": -61.0722,
        "channel": "???",
        "isc_distance_deg": 108.58,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:37.08Z",
        "isc_residual_s": "0.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ZSN",
        "network": "IR",
        "latitude": 47.45,
        "longitude": 84.866,
        "channel": "???",
        "isc_distance_deg": 109.15,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:36.74Z",
        "isc_residual_s": "-0.2",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ZALV",
        "network": "IR",
        "latitude": 53.9481,
        "longitude": 84.8188,
        "channel": "???",
        "isc_distance_deg": 110.07,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:38.05Z",
        "isc_residual_s": "-0.4",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NRIK",
        "network": "IR",
        "latitude": 69.3396,
        "longitude": 87.555,
        "channel": "???",
        "isc_distance_deg": 110.14,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:38.45Z",
        "isc_residual_s": "0.3",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NRIK",
        "network": "IR",
        "latitude": 69.3396,
        "longitude": 87.555,
        "channel": "B?Z",
        "isc_distance_deg": 110.14,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:37.30Z",
        "isc_residual_s": "-0.9",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MK31",
        "network": "IR",
        "latitude": 46.7937,
        "longitude": 82.2904,
        "channel": "B?Z",
        "isc_distance_deg": 110.79,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:39.60Z",
        "isc_residual_s": "-0.5",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MKAR",
        "network": "IR",
        "latitude": 46.7937,
        "longitude": 82.2904,
        "channel": "???",
        "isc_distance_deg": 110.79,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:39.55Z",
        "isc_residual_s": "-0.6",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MAKZ",
        "network": "IR",
        "latitude": 46.808,
        "longitude": 81.977,
        "channel": "S?Z",
        "isc_distance_deg": 111.01,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:39.30Z",
        "isc_residual_s": "-1.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PTGB",
        "network": "IR",
        "latitude": -24.7161,
        "longitude": -52.0648,
        "channel": "???",
        "isc_distance_deg": 111.22,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:41.08Z",
        "isc_residual_s": "-0.7",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TRCB",
        "network": "IR",
        "latitude": -22.7953,
        "longitude": -52.634,
        "channel": "???",
        "isc_distance_deg": 111.89,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:43.26Z",
        "isc_residual_s": "0.2",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SHLS",
        "network": "IR",
        "latitude": 43.1611,
        "longitude": 79.8852,
        "channel": "L?Z",
        "isc_distance_deg": 112.32,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:42.20Z",
        "isc_residual_s": "-1.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SHLS",
        "network": "IR",
        "latitude": 43.1611,
        "longitude": 79.8852,
        "channel": "???",
        "isc_distance_deg": 112.32,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:42.28Z",
        "isc_residual_s": "-1.0",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PDGK",
        "network": "IR",
        "latitude": 43.329,
        "longitude": 79.49,
        "channel": "S?Z",
        "isc_distance_deg": 112.33,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:41.70Z",
        "isc_residual_s": "-1.6",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UZB",
        "network": "IR",
        "latitude": 43.147,
        "longitude": 79.022,
        "channel": "M?Z",
        "isc_distance_deg": 112.64,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:43.40Z",
        "isc_residual_s": "-0.5",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UZB",
        "network": "IR",
        "latitude": 43.147,
        "longitude": 79.022,
        "channel": "???",
        "isc_distance_deg": 112.64,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:43.45Z",
        "isc_residual_s": "-0.5",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KPKS",
        "network": "IR",
        "latitude": 43.466,
        "longitude": 78.684,
        "channel": "M?Z",
        "isc_distance_deg": 112.92,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:43.90Z",
        "isc_residual_s": "-0.5",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KPKS",
        "network": "IR",
        "latitude": 43.466,
        "longitude": 78.684,
        "channel": "???",
        "isc_distance_deg": 112.92,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:43.99Z",
        "isc_residual_s": "-0.4",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SATY",
        "network": "IR",
        "latitude": 43.059,
        "longitude": 78.406,
        "channel": "M?Z",
        "isc_distance_deg": 113.08,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:44.20Z",
        "isc_residual_s": "-0.6",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SATY",
        "network": "IR",
        "latitude": 43.059,
        "longitude": 78.406,
        "channel": "???",
        "isc_distance_deg": 113.08,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:44.29Z",
        "isc_residual_s": "-0.5",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ALE",
        "network": "II",
        "latitude": 82.5033,
        "longitude": -62.35,
        "channel": "BHZ",
        "isc_distance_deg": 113.12,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:43.65Z",
        "isc_residual_s": "0.0",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SALV",
        "network": "IR",
        "latitude": -15.9012,
        "longitude": -55.6936,
        "channel": "???",
        "isc_distance_deg": 113.14,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:45.93Z",
        "isc_residual_s": "0.4",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TDK",
        "network": "IR",
        "latitude": 45.002,
        "longitude": 78.406,
        "channel": "L?Z",
        "isc_distance_deg": 113.28,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:44.60Z",
        "isc_residual_s": "-0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TDK",
        "network": "IR",
        "latitude": 45.002,
        "longitude": 78.406,
        "channel": "???",
        "isc_distance_deg": 113.28,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:44.63Z",
        "isc_residual_s": "-0.3",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MAS01",
        "network": "IR",
        "latitude": -14.8875,
        "longitude": -55.8171,
        "channel": "???",
        "isc_distance_deg": 113.54,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:46.91Z",
        "isc_residual_s": "0.6",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KURK",
        "network": "IR",
        "latitude": 50.7154,
        "longitude": 78.6202,
        "channel": "???",
        "isc_distance_deg": 113.58,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:45.02Z",
        "isc_residual_s": "-0.2",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KURK",
        "network": "IR",
        "latitude": 50.7154,
        "longitude": 78.6202,
        "channel": "B?Z",
        "isc_distance_deg": 113.58,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:44.10Z",
        "isc_residual_s": "-1.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PCMB",
        "network": "IR",
        "latitude": -21.6071,
        "longitude": -51.2614,
        "channel": "???",
        "isc_distance_deg": 113.61,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:46.50Z",
        "isc_residual_s": "0.2",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ARXS",
        "network": "IR",
        "latitude": 44.213,
        "longitude": 77.827,
        "channel": "B?Z",
        "isc_distance_deg": 113.62,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:44.60Z",
        "isc_residual_s": "-1.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KURBB",
        "network": "IR",
        "latitude": 50.6226,
        "longitude": 78.5304,
        "channel": "???",
        "isc_distance_deg": 113.63,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:44.50Z",
        "isc_residual_s": "-0.9",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "C2SB",
        "network": "IR",
        "latitude": -18.7688,
        "longitude": -52.8377,
        "channel": "???",
        "isc_distance_deg": 113.96,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:47.07Z",
        "isc_residual_s": "0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MDOK",
        "network": "IR",
        "latitude": 43.1631,
        "longitude": 77.0481,
        "channel": "B?Z",
        "isc_distance_deg": 114.08,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:46.00Z",
        "isc_residual_s": "-0.7",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MDOK",
        "network": "IR",
        "latitude": 43.1631,
        "longitude": 77.0481,
        "channel": "???",
        "isc_distance_deg": 114.08,
        "isc_phase": "PKPdf",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:26:46.05Z",
        "isc_residual_s": "-0.7",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TNSS",
        "network": "IR",
        "latitude": 43.037,
        "longitude": 76.947,
        "channel": "S?Z",
        "isc_distance_deg": 114.14,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:46.40Z",
        "isc_residual_s": "-0.8",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TNSS",
        "network": "IR",
        "latitude": 43.037,
        "longitude": 76.947,
        "channel": "???",
        "isc_distance_deg": 114.14,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:46.49Z",
        "isc_residual_s": "-0.7",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ULHL",
        "network": "IR",
        "latitude": 42.2456,
        "longitude": 76.2417,
        "channel": "???",
        "isc_distance_deg": 114.58,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:48.31Z",
        "isc_residual_s": "0.5",
        "reporter": "KNET",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KSH2",
        "network": "IR",
        "latitude": 39.1453,
        "longitude": 75.5662,
        "channel": "???",
        "isc_distance_deg": 114.78,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:48.90Z",
        "isc_residual_s": "0.7",
        "reporter": "BJI",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PET01",
        "network": "IR",
        "latitude": -24.2901,
        "longitude": -47.2753,
        "channel": "???",
        "isc_distance_deg": 114.96,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:49.14Z",
        "isc_residual_s": "0.3",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TKM2",
        "network": "IR",
        "latitude": 42.9208,
        "longitude": 75.5966,
        "channel": "???",
        "isc_distance_deg": 115.12,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:48.20Z",
        "isc_residual_s": "-0.6",
        "reporter": "KNET",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ITRB",
        "network": "IR",
        "latitude": -19.7042,
        "longitude": -50.359,
        "channel": "???",
        "isc_distance_deg": 115.38,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:49.64Z",
        "isc_residual_s": "-0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HRV",
        "network": "IR",
        "latitude": 42.5064,
        "longitude": -71.5583,
        "channel": "S?Z",
        "isc_distance_deg": 115.42,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:49.10Z",
        "isc_residual_s": "0.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CLDB",
        "network": "IR",
        "latitude": -10.8732,
        "longitude": -55.7965,
        "channel": "???",
        "isc_distance_deg": 115.51,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:50.21Z",
        "isc_residual_s": "0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KBK",
        "network": "IR",
        "latitude": 42.6564,
        "longitude": 74.9478,
        "channel": "???",
        "isc_distance_deg": 115.57,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:49.85Z",
        "isc_residual_s": "0.2",
        "reporter": "KNET",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CHMS",
        "network": "IR",
        "latitude": 42.9986,
        "longitude": 74.7513,
        "channel": "???",
        "isc_distance_deg": 115.74,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:49.26Z",
        "isc_residual_s": "-0.5",
        "reporter": "KNET",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AAK",
        "network": "IR",
        "latitude": 42.6375,
        "longitude": 74.4942,
        "channel": "???",
        "isc_distance_deg": 115.91,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:49.79Z",
        "isc_residual_s": "-0.5",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AAK",
        "network": "IR",
        "latitude": 42.6375,
        "longitude": 74.4942,
        "channel": "???",
        "isc_distance_deg": 115.91,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:49.84Z",
        "isc_residual_s": "-0.4",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AAK",
        "network": "IR",
        "latitude": 42.6375,
        "longitude": 74.4942,
        "channel": "B?Z",
        "isc_distance_deg": 115.91,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:49.70Z",
        "isc_residual_s": "-0.6",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AAK",
        "network": "IR",
        "latitude": 42.6375,
        "longitude": 74.4942,
        "channel": "???",
        "isc_distance_deg": 115.91,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:49.85Z",
        "isc_residual_s": "-0.4",
        "reporter": "KNET",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AAK",
        "network": "IR",
        "latitude": 42.6375,
        "longitude": 74.4942,
        "channel": "???",
        "isc_distance_deg": 115.91,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:49.79Z",
        "isc_residual_s": "-0.5",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RCLB",
        "network": "IR",
        "latitude": -22.4191,
        "longitude": -47.531,
        "channel": "???",
        "isc_distance_deg": 115.92,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:50.88Z",
        "isc_residual_s": "0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "USP",
        "network": "IR",
        "latitude": 43.2669,
        "longitude": 74.4997,
        "channel": "???",
        "isc_distance_deg": 115.95,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:49.28Z",
        "isc_residual_s": "-0.9",
        "reporter": "KNET",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NIL",
        "network": "IR",
        "latitude": 33.65,
        "longitude": 73.2517,
        "channel": "???",
        "isc_distance_deg": 115.96,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:50.78Z",
        "isc_residual_s": "0.3",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NIL",
        "network": "IR",
        "latitude": 33.65,
        "longitude": 73.2517,
        "channel": "S?Z",
        "isc_distance_deg": 115.96,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:50.70Z",
        "isc_residual_s": "0.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VABB",
        "network": "IR",
        "latitude": -23.0,
        "longitude": -46.97,
        "channel": "???",
        "isc_distance_deg": 115.98,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:51.06Z",
        "isc_residual_s": "0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BB19B",
        "network": "IR",
        "latitude": -21.0659,
        "longitude": -48.5276,
        "channel": "???",
        "isc_distance_deg": 115.99,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:52.10Z",
        "isc_residual_s": "1.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SJG",
        "network": "IR",
        "latitude": 18.1117,
        "longitude": -66.15,
        "channel": "S?Z",
        "isc_distance_deg": 116.19,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:51.20Z",
        "isc_residual_s": "-0.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BTLS",
        "network": "IR",
        "latitude": 45.041,
        "longitude": 74.046,
        "channel": "L?Z",
        "isc_distance_deg": 116.37,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:50.20Z",
        "isc_residual_s": "-0.6",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BTLS",
        "network": "IR",
        "latitude": 45.041,
        "longitude": 74.046,
        "channel": "???",
        "isc_distance_deg": 116.37,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:50.27Z",
        "isc_residual_s": "-0.6",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "EKS2",
        "network": "IR",
        "latitude": 42.6615,
        "longitude": 73.7772,
        "channel": "???",
        "isc_distance_deg": 116.44,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:51.10Z",
        "isc_residual_s": "-0.1",
        "reporter": "KNET",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FOMA",
        "network": "IR",
        "latitude": -24.9756,
        "longitude": 46.9788,
        "channel": "???",
        "isc_distance_deg": 117.23,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:52.94Z",
        "isc_residual_s": "-0.2",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KULLO",
        "network": "DK",
        "latitude": 74.5805,
        "longitude": -57.2201,
        "channel": "HHZ",
        "isc_distance_deg": 117.31,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:51.49Z",
        "isc_residual_s": "-0.3",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BRZS",
        "network": "IR",
        "latitude": 49.982,
        "longitude": 72.678,
        "channel": "M?Z",
        "isc_distance_deg": 117.37,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:51.50Z",
        "isc_residual_s": "-1.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ITU3",
        "network": "IR",
        "latitude": -18.22,
        "longitude": -48.87,
        "channel": "???",
        "isc_distance_deg": 117.39,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:53.27Z",
        "isc_residual_s": "-0.4",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OTUK",
        "network": "IR",
        "latitude": 48.2445,
        "longitude": 72.3378,
        "channel": "S?Z",
        "isc_distance_deg": 117.61,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:52.60Z",
        "isc_residual_s": "-0.6",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NOR",
        "network": "DK",
        "latitude": 81.6047,
        "longitude": -16.6609,
        "channel": "BHZ",
        "isc_distance_deg": 117.98,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:53.00Z",
        "isc_residual_s": "0.0",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "IPMB",
        "network": "IR",
        "latitude": -17.983,
        "longitude": -48.2117,
        "channel": "???",
        "isc_distance_deg": 118.03,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:54.73Z",
        "isc_residual_s": "-0.2",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CANS",
        "network": "IR",
        "latitude": -20.2896,
        "longitude": -46.3921,
        "channel": "???",
        "isc_distance_deg": 118.06,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:54.80Z",
        "isc_residual_s": "-0.2",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DZA",
        "network": "IR",
        "latitude": 42.892,
        "longitude": 71.331,
        "channel": "???",
        "isc_distance_deg": 118.25,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:54.14Z",
        "isc_residual_s": "-0.4",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SCHQ",
        "network": "IR",
        "latitude": 54.8319,
        "longitude": -66.8336,
        "channel": "???",
        "isc_distance_deg": 118.49,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:54.10Z",
        "isc_residual_s": "-0.5",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ITTB",
        "network": "IR",
        "latitude": -4.3672,
        "longitude": -55.7343,
        "channel": "???",
        "isc_distance_deg": 118.51,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:55.95Z",
        "isc_residual_s": "-0.0",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BVA0",
        "network": "IR",
        "latitude": 53.0249,
        "longitude": 70.3885,
        "channel": "S?Z",
        "isc_distance_deg": 118.66,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:54.60Z",
        "isc_residual_s": "-0.4",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BVAR",
        "network": "IR",
        "latitude": 53.0249,
        "longitude": 70.3885,
        "channel": "???",
        "isc_distance_deg": 118.66,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:55.05Z",
        "isc_residual_s": "-0.0",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BORK",
        "network": "IR",
        "latitude": 53.0461,
        "longitude": 70.3184,
        "channel": "B?Z",
        "isc_distance_deg": 118.7,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:54.30Z",
        "isc_residual_s": "-0.8",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BSCB",
        "network": "IR",
        "latitude": -20.9977,
        "longitude": -44.7737,
        "channel": "???",
        "isc_distance_deg": 118.81,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:56.38Z",
        "isc_residual_s": "-0.0",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KK31",
        "network": "IR",
        "latitude": 43.1034,
        "longitude": 70.5115,
        "channel": "S?Z",
        "isc_distance_deg": 118.85,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:54.90Z",
        "isc_residual_s": "-0.8",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PMNB",
        "network": "IR",
        "latitude": -18.5373,
        "longitude": -46.438,
        "channel": "???",
        "isc_distance_deg": 119.07,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:57.01Z",
        "isc_residual_s": "0.0",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BRLS",
        "network": "IR",
        "latitude": 43.03,
        "longitude": 69.836,
        "channel": "M?Z",
        "isc_distance_deg": 119.35,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:56.20Z",
        "isc_residual_s": "-0.5",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BRLS",
        "network": "IR",
        "latitude": 43.03,
        "longitude": 69.836,
        "channel": "???",
        "isc_distance_deg": 119.35,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:56.29Z",
        "isc_residual_s": "-0.4",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VOI",
        "network": "IR",
        "latitude": -21.9065,
        "longitude": 46.7933,
        "channel": "???",
        "isc_distance_deg": 119.36,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:56.67Z",
        "isc_residual_s": "-0.8",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CHM",
        "network": "IR",
        "latitude": 42.319,
        "longitude": 69.603,
        "channel": "L?Z",
        "isc_distance_deg": 119.51,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:56.70Z",
        "isc_residual_s": "-0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CHM",
        "network": "IR",
        "latitude": 42.319,
        "longitude": 69.603,
        "channel": "???",
        "isc_distance_deg": 119.51,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:56.73Z",
        "isc_residual_s": "-0.3",
        "reporter": "SOME",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BDFB",
        "network": "IR",
        "latitude": -15.6412,
        "longitude": -48.0141,
        "channel": "???",
        "isc_distance_deg": 119.52,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:57.83Z",
        "isc_residual_s": "-0.1",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KBL",
        "network": "IR",
        "latitude": 34.5408,
        "longitude": 69.0432,
        "channel": "???",
        "isc_distance_deg": 119.53,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:57.97Z",
        "isc_residual_s": "0.3",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KBL",
        "network": "IR",
        "latitude": 34.5408,
        "longitude": 69.0432,
        "channel": "S?Z",
        "isc_distance_deg": 119.53,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:57.90Z",
        "isc_residual_s": "0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SIM2",
        "network": "IR",
        "latitude": -21.9159,
        "longitude": -42.8069,
        "channel": "???",
        "isc_distance_deg": 119.64,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:57.84Z",
        "isc_residual_s": "-0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NUUG",
        "network": "DK",
        "latitude": 71.5384,
        "longitude": -53.1996,
        "channel": "BHZ",
        "isc_distance_deg": 119.72,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:56.51Z",
        "isc_residual_s": "0.0",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KARAT",
        "network": "DK",
        "latitude": 71.6495,
        "longitude": -52.5177,
        "channel": "HHZ",
        "isc_distance_deg": 119.86,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:56.07Z",
        "isc_residual_s": "-0.8",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "H05S1",
        "network": "IR",
        "latitude": 14.425,
        "longitude": -60.8367,
        "channel": "???",
        "isc_distance_deg": 120.22,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:57.65Z",
        "isc_residual_s": "-1.4",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "H05N1",
        "network": "IR",
        "latitude": 16.3164,
        "longitude": -61.0567,
        "channel": "???",
        "isc_distance_deg": 120.48,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:58.16Z",
        "isc_residual_s": "-1.4",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CAM01",
        "network": "IR",
        "latitude": -21.8257,
        "longitude": -41.6574,
        "channel": "???",
        "isc_distance_deg": 120.5,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:59.60Z",
        "isc_residual_s": "-0.0",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SAATT",
        "network": "DK",
        "latitude": 70.8113,
        "longitude": -51.6311,
        "channel": "HHZ",
        "isc_distance_deg": 120.52,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:57.55Z",
        "isc_residual_s": "-0.5",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GDH",
        "network": "DK",
        "latitude": 69.25,
        "longitude": -53.533,
        "channel": "HHZ",
        "isc_distance_deg": 120.66,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:59.47Z",
        "isc_residual_s": "1.0",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ABPO",
        "network": "IR",
        "latitude": -19.0174,
        "longitude": 47.2275,
        "channel": "S?Z",
        "isc_distance_deg": 120.87,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:01.70Z",
        "isc_residual_s": "1.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ABPO",
        "network": "IR",
        "latitude": -19.0174,
        "longitude": 47.2275,
        "channel": "B?Z",
        "isc_distance_deg": 120.87,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:59.14Z",
        "isc_residual_s": "-1.4",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KBS",
        "network": "IR",
        "latitude": 78.9256,
        "longitude": 11.9417,
        "channel": "B?Z",
        "isc_distance_deg": 120.95,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:26:58.90Z",
        "isc_residual_s": "0.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KBS",
        "network": "IU",
        "latitude": 78.9154,
        "longitude": 11.9385,
        "channel": "HHZ",
        "isc_distance_deg": 120.96,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:59.37Z",
        "isc_residual_s": "0.5",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KBS",
        "network": "GE",
        "latitude": 78.9256,
        "longitude": 11.9417,
        "channel": "BHZ",
        "isc_distance_deg": 120.96,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:59.18Z",
        "isc_residual_s": "0.3",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OPO",
        "network": "IR",
        "latitude": -18.5706,
        "longitude": 47.1879,
        "channel": "???",
        "isc_distance_deg": 121.18,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:01.44Z",
        "isc_residual_s": "0.1",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DIAM",
        "network": "IR",
        "latitude": -18.2966,
        "longitude": -43.6721,
        "channel": "H?Z",
        "isc_distance_deg": 121.3,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:00.93Z",
        "isc_residual_s": "-0.4",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DIAM",
        "network": "IR",
        "latitude": -18.2966,
        "longitude": -43.6721,
        "channel": "???",
        "isc_distance_deg": 121.3,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:01.44Z",
        "isc_residual_s": "-0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ILULI",
        "network": "DK",
        "latitude": 69.2121,
        "longitude": -51.1048,
        "channel": "HHZ",
        "isc_distance_deg": 121.45,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:59.62Z",
        "isc_residual_s": "-0.2",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SPA0",
        "network": "NO",
        "latitude": 78.1777,
        "longitude": 16.37,
        "channel": "HHZ",
        "isc_distance_deg": 121.47,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:26:59.04Z",
        "isc_residual_s": "-0.7",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SPITS",
        "network": "IR",
        "latitude": 78.1777,
        "longitude": 16.37,
        "channel": "???",
        "isc_distance_deg": 121.47,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:59.81Z",
        "isc_residual_s": "0.0",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "H09W1",
        "network": "IR",
        "latitude": -37.0972,
        "longitude": -12.335,
        "channel": "???",
        "isc_distance_deg": 121.56,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:59.96Z",
        "isc_residual_s": "-1.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "H09N1",
        "network": "IR",
        "latitude": -37.0681,
        "longitude": -12.3152,
        "channel": "???",
        "isc_distance_deg": 121.59,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:00.03Z",
        "isc_residual_s": "-1.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ISF",
        "network": "NS",
        "latitude": 78.0617,
        "longitude": 13.6198,
        "channel": "HHZ",
        "isc_distance_deg": 121.73,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:26:46.09Z",
        "isc_residual_s": "-14.1",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ALF01",
        "network": "IR",
        "latitude": -20.6169,
        "longitude": -40.7252,
        "channel": "H?Z",
        "isc_distance_deg": 121.95,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:01.88Z",
        "isc_residual_s": "-0.4",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ALF01",
        "network": "IR",
        "latitude": -20.6169,
        "longitude": -40.7252,
        "channel": "???",
        "isc_distance_deg": 121.95,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:02.06Z",
        "isc_residual_s": "-0.3",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MC01",
        "network": "IR",
        "latitude": -16.7074,
        "longitude": -43.9417,
        "channel": "???",
        "isc_distance_deg": 122.07,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:03.73Z",
        "isc_residual_s": "0.8",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MC01",
        "network": "IR",
        "latitude": -16.7074,
        "longitude": -43.9417,
        "channel": "???",
        "isc_distance_deg": 122.07,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:53.34Z",
        "isc_residual_s": "-9.4",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SBV",
        "network": "IR",
        "latitude": -13.4584,
        "longitude": 49.9212,
        "channel": "???",
        "isc_distance_deg": 122.08,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:02.63Z",
        "isc_residual_s": "-0.1",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SBV",
        "network": "IR",
        "latitude": -13.4584,
        "longitude": 49.9212,
        "channel": "B?Z",
        "isc_distance_deg": 122.08,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:02.44Z",
        "isc_residual_s": "-0.3",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MC08",
        "network": "IR",
        "latitude": -16.6793,
        "longitude": -43.9034,
        "channel": "???",
        "isc_distance_deg": 122.11,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:03.19Z",
        "isc_residual_s": "0.2",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MC08",
        "network": "IR",
        "latitude": -16.6793,
        "longitude": -43.9034,
        "channel": "???",
        "isc_distance_deg": 122.11,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:52.86Z",
        "isc_residual_s": "-10.0",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BMO2",
        "network": "IR",
        "latitude": -3.3474,
        "longitude": -51.9887,
        "channel": "???",
        "isc_distance_deg": 122.31,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:52.98Z",
        "isc_residual_s": "-10.2",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DAG",
        "network": "IR",
        "latitude": 76.7714,
        "longitude": -18.6547,
        "channel": "S?Z",
        "isc_distance_deg": 122.38,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:00.60Z",
        "isc_residual_s": "-0.9",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DAG",
        "network": "GE",
        "latitude": 76.7713,
        "longitude": -18.655,
        "channel": "BHZ",
        "isc_distance_deg": 122.38,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:00.00Z",
        "isc_residual_s": "-1.5",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HSPB",
        "network": "IR",
        "latitude": 77.0019,
        "longitude": 15.5332,
        "channel": "??Z",
        "isc_distance_deg": 122.66,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:00.81Z",
        "isc_residual_s": "-1.2",
        "reporter": "WAR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HSPB",
        "network": "IR",
        "latitude": 77.0019,
        "longitude": 15.5332,
        "channel": "S?Z",
        "isc_distance_deg": 122.66,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:00.80Z",
        "isc_residual_s": "-1.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SFJD",
        "network": "IR",
        "latitude": 66.996,
        "longitude": -50.6215,
        "channel": "S?Z",
        "isc_distance_deg": 122.66,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:01.70Z",
        "isc_residual_s": "-0.5",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SFJD",
        "network": "IR",
        "latitude": 66.996,
        "longitude": -50.6215,
        "channel": "???",
        "isc_distance_deg": 122.66,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:50.53Z",
        "isc_residual_s": "-11.7",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SFJD",
        "network": "IR",
        "latitude": 66.996,
        "longitude": -50.6215,
        "channel": "???",
        "isc_distance_deg": 122.66,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:01.45Z",
        "isc_residual_s": "-1.0",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SFJD",
        "network": "IU",
        "latitude": 66.9961,
        "longitude": -50.6208,
        "channel": "BHZ",
        "isc_distance_deg": 122.66,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:02.00Z",
        "isc_residual_s": "-0.3",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SUMG",
        "network": "GE",
        "latitude": 72.5742,
        "longitude": -38.4626,
        "channel": "BHZ",
        "isc_distance_deg": 122.83,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:03.00Z",
        "isc_residual_s": "0.0",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SJMB",
        "network": "IR",
        "latitude": -18.7032,
        "longitude": -41.1848,
        "channel": "H?Z",
        "isc_distance_deg": 122.87,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:03.61Z",
        "isc_residual_s": "-0.6",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SJMB",
        "network": "IR",
        "latitude": -18.7032,
        "longitude": -41.1848,
        "channel": "???",
        "isc_distance_deg": 122.87,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:04.12Z",
        "isc_residual_s": "-0.0",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SJMB",
        "network": "IR",
        "latitude": -18.7032,
        "longitude": -41.1848,
        "channel": "???",
        "isc_distance_deg": 122.87,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:26:50.83Z",
        "isc_residual_s": "-13.3",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PRPB",
        "network": "IR",
        "latitude": -6.1724,
        "longitude": -49.815,
        "channel": "H?Z",
        "isc_distance_deg": 122.94,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:04.38Z",
        "isc_residual_s": "-0.1",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PRPB",
        "network": "IR",
        "latitude": -6.1724,
        "longitude": -49.815,
        "channel": "???",
        "isc_distance_deg": 122.94,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:04.84Z",
        "isc_residual_s": "0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SMTB",
        "network": "IR",
        "latitude": -8.8617,
        "longitude": -47.5887,
        "channel": "H?Z",
        "isc_distance_deg": 123.52,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:05.72Z",
        "isc_residual_s": "-0.1",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SMTB",
        "network": "IR",
        "latitude": -8.8617,
        "longitude": -47.5887,
        "channel": "???",
        "isc_distance_deg": 123.52,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:06.13Z",
        "isc_residual_s": "0.3",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NUUK",
        "network": "DK",
        "latitude": 64.1838,
        "longitude": -51.6679,
        "channel": "HHZ",
        "isc_distance_deg": 123.53,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:04.80Z",
        "isc_residual_s": "0.5",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SVE",
        "network": "IR",
        "latitude": 56.827,
        "longitude": 60.637,
        "channel": "B?Z",
        "isc_distance_deg": 123.56,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:04.20Z",
        "isc_residual_s": "-0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SDBA",
        "network": "IR",
        "latitude": -12.4085,
        "longitude": -44.903,
        "channel": "H?Z",
        "isc_distance_deg": 123.83,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:06.29Z",
        "isc_residual_s": "0.1",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SDBA",
        "network": "IR",
        "latitude": -12.4085,
        "longitude": -44.903,
        "channel": "???",
        "isc_distance_deg": 123.83,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:06.22Z",
        "isc_residual_s": "0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SUR",
        "network": "IR",
        "latitude": -32.38,
        "longitude": 20.8117,
        "channel": "???",
        "isc_distance_deg": 124.72,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:06.83Z",
        "isc_residual_s": "-0.8",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CN01",
        "network": "IR",
        "latitude": 0.9309,
        "longitude": -51.2578,
        "channel": "???",
        "isc_distance_deg": 124.78,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:08.05Z",
        "isc_residual_s": "0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BJO1",
        "network": "NS",
        "latitude": 74.5037,
        "longitude": 19.0038,
        "channel": "HHZ",
        "isc_distance_deg": 124.79,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:06.86Z",
        "isc_residual_s": "0.3",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ARTI",
        "network": "IR",
        "latitude": 56.3879,
        "longitude": 58.3849,
        "channel": "???",
        "isc_distance_deg": 124.88,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:06.41Z",
        "isc_residual_s": "-0.4",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GUA01",
        "network": "IR",
        "latitude": -16.5835,
        "longitude": -39.8053,
        "channel": "H?Z",
        "isc_distance_deg": 125.24,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:08.02Z",
        "isc_residual_s": "-0.7",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GUA01",
        "network": "IR",
        "latitude": -16.5835,
        "longitude": -39.8053,
        "channel": "???",
        "isc_distance_deg": 125.24,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:08.52Z",
        "isc_residual_s": "-0.2",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AB31",
        "network": "IR",
        "latitude": 49.2556,
        "longitude": 59.9431,
        "channel": "S?Z",
        "isc_distance_deg": 125.67,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:08.60Z",
        "isc_residual_s": "0.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "IVI",
        "network": "G",
        "latitude": 61.2058,
        "longitude": -48.1712,
        "channel": "BHZ",
        "isc_distance_deg": 126.29,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:08.79Z",
        "isc_residual_s": "-0.5",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "JLN",
        "network": "IR",
        "latitude": 22.1505,
        "longitude": 59.4108,
        "channel": "???",
        "isc_distance_deg": 126.49,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:10.96Z",
        "isc_residual_s": "0.0",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BOSA",
        "network": "IR",
        "latitude": -28.6137,
        "longitude": 25.256,
        "channel": "???",
        "isc_distance_deg": 126.52,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:11.27Z",
        "isc_residual_s": "0.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BOSA",
        "network": "IR",
        "latitude": -28.6137,
        "longitude": 25.256,
        "channel": "B?Z",
        "isc_distance_deg": 126.52,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:11.62Z",
        "isc_residual_s": "-0.0",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BOSA",
        "network": "IR",
        "latitude": -28.6137,
        "longitude": 25.256,
        "channel": "???",
        "isc_distance_deg": 126.52,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:11.49Z",
        "isc_residual_s": "-0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBIT",
        "network": "IR",
        "latitude": -14.9307,
        "longitude": -39.4345,
        "channel": "???",
        "isc_distance_deg": 126.56,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:11.11Z",
        "isc_residual_s": "-0.2",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AKTO",
        "network": "IR",
        "latitude": 50.4348,
        "longitude": 58.0164,
        "channel": "???",
        "isc_distance_deg": 126.65,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:09.97Z",
        "isc_residual_s": "-0.4",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AKTO",
        "network": "IR",
        "latitude": 50.4348,
        "longitude": 58.0164,
        "channel": "S?Z",
        "isc_distance_deg": 126.65,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:10.00Z",
        "isc_residual_s": "-0.4",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WBK",
        "network": "IR",
        "latitude": 22.61,
        "longitude": 58.97,
        "channel": "???",
        "isc_distance_deg": 126.97,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:11.71Z",
        "isc_residual_s": "-0.2",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VADS",
        "network": "NS",
        "latitude": 70.1202,
        "longitude": 29.365,
        "channel": "HHZ",
        "isc_distance_deg": 127.31,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:26:59.12Z",
        "isc_residual_s": "-12.0",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NRS",
        "network": "IR",
        "latitude": 61.1595,
        "longitude": -45.4188,
        "channel": "S?Z",
        "isc_distance_deg": 127.48,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:11.60Z",
        "isc_residual_s": "-0.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NRS",
        "network": "DK",
        "latitude": 61.1595,
        "longitude": -45.4188,
        "channel": "BHZ",
        "isc_distance_deg": 127.48,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:11.00Z",
        "isc_residual_s": "-0.6",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MHTO",
        "network": "IR",
        "latitude": 20.9865,
        "longitude": 57.968,
        "channel": "???",
        "isc_distance_deg": 127.63,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:12.96Z",
        "isc_residual_s": "-0.2",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SCO",
        "network": "IR",
        "latitude": 70.4833,
        "longitude": -21.95,
        "channel": "S?Z",
        "isc_distance_deg": 127.67,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:12.50Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SCO",
        "network": "DK",
        "latitude": 70.4856,
        "longitude": -21.9497,
        "channel": "BHZ",
        "isc_distance_deg": 127.67,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:13.00Z",
        "isc_residual_s": "0.6",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "JMDO",
        "network": "IR",
        "latitude": 22.3701,
        "longitude": 58.1035,
        "channel": "???",
        "isc_distance_deg": 127.72,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:13.00Z",
        "isc_residual_s": "-0.3",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HAMF",
        "network": "NS",
        "latitude": 70.6425,
        "longitude": 23.6842,
        "channel": "HHZ",
        "isc_distance_deg": 127.81,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:13.93Z",
        "isc_residual_s": "1.3",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DQM",
        "network": "IR",
        "latitude": 19.9289,
        "longitude": 57.5393,
        "channel": "???",
        "isc_distance_deg": 127.84,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:13.11Z",
        "isc_residual_s": "-0.4",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BIDO",
        "network": "IR",
        "latitude": 23.5211,
        "longitude": 58.1269,
        "channel": "???",
        "isc_distance_deg": 127.87,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:13.02Z",
        "isc_residual_s": "-0.5",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SMDO",
        "network": "IR",
        "latitude": 23.0589,
        "longitude": 58.0492,
        "channel": "???",
        "isc_distance_deg": 127.87,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:13.96Z",
        "isc_residual_s": "0.3",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LVZ",
        "network": "IR",
        "latitude": 67.8979,
        "longitude": 34.6514,
        "channel": "B?Z",
        "isc_distance_deg": 127.96,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:11.80Z",
        "isc_residual_s": "-0.7",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KEV",
        "network": "IR",
        "latitude": 69.7553,
        "longitude": 27.0067,
        "channel": "S?Z",
        "isc_distance_deg": 128.06,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:12.90Z",
        "isc_residual_s": "-0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KEV",
        "network": "IR",
        "latitude": 69.7553,
        "longitude": 27.0067,
        "channel": "???",
        "isc_distance_deg": 128.06,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:12.23Z",
        "isc_residual_s": "-0.3",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ARA0",
        "network": "NO",
        "latitude": 69.5349,
        "longitude": 25.5058,
        "channel": "HHZ",
        "isc_distance_deg": 128.51,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:00.22Z",
        "isc_residual_s": "-13.2",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ARCES",
        "network": "IR",
        "latitude": 69.5349,
        "longitude": 25.5058,
        "channel": "???",
        "isc_distance_deg": 128.51,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:13.36Z",
        "isc_residual_s": "-0.1",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "APA",
        "network": "IR",
        "latitude": 67.569,
        "longitude": 33.4051,
        "channel": "S?Z",
        "isc_distance_deg": 128.52,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:12.00Z",
        "isc_residual_s": "-1.4",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KIRV",
        "network": "IR",
        "latitude": 58.585,
        "longitude": 49.4158,
        "channel": "???",
        "isc_distance_deg": 128.53,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:13.52Z",
        "isc_residual_s": "-0.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KIRV",
        "network": "IR",
        "latitude": 58.585,
        "longitude": 49.4158,
        "channel": "B?Z",
        "isc_distance_deg": 128.53,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:13.30Z",
        "isc_residual_s": "-0.4",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BSY",
        "network": "IR",
        "latitude": 22.7446,
        "longitude": 57.1995,
        "channel": "???",
        "isc_distance_deg": 128.6,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:15.04Z",
        "isc_residual_s": "0.0",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HOQ",
        "network": "IR",
        "latitude": 23.5824,
        "longitude": 57.3109,
        "channel": "???",
        "isc_distance_deg": 128.62,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:14.96Z",
        "isc_residual_s": "-0.0",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "JNW",
        "network": "NS",
        "latitude": 71.0273,
        "longitude": -8.4237,
        "channel": "HHZ",
        "isc_distance_deg": 128.72,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:10.36Z",
        "isc_residual_s": "-3.4",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "JNE",
        "network": "NS",
        "latitude": 70.9883,
        "longitude": -8.2927,
        "channel": "HHZ",
        "isc_distance_deg": 128.77,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:18.11Z",
        "isc_residual_s": "3.5",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MAKGR",
        "network": "IR",
        "latitude": -25.8118,
        "longitude": 24.8008,
        "channel": "B?Z",
        "isc_distance_deg": 129.17,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:16.50Z",
        "isc_residual_s": "0.3",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SHAO",
        "network": "IR",
        "latitude": 18.0228,
        "longitude": 55.6251,
        "channel": "???",
        "isc_distance_deg": 129.26,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:16.52Z",
        "isc_residual_s": "0.2",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "JETT",
        "network": "NO",
        "latitude": 69.5557,
        "longitude": 20.4095,
        "channel": "HHZ",
        "isc_distance_deg": 129.27,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:02.38Z",
        "isc_residual_s": "-12.5",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ARQ",
        "network": "IR",
        "latitude": 23.3366,
        "longitude": 56.5219,
        "channel": "???",
        "isc_distance_deg": 129.3,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:17.04Z",
        "isc_residual_s": "-0.1",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KTK1",
        "network": "NS",
        "latitude": 69.0117,
        "longitude": 23.2372,
        "channel": "HHZ",
        "isc_distance_deg": 129.35,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:01.46Z",
        "isc_residual_s": "-13.5",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SOHO",
        "network": "IR",
        "latitude": 24.1342,
        "longitude": 56.5336,
        "channel": "???",
        "isc_distance_deg": 129.39,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:15.59Z",
        "isc_residual_s": "-0.8",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SOHO",
        "network": "IR",
        "latitude": 24.1342,
        "longitude": 56.5336,
        "channel": "???",
        "isc_distance_deg": 129.39,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:16.15Z",
        "isc_residual_s": "-0.2",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TRO",
        "network": "NS",
        "latitude": 69.635,
        "longitude": 18.911,
        "channel": "HHZ",
        "isc_distance_deg": 129.39,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:00.72Z",
        "isc_residual_s": "-14.3",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LBTB",
        "network": "IR",
        "latitude": -25.0145,
        "longitude": 25.597,
        "channel": "S?Z",
        "isc_distance_deg": 129.52,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:13.70Z",
        "isc_residual_s": "-3.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MDH",
        "network": "IR",
        "latitude": 25.2987,
        "longitude": 56.2983,
        "channel": "???",
        "isc_distance_deg": 129.73,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:15.94Z",
        "isc_residual_s": "-1.1",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MDH",
        "network": "IR",
        "latitude": 25.2987,
        "longitude": 56.2983,
        "channel": "???",
        "isc_distance_deg": 129.73,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:16.96Z",
        "isc_residual_s": "-0.1",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UOSS",
        "network": "IR",
        "latitude": 24.9453,
        "longitude": 56.2042,
        "channel": "???",
        "isc_distance_deg": 129.78,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:15.85Z",
        "isc_residual_s": "-1.3",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UOSS",
        "network": "IR",
        "latitude": 24.9453,
        "longitude": 56.2042,
        "channel": "???",
        "isc_distance_deg": 129.78,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:16.60Z",
        "isc_residual_s": "-0.5",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UOSS",
        "network": "II",
        "latitude": 24.9453,
        "longitude": 56.2042,
        "channel": "???",
        "isc_distance_deg": 129.78,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:17.60Z",
        "isc_residual_s": "-0.4",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BANOM",
        "network": "IR",
        "latitude": 25.9233,
        "longitude": 56.2996,
        "channel": "???",
        "isc_distance_deg": 129.79,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:16.89Z",
        "isc_residual_s": "-0.3",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ADZR",
        "network": "IR",
        "latitude": 63.9295,
        "longitude": 38.2853,
        "channel": "S?Z",
        "isc_distance_deg": 129.83,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:12.80Z",
        "isc_residual_s": "-3.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DMTO",
        "network": "IR",
        "latitude": 17.7163,
        "longitude": 54.9475,
        "channel": "???",
        "isc_distance_deg": 129.83,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:16.76Z",
        "isc_residual_s": "-0.6",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HATD",
        "network": "IR",
        "latitude": 24.8257,
        "longitude": 56.1321,
        "channel": "???",
        "isc_distance_deg": 129.83,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:16.68Z",
        "isc_residual_s": "-0.6",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HATD",
        "network": "IR",
        "latitude": 24.8257,
        "longitude": 56.1321,
        "channel": "???",
        "isc_distance_deg": 129.83,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:16.60Z",
        "isc_residual_s": "-0.7",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MSFE",
        "network": "IR",
        "latitude": 25.3596,
        "longitude": 56.1595,
        "channel": "???",
        "isc_distance_deg": 129.86,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:16.95Z",
        "isc_residual_s": "-0.4",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MASF",
        "network": "IR",
        "latitude": 25.3712,
        "longitude": 56.1592,
        "channel": "???",
        "isc_distance_deg": 129.86,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:17.03Z",
        "isc_residual_s": "-0.3",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ASHO",
        "network": "IR",
        "latitude": 24.6839,
        "longitude": 56.0583,
        "channel": "???",
        "isc_distance_deg": 129.88,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:16.78Z",
        "isc_residual_s": "-0.6",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ASHO",
        "network": "IR",
        "latitude": 24.6839,
        "longitude": 56.0583,
        "channel": "???",
        "isc_distance_deg": 129.88,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:18.25Z",
        "isc_residual_s": "-0.0",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SHME",
        "network": "IR",
        "latitude": 26.0497,
        "longitude": 56.1414,
        "channel": "???",
        "isc_distance_deg": 129.94,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:17.03Z",
        "isc_residual_s": "-0.4",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SHME",
        "network": "IR",
        "latitude": 26.0497,
        "longitude": 56.1414,
        "channel": "???",
        "isc_distance_deg": 129.94,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:17.37Z",
        "isc_residual_s": "-0.0",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ALNE",
        "network": "IR",
        "latitude": 24.0561,
        "longitude": 55.7509,
        "channel": "???",
        "isc_distance_deg": 130.09,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:17.44Z",
        "isc_residual_s": "-0.3",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ALNE",
        "network": "IR",
        "latitude": 24.0561,
        "longitude": 55.7509,
        "channel": "???",
        "isc_distance_deg": 130.09,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:18.25Z",
        "isc_residual_s": "-0.4",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBLA",
        "network": "IR",
        "latitude": -10.9925,
        "longitude": -37.789,
        "channel": "H?Z",
        "isc_distance_deg": 130.27,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:17.90Z",
        "isc_residual_s": "-0.5",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NAZ",
        "network": "IR",
        "latitude": 24.9884,
        "longitude": 55.6618,
        "channel": "???",
        "isc_distance_deg": 130.27,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:17.32Z",
        "isc_residual_s": "-0.7",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NAZ",
        "network": "IR",
        "latitude": 24.9884,
        "longitude": 55.6618,
        "channel": "???",
        "isc_distance_deg": 130.27,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:18.08Z",
        "isc_residual_s": "0.0",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FAQ",
        "network": "IR",
        "latitude": 24.7453,
        "longitude": 55.5924,
        "channel": "???",
        "isc_distance_deg": 130.31,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:17.56Z",
        "isc_residual_s": "-0.6",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FAQ",
        "network": "IR",
        "latitude": 24.7453,
        "longitude": 55.5924,
        "channel": "???",
        "isc_distance_deg": 130.31,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:18.96Z",
        "isc_residual_s": "-0.1",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UMQ",
        "network": "IR",
        "latitude": 25.5252,
        "longitude": 55.6623,
        "channel": "???",
        "isc_distance_deg": 130.33,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:18.13Z",
        "isc_residual_s": "0.0",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UMQ",
        "network": "IR",
        "latitude": 25.5252,
        "longitude": 55.6623,
        "channel": "???",
        "isc_distance_deg": 130.33,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:19.14Z",
        "isc_residual_s": "0.1",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UMZA",
        "network": "IR",
        "latitude": 22.7138,
        "longitude": 55.1609,
        "channel": "???",
        "isc_distance_deg": 130.47,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:19.16Z",
        "isc_residual_s": "-0.3",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RBK",
        "network": "IR",
        "latitude": 17.5035,
        "longitude": 54.2039,
        "channel": "???",
        "isc_distance_deg": 130.48,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:18.30Z",
        "isc_residual_s": "-0.4",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ASUD",
        "network": "IR",
        "latitude": 24.626,
        "longitude": 55.3292,
        "channel": "???",
        "isc_distance_deg": 130.54,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:18.33Z",
        "isc_residual_s": "-0.2",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ASUD",
        "network": "IR",
        "latitude": 24.626,
        "longitude": 55.3292,
        "channel": "???",
        "isc_distance_deg": 130.54,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:17.97Z",
        "isc_residual_s": "-0.6",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SKOMA",
        "network": "IR",
        "latitude": -24.5153,
        "longitude": 23.9328,
        "channel": "B?Z",
        "isc_distance_deg": 130.68,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:19.51Z",
        "isc_residual_s": "0.4",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DOK",
        "network": "IR",
        "latitude": 18.618,
        "longitude": 54.1288,
        "channel": "???",
        "isc_distance_deg": 130.77,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:18.99Z",
        "isc_residual_s": "-0.2",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LPHEP",
        "network": "IR",
        "latitude": -23.363,
        "longitude": 25.8597,
        "channel": "B?Z",
        "isc_distance_deg": 130.82,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:19.83Z",
        "isc_residual_s": "0.5",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AJN",
        "network": "IR",
        "latitude": 24.5626,
        "longitude": 54.9865,
        "channel": "???",
        "isc_distance_deg": 130.84,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:18.19Z",
        "isc_residual_s": "-0.9",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AJN",
        "network": "IR",
        "latitude": 24.5626,
        "longitude": 54.9865,
        "channel": "???",
        "isc_distance_deg": 130.84,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:18.39Z",
        "isc_residual_s": "-0.7",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LOSSI",
        "network": "NS",
        "latitude": 68.191,
        "longitude": 17.7915,
        "channel": "HHZ",
        "isc_distance_deg": 130.89,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:05.26Z",
        "isc_residual_s": "-12.8",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WHFO",
        "network": "IR",
        "latitude": 17.919,
        "longitude": 53.7706,
        "channel": "???",
        "isc_distance_deg": 130.97,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:18.88Z",
        "isc_residual_s": "-0.7",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VBYGD",
        "network": "NS",
        "latitude": 68.3603,
        "longitude": 15.3903,
        "channel": "HHZ",
        "isc_distance_deg": 131.02,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:04.37Z",
        "isc_residual_s": "-13.8",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBPS",
        "network": "IR",
        "latitude": -4.394,
        "longitude": -41.4457,
        "channel": "H?Z",
        "isc_distance_deg": 131.04,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:20.10Z",
        "isc_residual_s": "0.1",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBTA",
        "network": "IR",
        "latitude": -9.1218,
        "longitude": -38.0632,
        "channel": "H?Z",
        "isc_distance_deg": 131.19,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:19.81Z",
        "isc_residual_s": "-0.4",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBTA",
        "network": "IR",
        "latitude": -9.1218,
        "longitude": -38.0632,
        "channel": "???",
        "isc_distance_deg": 131.19,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:06.19Z",
        "isc_residual_s": "-14.0",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBTA",
        "network": "IR",
        "latitude": -9.1218,
        "longitude": -38.0632,
        "channel": "???",
        "isc_distance_deg": 131.19,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:20.44Z",
        "isc_residual_s": "-0.8",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ABTO",
        "network": "IR",
        "latitude": 17.3538,
        "longitude": 53.2966,
        "channel": "???",
        "isc_distance_deg": 131.3,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:19.63Z",
        "isc_residual_s": "-0.6",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LOF",
        "network": "NS",
        "latitude": 68.1325,
        "longitude": 13.5398,
        "channel": "HHZ",
        "isc_distance_deg": 131.43,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:05.72Z",
        "isc_residual_s": "-13.2",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "STEI",
        "network": "NS",
        "latitude": 67.93,
        "longitude": 15.242,
        "channel": "HHZ",
        "isc_distance_deg": 131.45,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:03.87Z",
        "isc_residual_s": "-15.1",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KLMR",
        "network": "IR",
        "latitude": 60.8539,
        "longitude": 39.5186,
        "channel": "S?Z",
        "isc_distance_deg": 131.45,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:19.50Z",
        "isc_residual_s": "0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ESRB",
        "network": "IR",
        "latitude": 25.22,
        "longitude": 54.2339,
        "channel": "???",
        "isc_distance_deg": 131.59,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:19.77Z",
        "isc_residual_s": "-0.7",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBMA",
        "network": "IR",
        "latitude": -7.3654,
        "longitude": -38.7641,
        "channel": "???",
        "isc_distance_deg": 131.66,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:07.08Z",
        "isc_residual_s": "-14.0",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBMA",
        "network": "IR",
        "latitude": -7.3654,
        "longitude": -38.7641,
        "channel": "???",
        "isc_distance_deg": 131.66,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:21.55Z",
        "isc_residual_s": "-0.6",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MZWR",
        "network": "IR",
        "latitude": 23.761,
        "longitude": 53.8799,
        "channel": "???",
        "isc_distance_deg": 131.76,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:20.65Z",
        "isc_residual_s": "-0.2",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MZWR",
        "network": "IR",
        "latitude": 23.761,
        "longitude": 53.8799,
        "channel": "???",
        "isc_distance_deg": 131.76,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:21.02Z",
        "isc_residual_s": "0.1",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FAUS",
        "network": "NS",
        "latitude": 67.382,
        "longitude": 15.2877,
        "channel": "HHZ",
        "isc_distance_deg": 131.97,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:05.73Z",
        "isc_residual_s": "-14.2",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MATP",
        "network": "IR",
        "latitude": -20.4258,
        "longitude": 28.4994,
        "channel": "???",
        "isc_distance_deg": 131.97,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:21.85Z",
        "isc_residual_s": "0.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MZR",
        "network": "IR",
        "latitude": 23.0024,
        "longitude": 53.4794,
        "channel": "???",
        "isc_distance_deg": 132.04,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:20.38Z",
        "isc_residual_s": "-1.1",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MZR",
        "network": "IR",
        "latitude": 23.0024,
        "longitude": 53.4794,
        "channel": "???",
        "isc_distance_deg": 132.04,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:21.18Z",
        "isc_residual_s": "-0.3",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LKGWB",
        "network": "IR",
        "latitude": -24.1135,
        "longitude": 21.2178,
        "channel": "B?Z",
        "isc_distance_deg": 132.14,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:21.91Z",
        "isc_residual_s": "0.0",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ROEST",
        "network": "NS",
        "latitude": 67.5202,
        "longitude": 12.1045,
        "channel": "HHZ",
        "isc_distance_deg": 132.15,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:08.07Z",
        "isc_residual_s": "-12.2",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GILDE",
        "network": "NS",
        "latitude": 66.9293,
        "longitude": 14.1897,
        "channel": "HHZ",
        "isc_distance_deg": 132.53,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:06.51Z",
        "isc_residual_s": "-14.6",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GHWR",
        "network": "IR",
        "latitude": 23.9635,
        "longitude": 52.8755,
        "channel": "???",
        "isc_distance_deg": 132.7,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:22.39Z",
        "isc_residual_s": "-0.2",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GHWR",
        "network": "IR",
        "latitude": 23.9635,
        "longitude": 52.8755,
        "channel": "???",
        "isc_distance_deg": 132.7,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:21.94Z",
        "isc_residual_s": "-0.7",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBMO",
        "network": "IR",
        "latitude": -3.3107,
        "longitude": -40.0414,
        "channel": "H?Z",
        "isc_distance_deg": 132.8,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:23.55Z",
        "isc_residual_s": "0.4",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBMO",
        "network": "IR",
        "latitude": -3.3107,
        "longitude": -40.0414,
        "channel": "???",
        "isc_distance_deg": 132.8,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:23.57Z",
        "isc_residual_s": "0.4",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BORG",
        "network": "IR",
        "latitude": 64.7474,
        "longitude": -21.3268,
        "channel": "S?Z",
        "isc_distance_deg": 132.81,
        "isc_phase": "PKiKP",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:25.60Z",
        "isc_residual_s": "2.7",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "JRN",
        "network": "IR",
        "latitude": 24.9422,
        "longitude": 52.8491,
        "channel": "???",
        "isc_distance_deg": 132.81,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:23.16Z",
        "isc_residual_s": "0.3",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VAGH",
        "network": "NS",
        "latitude": 66.7303,
        "longitude": 13.2705,
        "channel": "HHZ",
        "isc_distance_deg": 132.81,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:09.31Z",
        "isc_residual_s": "-12.2",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KGCAE",
        "network": "IR",
        "latitude": -22.8538,
        "longitude": 22.2068,
        "channel": "B?Z",
        "isc_distance_deg": 132.87,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:23.83Z",
        "isc_residual_s": "0.6",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RAUS",
        "network": "NS",
        "latitude": 66.4365,
        "longitude": 14.042,
        "channel": "HHZ",
        "isc_distance_deg": 133.02,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:08.65Z",
        "isc_residual_s": "-13.3",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SOOWA",
        "network": "IR",
        "latitude": -20.5628,
        "longitude": 26.2178,
        "channel": "B?Z",
        "isc_distance_deg": 133.03,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:24.13Z",
        "isc_residual_s": "0.5",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KONS",
        "network": "NS",
        "latitude": 66.4992,
        "longitude": 13.1177,
        "channel": "HHZ",
        "isc_distance_deg": 133.05,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:10.86Z",
        "isc_residual_s": "-11.2",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MOR8",
        "network": "NS",
        "latitude": 66.283,
        "longitude": 14.7308,
        "channel": "HHZ",
        "isc_distance_deg": 133.09,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:06.78Z",
        "isc_residual_s": "-15.4",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBLV",
        "network": "IR",
        "latitude": -7.3584,
        "longitude": -36.9203,
        "channel": "H?Z",
        "isc_distance_deg": 133.14,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:24.03Z",
        "isc_residual_s": "0.1",
        "reporter": "VAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "STOK",
        "network": "NS",
        "latitude": 66.333,
        "longitude": 13.0177,
        "channel": "HHZ",
        "isc_distance_deg": 133.23,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:09.75Z",
        "isc_residual_s": "-12.6",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBCA",
        "network": "IR",
        "latitude": -8.2258,
        "longitude": -36.013,
        "channel": "???",
        "isc_distance_deg": 133.34,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:10.53Z",
        "isc_residual_s": "-13.7",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBCA",
        "network": "IR",
        "latitude": -8.2258,
        "longitude": -36.013,
        "channel": "???",
        "isc_distance_deg": 133.34,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:25.68Z",
        "isc_residual_s": "0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LEIR",
        "network": "NS",
        "latitude": 66.1118,
        "longitude": 13.115,
        "channel": "HHZ",
        "isc_distance_deg": 133.43,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:10.47Z",
        "isc_residual_s": "-12.3",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SLWR",
        "network": "IR",
        "latitude": 24.0696,
        "longitude": 51.6913,
        "channel": "???",
        "isc_distance_deg": 133.78,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:23.68Z",
        "isc_residual_s": "-1.0",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SLWR",
        "network": "IR",
        "latitude": 24.0696,
        "longitude": 51.6913,
        "channel": "???",
        "isc_distance_deg": 133.78,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:25.19Z",
        "isc_residual_s": "0.5",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBCL",
        "network": "IR",
        "latitude": -4.2243,
        "longitude": -38.291,
        "channel": "???",
        "isc_distance_deg": 133.79,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:11.89Z",
        "isc_residual_s": "-13.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBCL",
        "network": "IR",
        "latitude": -4.2243,
        "longitude": -38.291,
        "channel": "???",
        "isc_distance_deg": 133.79,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:25.69Z",
        "isc_residual_s": "-0.8",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NBPA",
        "network": "IR",
        "latitude": -5.7503,
        "longitude": -37.1121,
        "channel": "???",
        "isc_distance_deg": 133.92,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:13.01Z",
        "isc_residual_s": "-12.3",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TRNA",
        "network": "IR",
        "latitude": 24.7355,
        "longitude": 51.2125,
        "channel": "???",
        "isc_distance_deg": 134.28,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:25.96Z",
        "isc_residual_s": "0.4",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SHMA",
        "network": "IR",
        "latitude": 25.8565,
        "longitude": 50.964,
        "channel": "???",
        "isc_distance_deg": 134.58,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:27.98Z",
        "isc_residual_s": "0.3",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GRTLG",
        "network": "IR",
        "latitude": -21.3618,
        "longitude": 21.2157,
        "channel": "B?Z",
        "isc_distance_deg": 134.61,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:28.52Z",
        "isc_residual_s": "0.4",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SMRA",
        "network": "IR",
        "latitude": 24.7453,
        "longitude": 50.8239,
        "channel": "???",
        "isc_distance_deg": 134.63,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:25.91Z",
        "isc_residual_s": "-0.3",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SAKB",
        "network": "IR",
        "latitude": 25.9782,
        "longitude": 50.5365,
        "channel": "???",
        "isc_distance_deg": 134.97,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:26.60Z",
        "isc_residual_s": "-0.2",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WIN",
        "network": "IR",
        "latitude": -22.5667,
        "longitude": 17.1,
        "channel": "B?Z",
        "isc_distance_deg": 135.02,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:27.85Z",
        "isc_residual_s": "0.4",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NSS",
        "network": "NS",
        "latitude": 64.5303,
        "longitude": 11.9658,
        "channel": "HHZ",
        "isc_distance_deg": 135.09,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:13.63Z",
        "isc_residual_s": "-12.3",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FINES",
        "network": "IR",
        "latitude": 61.4436,
        "longitude": 26.0771,
        "channel": "???",
        "isc_distance_deg": 135.35,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:25.88Z",
        "isc_residual_s": "-0.6",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PUL",
        "network": "IR",
        "latitude": 59.7728,
        "longitude": 30.3222,
        "channel": "S?Z",
        "isc_distance_deg": 135.45,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:27.10Z",
        "isc_residual_s": "0.4",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MOS",
        "network": "IR",
        "latitude": 55.7383,
        "longitude": 37.625,
        "channel": "S?Z",
        "isc_distance_deg": 135.47,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:26.70Z",
        "isc_residual_s": "-0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LKRN",
        "network": "IR",
        "latitude": 38.71,
        "longitude": 48.779,
        "channel": "S?Z",
        "isc_distance_deg": 135.6,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:28.10Z",
        "isc_residual_s": "0.5",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KSANE",
        "network": "IR",
        "latitude": -17.8002,
        "longitude": 25.1618,
        "channel": "B?Z",
        "isc_distance_deg": 135.89,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:29.40Z",
        "isc_residual_s": "0.4",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GMARE",
        "network": "IR",
        "latitude": -19.3893,
        "longitude": 22.1625,
        "channel": "B?Z",
        "isc_distance_deg": 135.96,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:28.62Z",
        "isc_residual_s": "-0.5",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SEKA",
        "network": "IR",
        "latitude": 41.209,
        "longitude": 47.198,
        "channel": "S?Z",
        "isc_distance_deg": 136.23,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:29.00Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LADE",
        "network": "NS",
        "latitude": 63.455,
        "longitude": 10.4458,
        "channel": "HHZ",
        "isc_distance_deg": 136.29,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:18.31Z",
        "isc_residual_s": "-9.8",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LSZ",
        "network": "IR",
        "latitude": -15.2766,
        "longitude": 28.1882,
        "channel": "B?Z",
        "isc_distance_deg": 136.29,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:30.05Z",
        "isc_residual_s": "0.3",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LENS",
        "network": "NS",
        "latitude": 63.4957,
        "longitude": 9.7342,
        "channel": "HHZ",
        "isc_distance_deg": 136.31,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:17.13Z",
        "isc_residual_s": "-11.1",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OBN",
        "network": "IR",
        "latitude": 55.1138,
        "longitude": 36.5687,
        "channel": "???",
        "isc_distance_deg": 136.33,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:28.31Z",
        "isc_residual_s": "-0.1",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OBN",
        "network": "IR",
        "latitude": 55.1138,
        "longitude": 36.5687,
        "channel": "S?Z",
        "isc_distance_deg": 136.33,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:28.10Z",
        "isc_residual_s": "-0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MNGR",
        "network": "IR",
        "latitude": 40.773,
        "longitude": 47.085,
        "channel": "S?Z",
        "isc_distance_deg": 136.43,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:29.50Z",
        "isc_residual_s": "0.4",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GANJ",
        "network": "IR",
        "latitude": 40.646,
        "longitude": 46.322,
        "channel": "S?Z",
        "isc_distance_deg": 137.02,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:30.50Z",
        "isc_residual_s": "0.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "XAUDM",
        "network": "IR",
        "latitude": -18.5395,
        "longitude": 21.3382,
        "channel": "B?Z",
        "isc_distance_deg": 137.07,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:31.79Z",
        "isc_residual_s": "0.6",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MOL",
        "network": "NS",
        "latitude": 62.57,
        "longitude": 7.5468,
        "channel": "HHZ",
        "isc_distance_deg": 137.39,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:19.73Z",
        "isc_residual_s": "-10.5",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VORD",
        "network": "IR",
        "latitude": 50.966,
        "longitude": 39.293,
        "channel": "B?Z",
        "isc_distance_deg": 137.41,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:29.00Z",
        "isc_residual_s": "-1.6",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DOMB",
        "network": "NS",
        "latitude": 62.0732,
        "longitude": 9.1122,
        "channel": "HHZ",
        "isc_distance_deg": 137.77,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:21.38Z",
        "isc_residual_s": "-9.6",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TSUM",
        "network": "IR",
        "latitude": -19.2022,
        "longitude": 17.5838,
        "channel": "B?Z",
        "isc_distance_deg": 137.97,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:32.77Z",
        "isc_residual_s": "-0.1",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NAX",
        "network": "IR",
        "latitude": 39.174,
        "longitude": 45.495,
        "channel": "S?Z",
        "isc_distance_deg": 138.0,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:32.10Z",
        "isc_residual_s": "-0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KIBK",
        "network": "IR",
        "latitude": -2.34,
        "longitude": 38.046,
        "channel": "BHZ",
        "isc_distance_deg": 138.01,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:22.88Z",
        "isc_residual_s": "-10.1",
        "reporter": "SFS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KIBK",
        "network": "IR",
        "latitude": -2.34,
        "longitude": 38.046,
        "channel": "???",
        "isc_distance_deg": 138.01,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:24.84Z",
        "isc_residual_s": "-8.1",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KBZ",
        "network": "IR",
        "latitude": 43.7286,
        "longitude": 42.8975,
        "channel": "???",
        "isc_distance_deg": 138.49,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:32.97Z",
        "isc_residual_s": "0.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KBZ",
        "network": "IR",
        "latitude": 43.7286,
        "longitude": 42.8975,
        "channel": "S?Z",
        "isc_distance_deg": 138.49,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:33.00Z",
        "isc_residual_s": "0.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FOO",
        "network": "NS",
        "latitude": 61.5973,
        "longitude": 5.0427,
        "channel": "HHZ",
        "isc_distance_deg": 138.49,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:24.66Z",
        "isc_residual_s": "-7.6",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KIV",
        "network": "IR",
        "latitude": 43.9553,
        "longitude": 42.6863,
        "channel": "???",
        "isc_distance_deg": 138.55,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:25.52Z",
        "isc_residual_s": "-7.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KIV",
        "network": "IR",
        "latitude": 43.9553,
        "longitude": 42.6863,
        "channel": "???",
        "isc_distance_deg": 138.55,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:24.86Z",
        "isc_residual_s": "-8.2",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KIV",
        "network": "IR",
        "latitude": 43.9553,
        "longitude": 42.6863,
        "channel": "???",
        "isc_distance_deg": 138.55,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:34.17Z",
        "isc_residual_s": "-1.2",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NB2",
        "network": "IR",
        "latitude": 61.0397,
        "longitude": 11.2148,
        "channel": "S?Z",
        "isc_distance_deg": 138.58,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:24.20Z",
        "isc_residual_s": "-8.3",
        "reporter": "NAO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NB2",
        "network": "IR",
        "latitude": 61.0397,
        "longitude": 11.2148,
        "channel": "S?Z",
        "isc_distance_deg": 138.58,
        "isc_phase": "PKPdf",
        "reported_phase": "AMP",
        "arrival_utc": "2022-11-12T07:27:24.20Z",
        "isc_residual_s": "-8.3",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NB2",
        "network": "IR",
        "latitude": 61.0397,
        "longitude": 11.2148,
        "channel": "S?Z",
        "isc_distance_deg": 138.58,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:24.20Z",
        "isc_residual_s": "-8.3",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NB2",
        "network": "IR",
        "latitude": 61.0397,
        "longitude": 11.2148,
        "channel": "S?Z",
        "isc_distance_deg": 138.58,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:24.20Z",
        "isc_residual_s": "-8.3",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NC602",
        "network": "NO",
        "latitude": 60.7352,
        "longitude": 11.5414,
        "channel": "BHZ",
        "isc_distance_deg": 138.84,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:24.35Z",
        "isc_residual_s": "-8.6",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HYA",
        "network": "NS",
        "latitude": 61.1633,
        "longitude": 6.1855,
        "channel": "HHZ",
        "isc_distance_deg": 138.87,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:25.97Z",
        "isc_residual_s": "-7.0",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SUE",
        "network": "NS",
        "latitude": 61.057,
        "longitude": 4.7595,
        "channel": "HHZ",
        "isc_distance_deg": 139.04,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:25.83Z",
        "isc_residual_s": "-7.4",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SKAR",
        "network": "NS",
        "latitude": 60.6807,
        "longitude": 8.3043,
        "channel": "HHZ",
        "isc_distance_deg": 139.21,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:25.35Z",
        "isc_residual_s": "-8.3",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KMBO",
        "network": "IR",
        "latitude": -1.1268,
        "longitude": 37.2523,
        "channel": "???",
        "isc_distance_deg": 139.34,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:26.89Z",
        "isc_residual_s": "-8.7",
        "reporter": "DSN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HNA0",
        "network": "QE",
        "latitude": 60.6106,
        "longitude": 4.9571,
        "channel": "HHZ",
        "isc_distance_deg": 139.48,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:27.61Z",
        "isc_residual_s": "-6.4",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LABN",
        "network": "IR",
        "latitude": 44.6413,
        "longitude": 40.7236,
        "channel": "S?Z",
        "isc_distance_deg": 139.57,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:35.10Z",
        "isc_residual_s": "0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ASK",
        "network": "NS",
        "latitude": 60.4753,
        "longitude": 5.1997,
        "channel": "HHZ",
        "isc_distance_deg": 139.61,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:28.29Z",
        "isc_residual_s": "-6.0",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ATD",
        "network": "IR",
        "latitude": 11.53,
        "longitude": 42.847,
        "channel": "???",
        "isc_distance_deg": 139.68,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:30.45Z",
        "isc_residual_s": "-5.4",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BER",
        "network": "NS",
        "latitude": 60.3845,
        "longitude": 5.3295,
        "channel": "HHZ",
        "isc_distance_deg": 139.69,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:27.52Z",
        "isc_residual_s": "-6.9",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OSL",
        "network": "NS",
        "latitude": 59.9373,
        "longitude": 10.7218,
        "channel": "HHZ",
        "isc_distance_deg": 139.71,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:27.33Z",
        "isc_residual_s": "-7.2",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BAS16",
        "network": "NS",
        "latitude": 60.2332,
        "longitude": 5.1777,
        "channel": "HHZ",
        "isc_distance_deg": 139.85,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:27.77Z",
        "isc_residual_s": "-7.0",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BATM",
        "network": "IR",
        "latitude": 41.604,
        "longitude": 41.6936,
        "channel": "???",
        "isc_distance_deg": 140.07,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:27.92Z",
        "isc_residual_s": "-7.8",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ODD1",
        "network": "NS",
        "latitude": 59.9108,
        "longitude": 6.6263,
        "channel": "HHZ",
        "isc_distance_deg": 140.1,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:29.25Z",
        "isc_residual_s": "-6.0",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BCA",
        "network": "IR",
        "latitude": 41.445,
        "longitude": 41.6223,
        "channel": "???",
        "isc_distance_deg": 140.17,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:28.40Z",
        "isc_residual_s": "-7.6",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HOMI",
        "network": "IR",
        "latitude": 40.0452,
        "longitude": 41.9055,
        "channel": "???",
        "isc_distance_deg": 140.43,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:29.89Z",
        "isc_residual_s": "-6.9",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "STRU",
        "network": "UP",
        "latitude": 59.0345,
        "longitude": 11.1825,
        "channel": "HHZ",
        "isc_distance_deg": 140.53,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:26.85Z",
        "isc_residual_s": "-9.2",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BLS5",
        "network": "NS",
        "latitude": 59.423,
        "longitude": 6.4575,
        "channel": "HHZ",
        "isc_distance_deg": 140.59,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:30.38Z",
        "isc_residual_s": "-5.8",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KMY",
        "network": "NS",
        "latitude": 59.2108,
        "longitude": 5.2453,
        "channel": "HHZ",
        "isc_distance_deg": 140.87,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:29.36Z",
        "isc_residual_s": "-7.2",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BIGH",
        "network": "IR",
        "latitude": 58.4933,
        "longitude": -3.9105,
        "channel": "H?Z",
        "isc_distance_deg": 141.44,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:33.28Z",
        "isc_residual_s": "-4.4",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TJOU",
        "network": "UP",
        "latitude": 58.032,
        "longitude": 11.625,
        "channel": "HHZ",
        "isc_distance_deg": 141.45,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:31.28Z",
        "isc_residual_s": "-6.4",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OSKU",
        "network": "UP",
        "latitude": 57.1947,
        "longitude": 16.0994,
        "channel": "HHZ",
        "isc_distance_deg": 141.49,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:31.57Z",
        "isc_residual_s": "-6.2",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HOMB",
        "network": "NS",
        "latitude": 58.2702,
        "longitude": 8.5052,
        "channel": "HHZ",
        "isc_distance_deg": 141.58,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:32.48Z",
        "isc_residual_s": "-5.4",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HOMB",
        "network": "NS",
        "latitude": 58.2702,
        "longitude": 8.5052,
        "channel": "HHZ",
        "isc_distance_deg": 141.58,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:32.88Z",
        "isc_residual_s": "-5.0",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SNART",
        "network": "NS",
        "latitude": 58.3387,
        "longitude": 7.2097,
        "channel": "HHZ",
        "isc_distance_deg": 141.62,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:32.27Z",
        "isc_residual_s": "-5.7",
        "reporter": "BER",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BORU",
        "network": "UP",
        "latitude": 57.6345,
        "longitude": 12.7697,
        "channel": "HHZ",
        "isc_distance_deg": 141.67,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:32.16Z",
        "isc_residual_s": "-5.9",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VXJU",
        "network": "UP",
        "latitude": 56.9212,
        "longitude": 14.9391,
        "channel": "HHZ",
        "isc_distance_deg": 141.97,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:33.17Z",
        "isc_residual_s": "-5.5",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ONAU",
        "network": "UP",
        "latitude": 57.3965,
        "longitude": 11.9177,
        "channel": "HHZ",
        "isc_distance_deg": 142.03,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:33.32Z",
        "isc_residual_s": "-5.4",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FABU",
        "network": "UP",
        "latitude": 57.0011,
        "longitude": 12.7184,
        "channel": "HHZ",
        "isc_distance_deg": 142.28,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:34.92Z",
        "isc_residual_s": "-4.3",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OVD",
        "network": "DK",
        "latitude": 57.317,
        "longitude": 10.206,
        "channel": "HHZ",
        "isc_distance_deg": 142.33,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:35.22Z",
        "isc_residual_s": "-4.0",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BLEU",
        "network": "UP",
        "latitude": 56.304,
        "longitude": 15.8145,
        "channel": "HHZ",
        "isc_distance_deg": 142.37,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:34.78Z",
        "isc_residual_s": "-4.6",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MCD",
        "network": "IR",
        "latitude": 57.5828,
        "longitude": -3.2542,
        "channel": "H?Z",
        "isc_distance_deg": 142.4,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:36.30Z",
        "isc_residual_s": "-3.1",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SUW",
        "network": "IR",
        "latitude": 54.0125,
        "longitude": 23.1808,
        "channel": "??Z",
        "isc_distance_deg": 142.41,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:36.51Z",
        "isc_residual_s": "-3.0",
        "reporter": "WAR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KPL",
        "network": "IR",
        "latitude": 57.3391,
        "longitude": -5.6527,
        "channel": "H?Z",
        "isc_distance_deg": 142.42,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:36.89Z",
        "isc_residual_s": "-2.5",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DEL",
        "network": "UP",
        "latitude": 56.4701,
        "longitude": 13.8705,
        "channel": "HHZ",
        "isc_distance_deg": 142.59,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:35.80Z",
        "isc_residual_s": "-4.0",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KIEV",
        "network": "IR",
        "latitude": 50.6944,
        "longitude": 29.2083,
        "channel": "???",
        "isc_distance_deg": 142.59,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:36.49Z",
        "isc_residual_s": "-3.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GOET",
        "network": "DK",
        "latitude": 57.035,
        "longitude": 9.225,
        "channel": "HHZ",
        "isc_distance_deg": 142.72,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:37.40Z",
        "isc_residual_s": "-2.5",
        "reporter": "DNK",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GVIE",
        "network": "IR",
        "latitude": 57.101,
        "longitude": -4.56,
        "channel": "H?Z",
        "isc_distance_deg": 142.77,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:37.63Z",
        "isc_residual_s": "-2.5",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TARL",
        "network": "IR",
        "latitude": 57.1608,
        "longitude": -2.8725,
        "channel": "H?Z",
        "isc_distance_deg": 142.84,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:37.88Z",
        "isc_residual_s": "-2.3",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LUBAN",
        "network": "IR",
        "latitude": -14.9059,
        "longitude": 13.4483,
        "channel": "B?Z",
        "isc_distance_deg": 143.36,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:42.04Z",
        "isc_residual_s": "-0.6",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SHEL",
        "network": "IR",
        "latitude": -15.9588,
        "longitude": -5.7457,
        "channel": "S?Z",
        "isc_distance_deg": 143.4,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:43.20Z",
        "isc_residual_s": "0.7",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SIM",
        "network": "IR",
        "latitude": 44.949,
        "longitude": 34.116,
        "channel": "S?Z",
        "isc_distance_deg": 143.57,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:41.70Z",
        "isc_residual_s": "-0.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ESY",
        "network": "IR",
        "latitude": 55.9178,
        "longitude": -2.6144,
        "channel": "H?Z",
        "isc_distance_deg": 144.1,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:42.31Z",
        "isc_residual_s": "-0.1",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "IDGL",
        "network": "IR",
        "latitude": 55.073,
        "longitude": -7.51,
        "channel": "???",
        "isc_distance_deg": 144.39,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:42.50Z",
        "isc_residual_s": "-0.4",
        "reporter": "DIAS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DIKM",
        "network": "IR",
        "latitude": 41.6496,
        "longitude": 35.2578,
        "channel": "???",
        "isc_distance_deg": 144.47,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:43.43Z",
        "isc_residual_s": "-0.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "EKA",
        "network": "IR",
        "latitude": 55.3331,
        "longitude": -3.1592,
        "channel": "???",
        "isc_distance_deg": 144.64,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:43.52Z",
        "isc_residual_s": "0.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NEWG",
        "network": "IR",
        "latitude": 55.1173,
        "longitude": -4.23,
        "channel": "H?Z",
        "isc_distance_deg": 144.76,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:43.61Z",
        "isc_residual_s": "0.0",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SORM",
        "network": "IR",
        "latitude": 48.1349,
        "longitude": 28.3513,
        "channel": "???",
        "isc_distance_deg": 144.78,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:43.61Z",
        "isc_residual_s": "-0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SORM",
        "network": "IR",
        "latitude": 48.1349,
        "longitude": 28.3513,
        "channel": "S?Z",
        "isc_distance_deg": 144.78,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:43.90Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BZK",
        "network": "IR",
        "latitude": 41.96,
        "longitude": 34.0035,
        "channel": "???",
        "isc_distance_deg": 145.16,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:44.86Z",
        "isc_residual_s": "0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "EDMD",
        "network": "IR",
        "latitude": 54.8312,
        "longitude": -1.9636,
        "channel": "H?Z",
        "isc_distance_deg": 145.22,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:44.76Z",
        "isc_residual_s": "0.4",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DRBR",
        "network": "IR",
        "latitude": 48.2073,
        "longitude": 26.6091,
        "channel": "???",
        "isc_distance_deg": 145.56,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:45.54Z",
        "isc_residual_s": "0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ILTH",
        "network": "IR",
        "latitude": 54.0279,
        "longitude": -6.3362,
        "channel": "H?Z",
        "isc_distance_deg": 145.58,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:45.69Z",
        "isc_residual_s": "0.7",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AP12",
        "network": "UR",
        "latitude": 54.31,
        "longitude": -3.2668,
        "channel": "H?Z",
        "isc_distance_deg": 145.65,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:45.72Z",
        "isc_residual_s": "0.6",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "IGLA",
        "network": "IR",
        "latitude": 53.4195,
        "longitude": -9.375,
        "channel": "???",
        "isc_distance_deg": 145.66,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:45.50Z",
        "isc_residual_s": "0.4",
        "reporter": "DIAS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "IGLA",
        "network": "IR",
        "latitude": 53.4195,
        "longitude": -9.375,
        "channel": "H?Z",
        "isc_distance_deg": 145.66,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:45.78Z",
        "isc_residual_s": "0.7",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GDLE",
        "network": "IR",
        "latitude": 54.4218,
        "longitude": -0.8157,
        "channel": "H?Z",
        "isc_distance_deg": 145.69,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:45.98Z",
        "isc_residual_s": "0.8",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "H07N1",
        "network": "IR",
        "latitude": 39.6912,
        "longitude": -31.1196,
        "channel": "???",
        "isc_distance_deg": 145.71,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:49.91Z",
        "isc_residual_s": "-0.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CTON",
        "network": "IR",
        "latitude": 54.254,
        "longitude": -1.938,
        "channel": "H?Z",
        "isc_distance_deg": 145.8,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:46.14Z",
        "isc_residual_s": "0.7",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AU18",
        "network": "UR",
        "latitude": 54.248,
        "longitude": -0.71,
        "channel": "H?Z",
        "isc_distance_deg": 145.87,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:46.44Z",
        "isc_residual_s": "1.0",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CIDE",
        "network": "IR",
        "latitude": 41.8821,
        "longitude": 32.9471,
        "channel": "???",
        "isc_distance_deg": 145.88,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:45.63Z",
        "isc_residual_s": "-0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AR05",
        "network": "UR",
        "latitude": 53.9908,
        "longitude": -2.0185,
        "channel": "H?Z",
        "isc_distance_deg": 146.06,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:46.87Z",
        "isc_residual_s": "1.0",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AQ10",
        "network": "UR",
        "latitude": 53.9148,
        "longitude": -2.827,
        "channel": "H?Z",
        "isc_distance_deg": 146.07,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:46.76Z",
        "isc_residual_s": "0.9",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NGRR",
        "network": "IR",
        "latitude": 46.8375,
        "longitude": 27.4422,
        "channel": "???",
        "isc_distance_deg": 146.11,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:46.96Z",
        "isc_residual_s": "0.8",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KWP",
        "network": "IR",
        "latitude": 49.6314,
        "longitude": 22.7075,
        "channel": "??Z",
        "isc_distance_deg": 146.15,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:47.23Z",
        "isc_residual_s": "1.1",
        "reporter": "WAR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KWP",
        "network": "IR",
        "latitude": 49.6314,
        "longitude": 22.7075,
        "channel": "S?Z",
        "isc_distance_deg": 146.15,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:47.20Z",
        "isc_residual_s": "1.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AU05",
        "network": "UR",
        "latitude": 53.86,
        "longitude": -0.874,
        "channel": "H?Z",
        "isc_distance_deg": 146.25,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:46.92Z",
        "isc_residual_s": "0.8",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AR01",
        "network": "UR",
        "latitude": 53.7022,
        "longitude": -2.345,
        "channel": "H?Z",
        "isc_distance_deg": 146.32,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.40Z",
        "isc_residual_s": "1.1",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DSB",
        "network": "IR",
        "latitude": 53.2452,
        "longitude": -6.3762,
        "channel": "???",
        "isc_distance_deg": 146.33,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:46.60Z",
        "isc_residual_s": "0.3",
        "reporter": "DIAS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DSB",
        "network": "IR",
        "latitude": 53.2452,
        "longitude": -6.3762,
        "channel": "H?Z",
        "isc_distance_deg": 146.33,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.16Z",
        "isc_residual_s": "0.9",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PRAR",
        "network": "IR",
        "latitude": 47.3617,
        "longitude": 26.2276,
        "channel": "???",
        "isc_distance_deg": 146.34,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.49Z",
        "isc_residual_s": "0.9",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GORT1",
        "network": "IR",
        "latitude": 52.9903,
        "longitude": 11.3075,
        "channel": "???",
        "isc_distance_deg": 146.37,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:47.10Z",
        "isc_residual_s": "0.8",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GORT1",
        "network": "IR",
        "latitude": 52.9903,
        "longitude": 11.3075,
        "channel": "???",
        "isc_distance_deg": 146.37,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:50.51Z",
        "isc_residual_s": "-0.4",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RUE",
        "network": "IR",
        "latitude": 52.48,
        "longitude": 13.78,
        "channel": "???",
        "isc_distance_deg": 146.38,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:46.96Z",
        "isc_residual_s": "0.6",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RUE",
        "network": "IR",
        "latitude": 52.48,
        "longitude": 13.78,
        "channel": "???",
        "isc_distance_deg": 146.38,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:50.29Z",
        "isc_residual_s": "-0.7",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BIR",
        "network": "IR",
        "latitude": 46.2664,
        "longitude": 27.6264,
        "channel": "???",
        "isc_distance_deg": 146.41,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.20Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BIR",
        "network": "IR",
        "latitude": 46.2664,
        "longitude": 27.6264,
        "channel": "S?Z",
        "isc_distance_deg": 146.41,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:47.20Z",
        "isc_residual_s": "0.5",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BEIL",
        "network": "IR",
        "latitude": 34.5417,
        "longitude": 36.1847,
        "channel": "S?Z",
        "isc_distance_deg": 146.42,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.01Z",
        "isc_residual_s": "-0.1",
        "reporter": "GRAL",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VLDR",
        "network": "IR",
        "latitude": 45.8443,
        "longitude": 28.0808,
        "channel": "???",
        "isc_distance_deg": 146.45,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.11Z",
        "isc_residual_s": "0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ASF",
        "network": "IR",
        "latitude": 32.173,
        "longitude": 36.8473,
        "channel": "???",
        "isc_distance_deg": 146.46,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:51.10Z",
        "isc_residual_s": "-0.9",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ASF",
        "network": "JS",
        "latitude": 32.173,
        "longitude": 36.8473,
        "channel": "???",
        "isc_distance_deg": 146.46,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.10Z",
        "isc_residual_s": "0.8",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TLCR",
        "network": "IR",
        "latitude": 45.1861,
        "longitude": 28.815,
        "channel": "???",
        "isc_distance_deg": 146.48,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.05Z",
        "isc_residual_s": "0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TLCR",
        "network": "IR",
        "latitude": 45.1861,
        "longitude": 28.815,
        "channel": "S?Z",
        "isc_distance_deg": 146.48,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:46.50Z",
        "isc_residual_s": "-0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AS02",
        "network": "UR",
        "latitude": 53.5542,
        "longitude": -1.9857,
        "channel": "H?Z",
        "isc_distance_deg": 146.49,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.41Z",
        "isc_residual_s": "0.8",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GIRR",
        "network": "IR",
        "latitude": 46.9551,
        "longitude": 26.5008,
        "channel": "???",
        "isc_distance_deg": 146.5,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.47Z",
        "isc_residual_s": "0.7",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KIRS",
        "network": "IR",
        "latitude": 39.1329,
        "longitude": 33.9168,
        "channel": "???",
        "isc_distance_deg": 146.5,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:46.72Z",
        "isc_residual_s": "-0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WLF1",
        "network": "IR",
        "latitude": 53.2894,
        "longitude": -4.3966,
        "channel": "H?Z",
        "isc_distance_deg": 146.54,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:46.98Z",
        "isc_residual_s": "0.4",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TATR",
        "network": "IR",
        "latitude": 45.5977,
        "longitude": 28.0499,
        "channel": "???",
        "isc_distance_deg": 146.63,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.38Z",
        "isc_residual_s": "0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BURAR",
        "network": "IR",
        "latitude": 47.6148,
        "longitude": 25.2168,
        "channel": "???",
        "isc_distance_deg": 146.63,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.63Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BURAR",
        "network": "IR",
        "latitude": 47.6148,
        "longitude": 25.2168,
        "channel": "S?Z",
        "isc_distance_deg": 146.63,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:47.50Z",
        "isc_residual_s": "0.4",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LBWR",
        "network": "IR",
        "latitude": 53.4016,
        "longitude": -1.7248,
        "channel": "H?Z",
        "isc_distance_deg": 146.66,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:46.69Z",
        "isc_residual_s": "-0.2",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LMK",
        "network": "IR",
        "latitude": 53.4569,
        "longitude": -0.3266,
        "channel": "H?Z",
        "isc_distance_deg": 146.68,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.69Z",
        "isc_residual_s": "0.8",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HWQ",
        "network": "IR",
        "latitude": 34.278,
        "longitude": 35.9463,
        "channel": "B?Z",
        "isc_distance_deg": 146.68,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.39Z",
        "isc_residual_s": "-0.3",
        "reporter": "GRAL",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BIZ",
        "network": "IR",
        "latitude": 46.9392,
        "longitude": 26.1042,
        "channel": "???",
        "isc_distance_deg": 146.7,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.97Z",
        "isc_residual_s": "0.8",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OJC",
        "network": "IR",
        "latitude": 50.2195,
        "longitude": 19.7984,
        "channel": "??Z",
        "isc_distance_deg": 146.73,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:47.33Z",
        "isc_residual_s": "0.2",
        "reporter": "WAR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OJC",
        "network": "IR",
        "latitude": 50.2195,
        "longitude": 19.7984,
        "channel": "S?Z",
        "isc_distance_deg": 146.73,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:47.30Z",
        "isc_residual_s": "0.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TESR",
        "network": "IR",
        "latitude": 46.5118,
        "longitude": 26.6489,
        "channel": "???",
        "isc_distance_deg": 146.73,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.58Z",
        "isc_residual_s": "0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NEGRR",
        "network": "IR",
        "latitude": 45.5373,
        "longitude": 27.8771,
        "channel": "???",
        "isc_distance_deg": 146.76,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.75Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SCHL",
        "network": "IR",
        "latitude": 45.5007,
        "longitude": 27.8302,
        "channel": "???",
        "isc_distance_deg": 146.81,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.78Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "IZVR",
        "network": "IR",
        "latitude": 45.5432,
        "longitude": 27.7672,
        "channel": "???",
        "isc_distance_deg": 146.82,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.59Z",
        "isc_residual_s": "0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SCHLR",
        "network": "IR",
        "latitude": 45.4702,
        "longitude": 27.8314,
        "channel": "???",
        "isc_distance_deg": 146.83,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.94Z",
        "isc_residual_s": "0.6",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TUDR",
        "network": "IR",
        "latitude": 45.5939,
        "longitude": 27.6687,
        "channel": "???",
        "isc_distance_deg": 146.83,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.18Z",
        "isc_residual_s": "0.8",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CFR",
        "network": "IR",
        "latitude": 45.178,
        "longitude": 28.1362,
        "channel": "???",
        "isc_distance_deg": 146.86,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.18Z",
        "isc_residual_s": "-0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CFR",
        "network": "IR",
        "latitude": 45.178,
        "longitude": 28.1362,
        "channel": "S?Z",
        "isc_distance_deg": 146.86,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:47.00Z",
        "isc_residual_s": "-0.4",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VAL",
        "network": "IR",
        "latitude": 51.9394,
        "longitude": -10.2442,
        "channel": "???",
        "isc_distance_deg": 146.87,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:47.20Z",
        "isc_residual_s": "-0.0",
        "reporter": "DIAS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KOLS",
        "network": "IR",
        "latitude": 48.9333,
        "longitude": 22.2731,
        "channel": "S?Z",
        "isc_distance_deg": 146.88,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.50Z",
        "isc_residual_s": "1.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KOLS",
        "network": "IR",
        "latitude": 48.9333,
        "longitude": 22.2731,
        "channel": "???",
        "isc_distance_deg": 146.88,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:48.49Z",
        "isc_residual_s": "1.1",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KOLS",
        "network": "IR",
        "latitude": 48.9333,
        "longitude": 22.2731,
        "channel": "???",
        "isc_distance_deg": 146.88,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:51.94Z",
        "isc_residual_s": "-0.3",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "H10S3",
        "network": "IR",
        "latitude": -8.9527,
        "longitude": -14.6629,
        "channel": "???",
        "isc_distance_deg": 146.88,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:52.26Z",
        "isc_residual_s": "-0.7",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "H10S2",
        "network": "IR",
        "latitude": -8.9591,
        "longitude": -14.6453,
        "channel": "???",
        "isc_distance_deg": 146.89,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:52.25Z",
        "isc_residual_s": "-0.7",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TPGR",
        "network": "IR",
        "latitude": 44.8565,
        "longitude": 28.4196,
        "channel": "???",
        "isc_distance_deg": 146.91,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.07Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PANC",
        "network": "IR",
        "latitude": 45.8723,
        "longitude": 27.1477,
        "channel": "???",
        "isc_distance_deg": 146.92,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.55Z",
        "isc_residual_s": "1.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ANTO",
        "network": "IR",
        "latitude": 39.8689,
        "longitude": 32.7936,
        "channel": "???",
        "isc_distance_deg": 146.95,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.78Z",
        "isc_residual_s": "-0.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ANTO",
        "network": "IR",
        "latitude": 39.8689,
        "longitude": 32.7936,
        "channel": "S?Z",
        "isc_distance_deg": 146.95,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:47.80Z",
        "isc_residual_s": "-0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ANTO",
        "network": "IR",
        "latitude": 39.8689,
        "longitude": 32.7936,
        "channel": "???",
        "isc_distance_deg": 146.95,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:46.60Z",
        "isc_residual_s": "-1.3",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RCY",
        "network": "IR",
        "latitude": 33.4965,
        "longitude": 35.8357,
        "channel": "S?Z",
        "isc_distance_deg": 146.98,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.36Z",
        "isc_residual_s": "0.1",
        "reporter": "GRAL",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ONER",
        "network": "IR",
        "latitude": 46.3349,
        "longitude": 26.394,
        "channel": "???",
        "isc_distance_deg": 146.98,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.73Z",
        "isc_residual_s": "1.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FLTG",
        "network": "IR",
        "latitude": 52.3306,
        "longitude": 11.2372,
        "channel": "???",
        "isc_distance_deg": 147.01,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:47.64Z",
        "isc_residual_s": "0.2",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FLTG",
        "network": "IR",
        "latitude": 52.3306,
        "longitude": 11.2372,
        "channel": "???",
        "isc_distance_deg": 147.01,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:51.81Z",
        "isc_residual_s": "-0.5",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BHL",
        "network": "IR",
        "latitude": 33.9042,
        "longitude": 35.6542,
        "channel": "B?Z",
        "isc_distance_deg": 147.02,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.93Z",
        "isc_residual_s": "-0.3",
        "reporter": "GRAL",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FOEL",
        "network": "IR",
        "latitude": 52.8898,
        "longitude": -3.2012,
        "channel": "H?Z",
        "isc_distance_deg": 147.06,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.48Z",
        "isc_residual_s": "0.9",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UZHM",
        "network": "IR",
        "latitude": 48.629,
        "longitude": 22.291,
        "channel": "???",
        "isc_distance_deg": 147.12,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:48.10Z",
        "isc_residual_s": "0.4",
        "reporter": "MCSM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "IWEX",
        "network": "IR",
        "latitude": 52.3743,
        "longitude": -6.7745,
        "channel": "???",
        "isc_distance_deg": 147.12,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:48.10Z",
        "isc_residual_s": "0.5",
        "reporter": "DIAS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VRI",
        "network": "IR",
        "latitude": 45.8657,
        "longitude": 26.7277,
        "channel": "???",
        "isc_distance_deg": 147.14,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.28Z",
        "isc_residual_s": "0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VRI",
        "network": "IR",
        "latitude": 45.8657,
        "longitude": 26.7277,
        "channel": "S?Z",
        "isc_distance_deg": 147.14,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.00Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KDZE",
        "network": "IR",
        "latitude": 41.3132,
        "longitude": 31.4429,
        "channel": "???",
        "isc_distance_deg": 147.15,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.01Z",
        "isc_residual_s": "-1.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "JOSR",
        "network": "IR",
        "latitude": 46.7059,
        "longitude": 25.5154,
        "channel": "???",
        "isc_distance_deg": 147.15,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:47.59Z",
        "isc_residual_s": "-0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TIRR",
        "network": "IR",
        "latitude": 44.4581,
        "longitude": 28.4128,
        "channel": "???",
        "isc_distance_deg": 147.16,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.38Z",
        "isc_residual_s": "0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TIRR",
        "network": "IR",
        "latitude": 44.4581,
        "longitude": 28.4128,
        "channel": "S?Z",
        "isc_distance_deg": 147.16,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.40Z",
        "isc_residual_s": "0.4",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PLOR",
        "network": "IR",
        "latitude": 45.8512,
        "longitude": 26.6498,
        "channel": "???",
        "isc_distance_deg": 147.19,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.05Z",
        "isc_residual_s": "0.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PLOR",
        "network": "IR",
        "latitude": 45.8512,
        "longitude": 26.6498,
        "channel": "S?Z",
        "isc_distance_deg": 147.19,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.10Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NIE",
        "network": "IR",
        "latitude": 49.4182,
        "longitude": 20.2996,
        "channel": "??Z",
        "isc_distance_deg": 147.23,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:49.00Z",
        "isc_residual_s": "1.0",
        "reporter": "WAR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NIE",
        "network": "IR",
        "latitude": 49.4182,
        "longitude": 20.2996,
        "channel": "S?Z",
        "isc_distance_deg": 147.23,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:49.00Z",
        "isc_residual_s": "1.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SWQJ",
        "network": "IR",
        "latitude": 31.4046,
        "longitude": 36.0995,
        "channel": "???",
        "isc_distance_deg": 147.25,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.60Z",
        "isc_residual_s": "-0.0",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SAHR",
        "network": "IR",
        "latitude": 45.7266,
        "longitude": 26.6855,
        "channel": "???",
        "isc_distance_deg": 147.26,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.56Z",
        "isc_residual_s": "0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KSP",
        "network": "IR",
        "latitude": 50.8428,
        "longitude": 16.2931,
        "channel": "??Z",
        "isc_distance_deg": 147.27,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:48.73Z",
        "isc_residual_s": "0.8",
        "reporter": "WAR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KSP",
        "network": "IR",
        "latitude": 50.8428,
        "longitude": 16.2931,
        "channel": "S?Z",
        "isc_distance_deg": 147.27,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.70Z",
        "isc_residual_s": "0.7",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "EFOR",
        "network": "IR",
        "latitude": 44.0795,
        "longitude": 28.6323,
        "channel": "???",
        "isc_distance_deg": 147.28,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.74Z",
        "isc_residual_s": "0.6",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HARR",
        "network": "IR",
        "latitude": 44.69,
        "longitude": 27.931,
        "channel": "???",
        "isc_distance_deg": 147.28,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.30Z",
        "isc_residual_s": "0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HARR",
        "network": "IR",
        "latitude": 44.69,
        "longitude": 27.931,
        "channel": "S?Z",
        "isc_distance_deg": 147.28,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.30Z",
        "isc_residual_s": "0.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ASSE",
        "network": "IR",
        "latitude": 52.1303,
        "longitude": 10.6729,
        "channel": "???",
        "isc_distance_deg": 147.3,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:48.12Z",
        "isc_residual_s": "0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ASSE",
        "network": "IR",
        "latitude": 52.1303,
        "longitude": 10.6729,
        "channel": "???",
        "isc_distance_deg": 147.3,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:52.80Z",
        "isc_residual_s": "-0.1",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TLBR",
        "network": "IR",
        "latitude": 44.5445,
        "longitude": 28.0467,
        "channel": "???",
        "isc_distance_deg": 147.31,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.60Z",
        "isc_residual_s": "0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MFTR",
        "network": "IR",
        "latitude": 44.1779,
        "longitude": 28.4224,
        "channel": "???",
        "isc_distance_deg": 147.34,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.50Z",
        "isc_residual_s": "0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GRER",
        "network": "IR",
        "latitude": 45.3801,
        "longitude": 26.9746,
        "channel": "???",
        "isc_distance_deg": 147.34,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.14Z",
        "isc_residual_s": "0.9",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CWF",
        "network": "IR",
        "latitude": 52.7383,
        "longitude": -1.3072,
        "channel": "H?Z",
        "isc_distance_deg": 147.35,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.88Z",
        "isc_residual_s": "0.9",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BMR",
        "network": "IR",
        "latitude": 47.6736,
        "longitude": 23.496,
        "channel": "???",
        "isc_distance_deg": 147.36,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.06Z",
        "isc_residual_s": "-0.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BMR",
        "network": "IR",
        "latitude": 47.6736,
        "longitude": 23.496,
        "channel": "S?Z",
        "isc_distance_deg": 147.36,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.10Z",
        "isc_residual_s": "-0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TGSR",
        "network": "IR",
        "latitude": 45.9929,
        "longitude": 26.1152,
        "channel": "???",
        "isc_distance_deg": 147.36,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.16Z",
        "isc_residual_s": "0.8",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BISRR",
        "network": "IR",
        "latitude": 45.5481,
        "longitude": 26.7099,
        "channel": "???",
        "isc_distance_deg": 147.37,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.31Z",
        "isc_residual_s": "-0.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WACR",
        "network": "IR",
        "latitude": 52.7247,
        "longitude": 0.6267,
        "channel": "H?Z",
        "isc_distance_deg": 147.44,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.22Z",
        "isc_residual_s": "0.1",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SPC",
        "network": "IR",
        "latitude": 49.1889,
        "longitude": 20.245,
        "channel": "???",
        "isc_distance_deg": 147.44,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:49.65Z",
        "isc_residual_s": "1.1",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SPC",
        "network": "IR",
        "latitude": 49.1889,
        "longitude": 20.245,
        "channel": "???",
        "isc_distance_deg": 147.44,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:53.65Z",
        "isc_residual_s": "0.1",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MMAI",
        "network": "IR",
        "latitude": 33.0153,
        "longitude": 35.4031,
        "channel": "???",
        "isc_distance_deg": 147.45,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:53.62Z",
        "isc_residual_s": "-0.4",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HLM1",
        "network": "IR",
        "latitude": 52.5184,
        "longitude": -2.8807,
        "channel": "H?Z",
        "isc_distance_deg": 147.45,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.01Z",
        "isc_residual_s": "0.8",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MANR",
        "network": "IR",
        "latitude": 43.8168,
        "longitude": 28.5876,
        "channel": "???",
        "isc_distance_deg": 147.47,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.93Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "QRNJ",
        "network": "IR",
        "latitude": 32.335,
        "longitude": 35.575,
        "channel": "???",
        "isc_distance_deg": 147.48,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.60Z",
        "isc_residual_s": "1.8",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BALJ",
        "network": "GE",
        "latitude": 32.0873,
        "longitude": 35.64,
        "channel": "???",
        "isc_distance_deg": 147.48,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.40Z",
        "isc_residual_s": "1.5",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CVSR",
        "network": "IR",
        "latitude": 45.7944,
        "longitude": 26.1238,
        "channel": "???",
        "isc_distance_deg": 147.5,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.55Z",
        "isc_residual_s": "0.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "IBBN",
        "network": "IR",
        "latitude": 52.3072,
        "longitude": 7.7566,
        "channel": "???",
        "isc_distance_deg": 147.53,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:48.58Z",
        "isc_residual_s": "0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "IBBN",
        "network": "IR",
        "latitude": 52.3072,
        "longitude": 7.7566,
        "channel": "???",
        "isc_distance_deg": 147.53,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:53.42Z",
        "isc_residual_s": "0.0",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OSTC",
        "network": "IR",
        "latitude": 50.5565,
        "longitude": 16.2156,
        "channel": "S?Z",
        "isc_distance_deg": 147.55,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.80Z",
        "isc_residual_s": "0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OSTC",
        "network": "IR",
        "latitude": 50.5565,
        "longitude": 16.2156,
        "channel": "???",
        "isc_distance_deg": 147.55,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:48.80Z",
        "isc_residual_s": "0.3",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DALR",
        "network": "IR",
        "latitude": 45.3629,
        "longitude": 26.5966,
        "channel": "???",
        "isc_distance_deg": 147.55,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.94Z",
        "isc_residual_s": "0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ODSR",
        "network": "IR",
        "latitude": 46.2969,
        "longitude": 25.2919,
        "channel": "???",
        "isc_distance_deg": 147.55,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.06Z",
        "isc_residual_s": "1.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CHVC",
        "network": "IR",
        "latitude": 50.5881,
        "longitude": 16.0547,
        "channel": "S?Z",
        "isc_distance_deg": 147.56,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.70Z",
        "isc_residual_s": "0.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CHVC",
        "network": "IR",
        "latitude": 50.5881,
        "longitude": 16.0547,
        "channel": "???",
        "isc_distance_deg": 147.56,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:48.70Z",
        "isc_residual_s": "0.2",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KARJ",
        "network": "IR",
        "latitude": 32.0169,
        "longitude": 35.5508,
        "channel": "???",
        "isc_distance_deg": 147.57,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:54.40Z",
        "isc_residual_s": "0.3",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OKC",
        "network": "IR",
        "latitude": 49.8375,
        "longitude": 18.1472,
        "channel": "S?Z",
        "isc_distance_deg": 147.6,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.60Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OKC",
        "network": "IR",
        "latitude": 49.8375,
        "longitude": 18.1472,
        "channel": "???",
        "isc_distance_deg": 147.6,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:48.60Z",
        "isc_residual_s": "0.1",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "IBZR",
        "network": "IR",
        "latitude": 45.6681,
        "longitude": 26.0567,
        "channel": "???",
        "isc_distance_deg": 147.62,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.58Z",
        "isc_residual_s": "0.8",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CLZ",
        "network": "IR",
        "latitude": 51.8416,
        "longitude": 10.3724,
        "channel": "???",
        "isc_distance_deg": 147.63,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:48.91Z",
        "isc_residual_s": "0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CLZ",
        "network": "IR",
        "latitude": 51.8416,
        "longitude": 10.3724,
        "channel": "???",
        "isc_distance_deg": 147.63,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:53.94Z",
        "isc_residual_s": "0.2",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CLL",
        "network": "IR",
        "latitude": 51.3077,
        "longitude": 13.0026,
        "channel": "S?Z",
        "isc_distance_deg": 147.65,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.60Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CLL",
        "network": "IR",
        "latitude": 51.3077,
        "longitude": 13.0026,
        "channel": "???",
        "isc_distance_deg": 147.65,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:48.91Z",
        "isc_residual_s": "0.4",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CLL",
        "network": "IR",
        "latitude": 51.3077,
        "longitude": 13.0026,
        "channel": "???",
        "isc_distance_deg": 147.65,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:53.68Z",
        "isc_residual_s": "-0.0",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CLL",
        "network": "IR",
        "latitude": 51.3077,
        "longitude": 13.0026,
        "channel": "??Z",
        "isc_distance_deg": 147.65,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:48.60Z",
        "isc_residual_s": "0.1",
        "reporter": "CLL",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CLL",
        "network": "IR",
        "latitude": 51.3077,
        "longitude": 13.0026,
        "channel": "??Z",
        "isc_distance_deg": 147.65,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPab",
        "arrival_utc": "2022-11-12T07:27:56.40Z",
        "isc_residual_s": "2.7",
        "reporter": "CLL",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UPC",
        "network": "IR",
        "latitude": 50.5074,
        "longitude": 16.0121,
        "channel": "S?Z",
        "isc_distance_deg": 147.65,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.70Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UPC",
        "network": "IR",
        "latitude": 50.5074,
        "longitude": 16.0121,
        "channel": "???",
        "isc_distance_deg": 147.65,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:48.70Z",
        "isc_residual_s": "0.1",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PGOR",
        "network": "IR",
        "latitude": 44.908,
        "longitude": 26.9846,
        "channel": "???",
        "isc_distance_deg": 147.65,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.75Z",
        "isc_residual_s": "1.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UJAP",
        "network": "GE",
        "latitude": 31.952,
        "longitude": 35.4644,
        "channel": "???",
        "isc_distance_deg": 147.66,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.70Z",
        "isc_residual_s": "-0.6",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AMRR",
        "network": "IR",
        "latitude": 44.6103,
        "longitude": 27.3354,
        "channel": "???",
        "isc_distance_deg": 147.66,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.97Z",
        "isc_residual_s": "0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AMRR",
        "network": "IR",
        "latitude": 44.6103,
        "longitude": 27.3354,
        "channel": "S?Z",
        "isc_distance_deg": 147.66,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:49.00Z",
        "isc_residual_s": "0.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NEHR",
        "network": "IR",
        "latitude": 45.4272,
        "longitude": 26.2952,
        "channel": "???",
        "isc_distance_deg": 147.66,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.05Z",
        "isc_residual_s": "0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DPC",
        "network": "IR",
        "latitude": 50.3502,
        "longitude": 16.3222,
        "channel": "S?Z",
        "isc_distance_deg": 147.7,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:49.30Z",
        "isc_residual_s": "0.5",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DPC",
        "network": "IR",
        "latitude": 50.3502,
        "longitude": 16.3222,
        "channel": "???",
        "isc_distance_deg": 147.7,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:49.30Z",
        "isc_residual_s": "0.5",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DPC",
        "network": "IR",
        "latitude": 50.3502,
        "longitude": 16.3222,
        "channel": "???",
        "isc_distance_deg": 147.7,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPBC",
        "arrival_utc": "2022-11-12T07:27:53.50Z",
        "isc_residual_s": "-0.4",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ICOR",
        "network": "IR",
        "latitude": 44.1168,
        "longitude": 27.8009,
        "channel": "???",
        "isc_distance_deg": 147.72,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.20Z",
        "isc_residual_s": "0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DOPR",
        "network": "IR",
        "latitude": 45.9674,
        "longitude": 25.3887,
        "channel": "???",
        "isc_distance_deg": 147.74,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.42Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ISR",
        "network": "IR",
        "latitude": 45.1188,
        "longitude": 26.5431,
        "channel": "???",
        "isc_distance_deg": 147.74,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.99Z",
        "isc_residual_s": "0.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ISR",
        "network": "IR",
        "latitude": 45.1188,
        "longitude": 26.5431,
        "channel": "S?Z",
        "isc_distance_deg": 147.74,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:49.00Z",
        "isc_residual_s": "0.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TGMR",
        "network": "IR",
        "latitude": 46.5284,
        "longitude": 24.5222,
        "channel": "???",
        "isc_distance_deg": 147.75,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.01Z",
        "isc_residual_s": "1.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LANS",
        "network": "IR",
        "latitude": 49.1505,
        "longitude": 19.4678,
        "channel": "S?Z",
        "isc_distance_deg": 147.75,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:49.70Z",
        "isc_residual_s": "0.8",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LANS",
        "network": "IR",
        "latitude": 49.1505,
        "longitude": 19.4678,
        "channel": "???",
        "isc_distance_deg": 147.75,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:49.70Z",
        "isc_residual_s": "0.8",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LANS",
        "network": "IR",
        "latitude": 49.1505,
        "longitude": 19.4678,
        "channel": "???",
        "isc_distance_deg": 147.75,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:54.47Z",
        "isc_residual_s": "0.4",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SAHE",
        "network": "IR",
        "latitude": 40.8525,
        "longitude": 30.8536,
        "channel": "???",
        "isc_distance_deg": 147.76,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.09Z",
        "isc_residual_s": "-0.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CEIR",
        "network": "IR",
        "latitude": 47.6865,
        "longitude": 22.496,
        "channel": "???",
        "isc_distance_deg": 147.78,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.73Z",
        "isc_residual_s": "1.9",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MLR",
        "network": "IR",
        "latitude": 45.4909,
        "longitude": 25.945,
        "channel": "???",
        "isc_distance_deg": 147.8,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.71Z",
        "isc_residual_s": "-0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MLR",
        "network": "IR",
        "latitude": 45.4909,
        "longitude": 25.945,
        "channel": "???",
        "isc_distance_deg": 147.8,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.39Z",
        "isc_residual_s": "0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MLR",
        "network": "IR",
        "latitude": 45.4909,
        "longitude": 25.945,
        "channel": "S?Z",
        "isc_distance_deg": 147.8,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:48.70Z",
        "isc_residual_s": "-0.5",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LISJ",
        "network": "IR",
        "latitude": 31.24,
        "longitude": 35.481,
        "channel": "???",
        "isc_distance_deg": 147.8,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.20Z",
        "isc_residual_s": "1.9",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HSNJ",
        "network": "IR",
        "latitude": 30.256,
        "longitude": 35.687,
        "channel": "???",
        "isc_distance_deg": 147.81,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.40Z",
        "isc_residual_s": "1.7",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RSBS",
        "network": "IR",
        "latitude": 51.953,
        "longitude": -4.7448,
        "channel": "H?Z",
        "isc_distance_deg": 147.82,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:48.68Z",
        "isc_residual_s": "-0.1",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KALB",
        "network": "IR",
        "latitude": 43.4059,
        "longitude": 28.4162,
        "channel": "???",
        "isc_distance_deg": 147.82,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.27Z",
        "isc_residual_s": "1.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PSN",
        "network": "IR",
        "latitude": 43.6375,
        "longitude": 28.1359,
        "channel": "???",
        "isc_distance_deg": 147.84,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:54.60Z",
        "isc_residual_s": "0.3",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BRG",
        "network": "IR",
        "latitude": 50.8732,
        "longitude": 13.9428,
        "channel": "S?Z",
        "isc_distance_deg": 147.84,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:49.20Z",
        "isc_residual_s": "0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BRG",
        "network": "IR",
        "latitude": 50.8732,
        "longitude": 13.9428,
        "channel": "???",
        "isc_distance_deg": 147.84,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:49.21Z",
        "isc_residual_s": "0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BRG",
        "network": "IR",
        "latitude": 50.8732,
        "longitude": 13.9428,
        "channel": "???",
        "isc_distance_deg": 147.84,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:54.36Z",
        "isc_residual_s": "0.2",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MORC",
        "network": "IR",
        "latitude": 49.7768,
        "longitude": 17.5425,
        "channel": "???",
        "isc_distance_deg": 147.85,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.55Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MORC",
        "network": "IR",
        "latitude": 49.7768,
        "longitude": 17.5425,
        "channel": "S?Z",
        "isc_distance_deg": 147.85,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:49.30Z",
        "isc_residual_s": "0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MORC",
        "network": "IR",
        "latitude": 49.7768,
        "longitude": 17.5425,
        "channel": "???",
        "isc_distance_deg": 147.85,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:49.20Z",
        "isc_residual_s": "0.2",
        "reporter": "IPEC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SALP",
        "network": "GE",
        "latitude": 32.0686,
        "longitude": 35.1914,
        "channel": "???",
        "isc_distance_deg": 147.86,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.50Z",
        "isc_residual_s": "0.9",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MSBI",
        "network": "GE",
        "latitude": 31.3126,
        "longitude": 35.358,
        "channel": "???",
        "isc_distance_deg": 147.89,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.50Z",
        "isc_residual_s": "1.0",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BEDF",
        "network": "IR",
        "latitude": 52.2505,
        "longitude": 1.2591,
        "channel": "H?Z",
        "isc_distance_deg": 147.92,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.87Z",
        "isc_residual_s": "0.9",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "H10N3",
        "network": "IR",
        "latitude": -7.8409,
        "longitude": -14.5017,
        "channel": "???",
        "isc_distance_deg": 147.93,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:55.19Z",
        "isc_residual_s": "0.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "H10N1",
        "network": "IR",
        "latitude": -7.8457,
        "longitude": -14.4802,
        "channel": "???",
        "isc_distance_deg": 147.93,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:55.02Z",
        "isc_residual_s": "-0.0",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "H10N2",
        "network": "IR",
        "latitude": -7.8278,
        "longitude": -14.4875,
        "channel": "???",
        "isc_distance_deg": 147.94,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:55.32Z",
        "isc_residual_s": "0.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MCH1",
        "network": "IR",
        "latitude": 51.9974,
        "longitude": -2.9983,
        "channel": "H?Z",
        "isc_distance_deg": 147.96,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.58Z",
        "isc_residual_s": "0.6",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GTTG",
        "network": "IR",
        "latitude": 51.5451,
        "longitude": 9.9647,
        "channel": "???",
        "isc_distance_deg": 147.98,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:49.40Z",
        "isc_residual_s": "0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GTTG",
        "network": "IR",
        "latitude": 51.5451,
        "longitude": 9.9647,
        "channel": "???",
        "isc_distance_deg": 147.98,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:54.62Z",
        "isc_residual_s": "0.2",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NEUB",
        "network": "IR",
        "latitude": 51.2083,
        "longitude": 11.7752,
        "channel": "???",
        "isc_distance_deg": 147.99,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:49.44Z",
        "isc_residual_s": "0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NEUB",
        "network": "IR",
        "latitude": 51.2083,
        "longitude": 11.7752,
        "channel": "???",
        "isc_distance_deg": 147.99,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:54.68Z",
        "isc_residual_s": "0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MESR",
        "network": "IR",
        "latitude": 47.1499,
        "longitude": 22.9494,
        "channel": "???",
        "isc_distance_deg": 148.0,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.46Z",
        "isc_residual_s": "0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PVCC",
        "network": "IR",
        "latitude": 50.5282,
        "longitude": 14.5689,
        "channel": "???",
        "isc_distance_deg": 148.01,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:49.60Z",
        "isc_residual_s": "0.4",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PVCC",
        "network": "IR",
        "latitude": 50.5282,
        "longitude": 14.5689,
        "channel": "???",
        "isc_distance_deg": 148.01,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPBC",
        "arrival_utc": "2022-11-12T07:27:53.90Z",
        "isc_residual_s": "-0.6",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LEHL",
        "network": "IR",
        "latitude": 44.4739,
        "longitude": 26.8194,
        "channel": "???",
        "isc_distance_deg": 148.03,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.83Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CJR",
        "network": "IR",
        "latitude": 46.7133,
        "longitude": 23.5981,
        "channel": "???",
        "isc_distance_deg": 148.04,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.44Z",
        "isc_residual_s": "0.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ELMS",
        "network": "IR",
        "latitude": 52.0933,
        "longitude": 0.9895,
        "channel": "H?Z",
        "isc_distance_deg": 148.08,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.87Z",
        "isc_residual_s": "0.7",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CICN",
        "network": "IR",
        "latitude": 44.197,
        "longitude": 27.0261,
        "channel": "???",
        "isc_distance_deg": 148.1,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.26Z",
        "isc_residual_s": "0.8",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MDB",
        "network": "IR",
        "latitude": 46.1417,
        "longitude": 24.3806,
        "channel": "???",
        "isc_distance_deg": 148.1,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.16Z",
        "isc_residual_s": "0.7",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RICC",
        "network": "IR",
        "latitude": 50.5439,
        "longitude": 14.0978,
        "channel": "???",
        "isc_distance_deg": 148.11,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:49.40Z",
        "isc_residual_s": "0.1",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RICC",
        "network": "IR",
        "latitude": 50.5439,
        "longitude": 14.0978,
        "channel": "???",
        "isc_distance_deg": 148.11,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPBC",
        "arrival_utc": "2022-11-12T07:27:54.30Z",
        "isc_residual_s": "-0.4",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MONM",
        "network": "IR",
        "latitude": 51.8396,
        "longitude": -2.8054,
        "channel": "H?Z",
        "isc_distance_deg": 148.13,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.56Z",
        "isc_residual_s": "0.3",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PLAR",
        "network": "IR",
        "latitude": 44.9142,
        "longitude": 26.0274,
        "channel": "???",
        "isc_distance_deg": 148.15,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.75Z",
        "isc_residual_s": "1.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SULR",
        "network": "IR",
        "latitude": 44.6785,
        "longitude": 26.2551,
        "channel": "???",
        "isc_distance_deg": 148.19,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.71Z",
        "isc_residual_s": "0.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "STRD",
        "network": "IR",
        "latitude": 51.7763,
        "longitude": -2.1639,
        "channel": "H?Z",
        "isc_distance_deg": 148.25,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.10Z",
        "isc_residual_s": "0.6",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VOIR",
        "network": "IR",
        "latitude": 45.4371,
        "longitude": 25.0496,
        "channel": "???",
        "isc_distance_deg": 148.28,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.81Z",
        "isc_residual_s": "0.9",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MARR",
        "network": "IR",
        "latitude": 46.676,
        "longitude": 23.1189,
        "channel": "???",
        "isc_distance_deg": 148.28,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:49.98Z",
        "isc_residual_s": "0.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DRGR",
        "network": "IR",
        "latitude": 46.7916,
        "longitude": 22.7111,
        "channel": "???",
        "isc_distance_deg": 148.38,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.32Z",
        "isc_residual_s": "0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DRGR",
        "network": "IR",
        "latitude": 46.7916,
        "longitude": 22.7111,
        "channel": "S?Z",
        "isc_distance_deg": 148.38,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:49.30Z",
        "isc_residual_s": "-0.7",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SILT",
        "network": "IR",
        "latitude": 41.153,
        "longitude": 29.643,
        "channel": "???",
        "isc_distance_deg": 148.38,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.40Z",
        "isc_residual_s": "0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MTUR",
        "network": "IR",
        "latitude": 45.2261,
        "longitude": 25.063,
        "channel": "???",
        "isc_distance_deg": 148.43,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.50Z",
        "isc_residual_s": "0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BUG",
        "network": "IR",
        "latitude": 51.4406,
        "longitude": 7.2693,
        "channel": "???",
        "isc_distance_deg": 148.44,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:49.93Z",
        "isc_residual_s": "0.1",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BUG",
        "network": "IR",
        "latitude": 51.4406,
        "longitude": 7.2693,
        "channel": "???",
        "isc_distance_deg": 148.44,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:55.58Z",
        "isc_residual_s": "0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AQBJ",
        "network": "IR",
        "latitude": 29.728,
        "longitude": 35.05,
        "channel": "???",
        "isc_distance_deg": 148.45,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.80Z",
        "isc_residual_s": "1.3",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PRA",
        "network": "IR",
        "latitude": 50.0703,
        "longitude": 14.4331,
        "channel": "???",
        "isc_distance_deg": 148.46,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:49.90Z",
        "isc_residual_s": "-0.0",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PRA",
        "network": "IR",
        "latitude": 50.0703,
        "longitude": 14.4331,
        "channel": "???",
        "isc_distance_deg": 148.46,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPBC",
        "arrival_utc": "2022-11-12T07:27:54.90Z",
        "isc_residual_s": "-0.5",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TNR",
        "network": "IR",
        "latitude": 45.652,
        "longitude": 24.273,
        "channel": "???",
        "isc_distance_deg": 148.51,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.17Z",
        "isc_residual_s": "-0.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PRU",
        "network": "IR",
        "latitude": 49.9883,
        "longitude": 14.5417,
        "channel": "???",
        "isc_distance_deg": 148.51,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:50.00Z",
        "isc_residual_s": "0.0",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BUC1",
        "network": "IR",
        "latitude": 44.3479,
        "longitude": 26.0281,
        "channel": "???",
        "isc_distance_deg": 148.53,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.01Z",
        "isc_residual_s": "-0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ARR",
        "network": "IR",
        "latitude": 45.3675,
        "longitude": 24.6336,
        "channel": "???",
        "isc_distance_deg": 148.54,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.35Z",
        "isc_residual_s": "0.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SWN1",
        "network": "IR",
        "latitude": 51.5131,
        "longitude": -1.8004,
        "channel": "H?Z",
        "isc_distance_deg": 148.54,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.66Z",
        "isc_residual_s": "0.7",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KASTN",
        "network": "IR",
        "latitude": 51.2027,
        "longitude": 8.4162,
        "channel": "???",
        "isc_distance_deg": 148.54,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:50.39Z",
        "isc_residual_s": "0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KASTN",
        "network": "IR",
        "latitude": 51.2027,
        "longitude": 8.4162,
        "channel": "???",
        "isc_distance_deg": 148.54,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:56.10Z",
        "isc_residual_s": "0.5",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BORR",
        "network": "IR",
        "latitude": 47.0902,
        "longitude": 21.7856,
        "channel": "???",
        "isc_distance_deg": 148.54,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.94Z",
        "isc_residual_s": "0.8",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "EIL",
        "network": "GE",
        "latitude": 29.6699,
        "longitude": 34.9512,
        "channel": "???",
        "isc_distance_deg": 148.54,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.20Z",
        "isc_residual_s": "0.5",
        "reporter": "JSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PRD",
        "network": "IR",
        "latitude": 43.1602,
        "longitude": 27.4099,
        "channel": "???",
        "isc_distance_deg": 148.55,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:56.20Z",
        "isc_residual_s": "0.4",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VRAC",
        "network": "IR",
        "latitude": 49.3083,
        "longitude": 16.5935,
        "channel": "???",
        "isc_distance_deg": 148.55,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.79Z",
        "isc_residual_s": "0.7",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VRAC",
        "network": "IR",
        "latitude": 49.3083,
        "longitude": 16.5935,
        "channel": "???",
        "isc_distance_deg": 148.55,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:50.00Z",
        "isc_residual_s": "-0.1",
        "reporter": "IPEC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MOX",
        "network": "IR",
        "latitude": 50.6447,
        "longitude": 11.6156,
        "channel": "???",
        "isc_distance_deg": 148.56,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:50.19Z",
        "isc_residual_s": "0.1",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MOX",
        "network": "IR",
        "latitude": 50.6447,
        "longitude": 11.6156,
        "channel": "???",
        "isc_distance_deg": 148.56,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:56.10Z",
        "isc_residual_s": "0.5",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PLN",
        "network": "IR",
        "latitude": 50.4851,
        "longitude": 12.1573,
        "channel": "???",
        "isc_distance_deg": 148.6,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:50.36Z",
        "isc_residual_s": "0.2",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PLN",
        "network": "IR",
        "latitude": 50.4851,
        "longitude": 12.1573,
        "channel": "???",
        "isc_distance_deg": 148.6,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:56.11Z",
        "isc_residual_s": "0.4",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TANN",
        "network": "IR",
        "latitude": 50.4149,
        "longitude": 12.4616,
        "channel": "???",
        "isc_distance_deg": 148.6,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:50.52Z",
        "isc_residual_s": "0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TANN",
        "network": "IR",
        "latitude": 50.4149,
        "longitude": 12.4616,
        "channel": "???",
        "isc_distance_deg": 148.6,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:56.26Z",
        "isc_residual_s": "0.5",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ROIA",
        "network": "IR",
        "latitude": 43.0933,
        "longitude": 27.3778,
        "channel": "???",
        "isc_distance_deg": 148.61,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:56.70Z",
        "isc_residual_s": "0.7",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "JAVC",
        "network": "IR",
        "latitude": 48.8591,
        "longitude": 17.6707,
        "channel": "???",
        "isc_distance_deg": 148.61,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:51.40Z",
        "isc_residual_s": "1.1",
        "reporter": "IPEC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PSZ",
        "network": "IR",
        "latitude": 47.9184,
        "longitude": 19.8944,
        "channel": "???",
        "isc_distance_deg": 148.63,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.54Z",
        "isc_residual_s": "0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SGRR",
        "network": "IR",
        "latitude": 44.2228,
        "longitude": 25.9743,
        "channel": "???",
        "isc_distance_deg": 148.65,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.66Z",
        "isc_residual_s": "0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KLYT",
        "network": "IR",
        "latitude": 41.253,
        "longitude": 29.042,
        "channel": "???",
        "isc_distance_deg": 148.7,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.16Z",
        "isc_residual_s": "-0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "STFAR",
        "network": "IR",
        "latitude": 44.8629,
        "longitude": 24.9609,
        "channel": "???",
        "isc_distance_deg": 148.73,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.79Z",
        "isc_residual_s": "0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NKC",
        "network": "IR",
        "latitude": 50.2331,
        "longitude": 12.4479,
        "channel": "???",
        "isc_distance_deg": 148.78,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:50.80Z",
        "isc_residual_s": "0.3",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NKC",
        "network": "IR",
        "latitude": 50.2331,
        "longitude": 12.4479,
        "channel": "???",
        "isc_distance_deg": 148.78,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPBC",
        "arrival_utc": "2022-11-12T07:27:55.80Z",
        "isc_residual_s": "-0.3",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HTL",
        "network": "IR",
        "latitude": 50.9944,
        "longitude": -4.485,
        "channel": "H?Z",
        "isc_distance_deg": 148.79,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.44Z",
        "isc_residual_s": "1.1",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RAZG",
        "network": "IR",
        "latitude": 43.5662,
        "longitude": 26.5079,
        "channel": "???",
        "isc_distance_deg": 148.79,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.44Z",
        "isc_residual_s": "-0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KRUC",
        "network": "IR",
        "latitude": 49.0619,
        "longitude": 16.3952,
        "channel": "???",
        "isc_distance_deg": 148.83,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:50.50Z",
        "isc_residual_s": "-0.0",
        "reporter": "IPEC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BMOL",
        "network": "IR",
        "latitude": 51.2144,
        "longitude": 5.0861,
        "channel": "BHZ",
        "isc_distance_deg": 148.86,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:50.97Z",
        "isc_residual_s": "0.5",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LOZB",
        "network": "IR",
        "latitude": 43.3701,
        "longitude": 26.593,
        "channel": "???",
        "isc_distance_deg": 148.87,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.63Z",
        "isc_residual_s": "-0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TREC",
        "network": "IR",
        "latitude": 49.2948,
        "longitude": 15.4871,
        "channel": "???",
        "isc_distance_deg": 148.89,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:50.80Z",
        "isc_residual_s": "0.1",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LOT",
        "network": "IR",
        "latitude": 45.4481,
        "longitude": 23.7694,
        "channel": "???",
        "isc_distance_deg": 148.9,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.49Z",
        "isc_residual_s": "-0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BOPT",
        "network": "IR",
        "latitude": 51.1118,
        "longitude": 5.635,
        "channel": "SHZ",
        "isc_distance_deg": 148.92,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:50.95Z",
        "isc_residual_s": "0.4",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BOPT",
        "network": "IR",
        "latitude": 51.1118,
        "longitude": 5.635,
        "channel": "SHZ",
        "isc_distance_deg": 148.92,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:56.55Z",
        "isc_residual_s": "0.3",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BRDL",
        "network": "GB",
        "latitude": 51.188,
        "longitude": -0.2652,
        "channel": "H?Z",
        "isc_distance_deg": 148.95,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.88Z",
        "isc_residual_s": "1.3",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HUMR",
        "network": "IR",
        "latitude": 44.5281,
        "longitude": 24.9804,
        "channel": "???",
        "isc_distance_deg": 148.96,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.68Z",
        "isc_residual_s": "0.8",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DEV",
        "network": "IR",
        "latitude": 45.8833,
        "longitude": 22.9033,
        "channel": "???",
        "isc_distance_deg": 148.98,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.51Z",
        "isc_residual_s": "-0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DEV",
        "network": "IR",
        "latitude": 45.8833,
        "longitude": 22.9033,
        "channel": "S?Z",
        "isc_distance_deg": 148.98,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:50.50Z",
        "isc_residual_s": "-0.4",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SMOL",
        "network": "IR",
        "latitude": 48.5139,
        "longitude": 17.4287,
        "channel": "???",
        "isc_distance_deg": 148.99,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:51.39Z",
        "isc_residual_s": "0.6",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SMOL",
        "network": "IR",
        "latitude": 48.5139,
        "longitude": 17.4287,
        "channel": "???",
        "isc_distance_deg": 148.99,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:57.16Z",
        "isc_residual_s": "0.6",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BSZH",
        "network": "IR",
        "latitude": 47.2996,
        "longitude": 20.267,
        "channel": "???",
        "isc_distance_deg": 148.99,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.34Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ELSH",
        "network": "IR",
        "latitude": 51.1482,
        "longitude": 1.1345,
        "channel": "H?Z",
        "isc_distance_deg": 149.03,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.83Z",
        "isc_residual_s": "1.1",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CTYL",
        "network": "IR",
        "latitude": 41.475,
        "longitude": 28.2891,
        "channel": "???",
        "isc_distance_deg": 149.04,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.37Z",
        "isc_residual_s": "-0.7",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KEPZ",
        "network": "IR",
        "latitude": 36.9015,
        "longitude": 31.6039,
        "channel": "???",
        "isc_distance_deg": 149.09,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.40Z",
        "isc_residual_s": "-1.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "COPA",
        "network": "IR",
        "latitude": 44.1343,
        "longitude": 25.2172,
        "channel": "???",
        "isc_distance_deg": 149.11,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:50.57Z",
        "isc_residual_s": "-0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MODS",
        "network": "IR",
        "latitude": 48.3736,
        "longitude": 17.2774,
        "channel": "???",
        "isc_distance_deg": 149.16,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:51.70Z",
        "isc_residual_s": "0.6",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MODS",
        "network": "IR",
        "latitude": 48.3736,
        "longitude": 17.2774,
        "channel": "???",
        "isc_distance_deg": 149.16,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:57.59Z",
        "isc_residual_s": "0.6",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SZH",
        "network": "IR",
        "latitude": 43.2653,
        "longitude": 25.9763,
        "channel": "???",
        "isc_distance_deg": 149.28,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:57.60Z",
        "isc_residual_s": "0.3",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HMNX",
        "network": "IR",
        "latitude": 50.8674,
        "longitude": 0.3363,
        "channel": "H?Z",
        "isc_distance_deg": 149.29,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:52.39Z",
        "isc_residual_s": "1.2",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SRO",
        "network": "IR",
        "latitude": 47.8133,
        "longitude": 18.3133,
        "channel": "???",
        "isc_distance_deg": 149.29,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:52.32Z",
        "isc_residual_s": "1.0",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SRO",
        "network": "IR",
        "latitude": 47.8133,
        "longitude": 18.3133,
        "channel": "???",
        "isc_distance_deg": 149.29,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:57.90Z",
        "isc_residual_s": "0.7",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UCC",
        "network": "IR",
        "latitude": 50.7983,
        "longitude": 4.3594,
        "channel": "BHZ",
        "isc_distance_deg": 149.31,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:51.27Z",
        "isc_residual_s": "0.1",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UCC",
        "network": "IR",
        "latitude": 50.7983,
        "longitude": 4.3594,
        "channel": "BHZ",
        "isc_distance_deg": 149.31,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:57.48Z",
        "isc_residual_s": "0.4",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AHRW",
        "network": "IR",
        "latitude": 50.541,
        "longitude": 7.076,
        "channel": "???",
        "isc_distance_deg": 149.35,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:51.45Z",
        "isc_residual_s": "0.2",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AHRW",
        "network": "IR",
        "latitude": 50.541,
        "longitude": 7.076,
        "channel": "???",
        "isc_distance_deg": 149.35,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:57.82Z",
        "isc_residual_s": "0.6",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ZST",
        "network": "IR",
        "latitude": 48.1961,
        "longitude": 17.1025,
        "channel": "S?Z",
        "isc_distance_deg": 149.37,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP2",
        "arrival_utc": "2022-11-12T07:27:59.00Z",
        "isc_residual_s": "1.6",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ZST",
        "network": "IR",
        "latitude": 48.1961,
        "longitude": 17.1025,
        "channel": "???",
        "isc_distance_deg": 149.37,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:58.95Z",
        "isc_residual_s": "1.6",
        "reporter": "BRA",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MEM",
        "network": "IR",
        "latitude": 50.6087,
        "longitude": 6.0096,
        "channel": "SHZ",
        "isc_distance_deg": 149.39,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:51.40Z",
        "isc_residual_s": "0.0",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GZR",
        "network": "IR",
        "latitude": 45.3933,
        "longitude": 22.7767,
        "channel": "???",
        "isc_distance_deg": 149.41,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.36Z",
        "isc_residual_s": "-0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GZR",
        "network": "IR",
        "latitude": 45.3933,
        "longitude": 22.7767,
        "channel": "S?Z",
        "isc_distance_deg": 149.41,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:51.00Z",
        "isc_residual_s": "-0.7",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PHSR",
        "network": "IR",
        "latitude": 41.6307,
        "longitude": 27.5239,
        "channel": "???",
        "isc_distance_deg": 149.42,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:52.24Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SURR",
        "network": "IR",
        "latitude": 45.754,
        "longitude": 22.1246,
        "channel": "???",
        "isc_distance_deg": 149.44,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.89Z",
        "isc_residual_s": "0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ABNA",
        "network": "IR",
        "latitude": 48.6705,
        "longitude": 15.4332,
        "channel": "???",
        "isc_distance_deg": 149.47,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:51.96Z",
        "isc_residual_s": "0.4",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ABNA",
        "network": "IR",
        "latitude": 48.6705,
        "longitude": 15.4332,
        "channel": "???",
        "isc_distance_deg": 149.47,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:27:57.96Z",
        "isc_residual_s": "0.4",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "JMB",
        "network": "IR",
        "latitude": 42.4911,
        "longitude": 26.5297,
        "channel": "???",
        "isc_distance_deg": 149.47,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:58.20Z",
        "isc_residual_s": "0.5",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BSKQ",
        "network": "IR",
        "latitude": 50.6487,
        "longitude": 4.0796,
        "channel": "SHZ",
        "isc_distance_deg": 149.48,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:51.64Z",
        "isc_residual_s": "0.2",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BSKQ",
        "network": "IR",
        "latitude": 50.6487,
        "longitude": 4.0796,
        "channel": "SHZ",
        "isc_distance_deg": 149.48,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:57.25Z",
        "isc_residual_s": "-0.2",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TNS",
        "network": "IR",
        "latitude": 50.2225,
        "longitude": 8.4473,
        "channel": "???",
        "isc_distance_deg": 149.5,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:52.07Z",
        "isc_residual_s": "0.4",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TNS",
        "network": "IR",
        "latitude": 50.2225,
        "longitude": 8.4473,
        "channel": "???",
        "isc_distance_deg": 149.5,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:58.33Z",
        "isc_residual_s": "0.7",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TGA",
        "network": "IR",
        "latitude": 50.6046,
        "longitude": 4.5223,
        "channel": "BHZ",
        "isc_distance_deg": 149.5,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:51.90Z",
        "isc_residual_s": "0.4",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TGA",
        "network": "IR",
        "latitude": 50.6046,
        "longitude": 4.5223,
        "channel": "BHZ",
        "isc_distance_deg": 149.5,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:57.32Z",
        "isc_residual_s": "-0.2",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GRF",
        "network": "IR",
        "latitude": 49.6919,
        "longitude": 11.2217,
        "channel": "???",
        "isc_distance_deg": 149.54,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:51.88Z",
        "isc_residual_s": "0.2",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GRF",
        "network": "IR",
        "latitude": 49.6919,
        "longitude": 11.2217,
        "channel": "???",
        "isc_distance_deg": 149.54,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:58.55Z",
        "isc_residual_s": "0.9",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GRFO",
        "network": "IR",
        "latitude": 49.6909,
        "longitude": 11.2203,
        "channel": "S?Z",
        "isc_distance_deg": 149.54,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:51.30Z",
        "isc_residual_s": "-0.4",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KHC",
        "network": "IR",
        "latitude": 49.1309,
        "longitude": 13.5782,
        "channel": "S?Z",
        "isc_distance_deg": 149.54,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:51.80Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KHC",
        "network": "IR",
        "latitude": 49.1309,
        "longitude": 13.5782,
        "channel": "???",
        "isc_distance_deg": 149.54,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:51.80Z",
        "isc_residual_s": "0.1",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KHC",
        "network": "IR",
        "latitude": 49.1309,
        "longitude": 13.5782,
        "channel": "???",
        "isc_distance_deg": 149.54,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPBC",
        "arrival_utc": "2022-11-12T07:27:57.60Z",
        "isc_residual_s": "-0.2",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ELND",
        "network": "IR",
        "latitude": 42.9287,
        "longitude": 25.8751,
        "channel": "???",
        "isc_distance_deg": 149.56,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.80Z",
        "isc_residual_s": "-0.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SNF",
        "network": "IR",
        "latitude": 50.5086,
        "longitude": 4.2833,
        "channel": "BHZ",
        "isc_distance_deg": 149.61,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:51.99Z",
        "isc_residual_s": "0.3",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VLAD",
        "network": "IR",
        "latitude": 43.9986,
        "longitude": 24.4038,
        "channel": "???",
        "isc_distance_deg": 149.62,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.30Z",
        "isc_residual_s": "1.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CKRC",
        "network": "IR",
        "latitude": 48.8199,
        "longitude": 14.3095,
        "channel": "???",
        "isc_distance_deg": 149.64,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPDF",
        "arrival_utc": "2022-11-12T07:27:52.90Z",
        "isc_residual_s": "1.0",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CKRC",
        "network": "IR",
        "latitude": 48.8199,
        "longitude": 14.3095,
        "channel": "???",
        "isc_distance_deg": 149.64,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPBC",
        "arrival_utc": "2022-11-12T07:27:58.90Z",
        "isc_residual_s": "1.0",
        "reporter": "PRU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CRAR",
        "network": "IR",
        "latitude": 44.325,
        "longitude": 23.7999,
        "channel": "???",
        "isc_distance_deg": 149.7,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:52.19Z",
        "isc_residual_s": "0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CRAR",
        "network": "IR",
        "latitude": 44.325,
        "longitude": 23.7999,
        "channel": "S?Z",
        "isc_distance_deg": 149.7,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:52.20Z",
        "isc_residual_s": "0.2",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WET",
        "network": "IR",
        "latitude": 49.144,
        "longitude": 12.8782,
        "channel": "S?Z",
        "isc_distance_deg": 149.7,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:51.90Z",
        "isc_residual_s": "-0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WET",
        "network": "IR",
        "latitude": 49.144,
        "longitude": 12.8782,
        "channel": "???",
        "isc_distance_deg": 149.7,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:51.88Z",
        "isc_residual_s": "-0.1",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WET",
        "network": "IR",
        "latitude": 49.144,
        "longitude": 12.8782,
        "channel": "???",
        "isc_distance_deg": 149.7,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:58.92Z",
        "isc_residual_s": "0.9",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SRE",
        "network": "IR",
        "latitude": 44.6622,
        "longitude": 23.2052,
        "channel": "???",
        "isc_distance_deg": 149.75,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.12Z",
        "isc_residual_s": "1.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BZS",
        "network": "IR",
        "latitude": 45.6167,
        "longitude": 21.6167,
        "channel": "???",
        "isc_distance_deg": 149.77,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.96Z",
        "isc_residual_s": "-0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BZS",
        "network": "IR",
        "latitude": 45.6167,
        "longitude": 21.6167,
        "channel": "S?Z",
        "isc_distance_deg": 149.77,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:51.50Z",
        "isc_residual_s": "-0.6",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GEC2",
        "network": "IR",
        "latitude": 48.8451,
        "longitude": 13.7016,
        "channel": "S?Z",
        "isc_distance_deg": 149.78,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:52.10Z",
        "isc_residual_s": "-0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GEC2",
        "network": "IR",
        "latitude": 48.8451,
        "longitude": 13.7016,
        "channel": "???",
        "isc_distance_deg": 149.78,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:52.10Z",
        "isc_residual_s": "-0.1",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GEC2",
        "network": "IR",
        "latitude": 48.8451,
        "longitude": 13.7016,
        "channel": "???",
        "isc_distance_deg": 149.78,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:27:58.98Z",
        "isc_residual_s": "0.7",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WINA",
        "network": "IR",
        "latitude": 48.1074,
        "longitude": 16.0731,
        "channel": "???",
        "isc_distance_deg": 149.78,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:52.22Z",
        "isc_residual_s": "0.1",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WINA",
        "network": "IR",
        "latitude": 48.1074,
        "longitude": 16.0731,
        "channel": "???",
        "isc_distance_deg": 149.78,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:27:58.54Z",
        "isc_residual_s": "0.3",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BMRD",
        "network": "IR",
        "latitude": 50.3025,
        "longitude": 4.7675,
        "channel": "BHZ",
        "isc_distance_deg": 149.78,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:52.01Z",
        "isc_residual_s": "0.0",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BAND",
        "network": "IR",
        "latitude": 40.3553,
        "longitude": 28.0992,
        "channel": "???",
        "isc_distance_deg": 149.8,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.57Z",
        "isc_residual_s": "-0.8",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PVL",
        "network": "IR",
        "latitude": 43.1226,
        "longitude": 25.1731,
        "channel": "???",
        "isc_distance_deg": 149.81,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:59.10Z",
        "isc_residual_s": "0.7",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RCHB",
        "network": "IR",
        "latitude": 50.156,
        "longitude": 5.228,
        "channel": "BHZ",
        "isc_distance_deg": 149.9,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:52.22Z",
        "isc_residual_s": "0.1",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PLVB",
        "network": "IR",
        "latitude": 43.387,
        "longitude": 24.6207,
        "channel": "???",
        "isc_distance_deg": 149.93,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.12Z",
        "isc_residual_s": "0.7",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HERR",
        "network": "IR",
        "latitude": 44.881,
        "longitude": 22.416,
        "channel": "???",
        "isc_distance_deg": 149.96,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:52.18Z",
        "isc_residual_s": "-0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DOU",
        "network": "IR",
        "latitude": 50.096,
        "longitude": 4.595,
        "channel": "BHZ",
        "isc_distance_deg": 150.0,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:52.36Z",
        "isc_residual_s": "0.0",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DOU",
        "network": "IR",
        "latitude": 50.096,
        "longitude": 4.595,
        "channel": "BHZ",
        "isc_distance_deg": 150.0,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:58.37Z",
        "isc_residual_s": "-0.2",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CONA",
        "network": "IR",
        "latitude": 47.9282,
        "longitude": 15.8618,
        "channel": "???",
        "isc_distance_deg": 150.0,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:52.21Z",
        "isc_residual_s": "-0.3",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CONA",
        "network": "IR",
        "latitude": 47.9282,
        "longitude": 15.8618,
        "channel": "???",
        "isc_distance_deg": 150.0,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:27:59.12Z",
        "isc_residual_s": "0.3",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RONA",
        "network": "IR",
        "latitude": 47.6997,
        "longitude": 16.2963,
        "channel": "???",
        "isc_distance_deg": 150.07,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:52.35Z",
        "isc_residual_s": "-0.2",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RONA",
        "network": "IR",
        "latitude": 47.6997,
        "longitude": 16.2963,
        "channel": "???",
        "isc_distance_deg": 150.07,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:58.86Z",
        "isc_residual_s": "-0.0",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DJES",
        "network": "IR",
        "latitude": 44.666,
        "longitude": 22.52,
        "channel": "???",
        "isc_distance_deg": 150.07,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:51.60Z",
        "isc_residual_s": "-1.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BAIL",
        "network": "IR",
        "latitude": 44.0201,
        "longitude": 23.345,
        "channel": "???",
        "isc_distance_deg": 150.15,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:52.74Z",
        "isc_residual_s": "0.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BANR",
        "network": "IR",
        "latitude": 45.3825,
        "longitude": 21.1355,
        "channel": "???",
        "isc_distance_deg": 150.16,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.21Z",
        "isc_residual_s": "0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DIM",
        "network": "IR",
        "latitude": 42.0457,
        "longitude": 25.5778,
        "channel": "???",
        "isc_distance_deg": 150.3,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:00.50Z",
        "isc_residual_s": "1.0",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WLF",
        "network": "IR",
        "latitude": 49.6646,
        "longitude": 6.1526,
        "channel": "HHZ",
        "isc_distance_deg": 150.31,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:53.22Z",
        "isc_residual_s": "0.4",
        "reporter": "SFS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WLF",
        "network": "IR",
        "latitude": 49.6646,
        "longitude": 6.1526,
        "channel": "BHZ",
        "isc_distance_deg": 150.31,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:53.14Z",
        "isc_residual_s": "0.3",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WLF",
        "network": "IR",
        "latitude": 49.6646,
        "longitude": 6.1526,
        "channel": "BHZ",
        "isc_distance_deg": 150.31,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:59.63Z",
        "isc_residual_s": "0.4",
        "reporter": "UCC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WLF",
        "network": "IR",
        "latitude": 49.6646,
        "longitude": 6.1526,
        "channel": "S?Z",
        "isc_distance_deg": 150.31,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:53.60Z",
        "isc_residual_s": "0.8",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WLF",
        "network": "IR",
        "latitude": 49.6646,
        "longitude": 6.1526,
        "channel": "???",
        "isc_distance_deg": 150.31,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:53.55Z",
        "isc_residual_s": "0.7",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WLF",
        "network": "IR",
        "latitude": 49.6646,
        "longitude": 6.1526,
        "channel": "???",
        "isc_distance_deg": 150.31,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:28:00.74Z",
        "isc_residual_s": "1.5",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VALD",
        "network": "IR",
        "latitude": 43.6899,
        "longitude": 23.4356,
        "channel": "???",
        "isc_distance_deg": 150.34,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:00.40Z",
        "isc_residual_s": "0.9",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MDVR",
        "network": "IR",
        "latitude": 44.7815,
        "longitude": 21.7128,
        "channel": "???",
        "isc_distance_deg": 150.37,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:52.92Z",
        "isc_residual_s": "-0.2",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MPEP",
        "network": "IR",
        "latitude": 43.356,
        "longitude": 23.7401,
        "channel": "???",
        "isc_distance_deg": 150.42,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:02.50Z",
        "isc_residual_s": "2.8",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MORH",
        "network": "IR",
        "latitude": 46.2149,
        "longitude": 18.6435,
        "channel": "???",
        "isc_distance_deg": 150.52,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.18Z",
        "isc_residual_s": "-0.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MORH",
        "network": "IR",
        "latitude": 46.2149,
        "longitude": 18.6435,
        "channel": "???",
        "isc_distance_deg": 150.52,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.40Z",
        "isc_residual_s": "0.2",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MOA",
        "network": "IR",
        "latitude": 47.8495,
        "longitude": 14.2659,
        "channel": "???",
        "isc_distance_deg": 150.54,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:52.96Z",
        "isc_residual_s": "-0.3",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MOA",
        "network": "IR",
        "latitude": 47.8495,
        "longitude": 14.2659,
        "channel": "???",
        "isc_distance_deg": 150.54,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:59.73Z",
        "isc_residual_s": "-0.1",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PLD",
        "network": "IR",
        "latitude": 42.1469,
        "longitude": 24.7475,
        "channel": "???",
        "isc_distance_deg": 150.7,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:00.50Z",
        "isc_residual_s": "0.2",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KDZ",
        "network": "IR",
        "latitude": 41.6302,
        "longitude": 25.3396,
        "channel": "???",
        "isc_distance_deg": 150.7,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:00.50Z",
        "isc_residual_s": "0.2",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ARSA",
        "network": "IR",
        "latitude": 47.2505,
        "longitude": 15.5232,
        "channel": "???",
        "isc_distance_deg": 150.71,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:52.98Z",
        "isc_residual_s": "-0.6",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ARSA",
        "network": "IR",
        "latitude": 47.2505,
        "longitude": 15.5232,
        "channel": "???",
        "isc_distance_deg": 150.71,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:00.22Z",
        "isc_residual_s": "0.0",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TAVA",
        "network": "IR",
        "latitude": 37.4657,
        "longitude": 28.9132,
        "channel": "???",
        "isc_distance_deg": 150.76,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:54.41Z",
        "isc_residual_s": "0.4",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BLKB",
        "network": "IR",
        "latitude": 43.6227,
        "longitude": 22.675,
        "channel": "???",
        "isc_distance_deg": 150.77,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.44Z",
        "isc_residual_s": "-0.3",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PGB",
        "network": "IR",
        "latitude": 42.5137,
        "longitude": 24.1734,
        "channel": "???",
        "isc_distance_deg": 150.77,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:01.00Z",
        "isc_residual_s": "0.5",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "STU",
        "network": "IR",
        "latitude": 48.7719,
        "longitude": 9.195,
        "channel": "S?Z",
        "isc_distance_deg": 150.79,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:53.70Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "STU",
        "network": "IR",
        "latitude": 48.7719,
        "longitude": 9.195,
        "channel": "???",
        "isc_distance_deg": 150.79,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:53.72Z",
        "isc_residual_s": "0.1",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "STU",
        "network": "IR",
        "latitude": 48.7719,
        "longitude": 9.195,
        "channel": "???",
        "isc_distance_deg": 150.79,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:28:01.30Z",
        "isc_residual_s": "1.0",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CAVK",
        "network": "IR",
        "latitude": 40.6917,
        "longitude": 26.1677,
        "channel": "???",
        "isc_distance_deg": 150.8,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:52.80Z",
        "isc_residual_s": "-1.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "JSA",
        "network": "IR",
        "latitude": 49.1879,
        "longitude": -2.1709,
        "channel": "H?Z",
        "isc_distance_deg": 150.82,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.91Z",
        "isc_residual_s": "0.3",
        "reporter": "BGS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BIOA",
        "network": "IR",
        "latitude": 47.6913,
        "longitude": 13.6383,
        "channel": "???",
        "isc_distance_deg": 150.86,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:53.10Z",
        "isc_residual_s": "-0.7",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FRGS",
        "network": "IR",
        "latitude": 45.1577,
        "longitude": 19.8102,
        "channel": "???",
        "isc_distance_deg": 150.91,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.90Z",
        "isc_residual_s": "-0.0",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FUR",
        "network": "IR",
        "latitude": 48.1655,
        "longitude": 11.2763,
        "channel": "S?Z",
        "isc_distance_deg": 150.98,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:54.00Z",
        "isc_residual_s": "0.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FUR",
        "network": "IR",
        "latitude": 48.1655,
        "longitude": 11.2763,
        "channel": "???",
        "isc_distance_deg": 150.98,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:53.99Z",
        "isc_residual_s": "0.0",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FUR",
        "network": "IR",
        "latitude": 48.1655,
        "longitude": 11.2763,
        "channel": "???",
        "isc_distance_deg": 150.98,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:28:01.61Z",
        "isc_residual_s": "0.9",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BEHE",
        "network": "IR",
        "latitude": 46.4702,
        "longitude": 16.7755,
        "channel": "???",
        "isc_distance_deg": 150.99,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.90Z",
        "isc_residual_s": "-0.1",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RZN",
        "network": "IR",
        "latitude": 41.6941,
        "longitude": 24.7388,
        "channel": "???",
        "isc_distance_deg": 151.0,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:01.50Z",
        "isc_residual_s": "0.3",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BLSH",
        "network": "IR",
        "latitude": 42.8569,
        "longitude": 23.2792,
        "channel": "???",
        "isc_distance_deg": 151.01,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:02.20Z",
        "isc_residual_s": "1.2",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SESA",
        "network": "IR",
        "latitude": 47.1132,
        "longitude": 14.5668,
        "channel": "???",
        "isc_distance_deg": 151.12,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:53.85Z",
        "isc_residual_s": "-0.6",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KOGS",
        "network": "IR",
        "latitude": 46.4481,
        "longitude": 16.2503,
        "channel": "B?Z",
        "isc_distance_deg": 151.18,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:53.59Z",
        "isc_residual_s": "-0.7",
        "reporter": "LJU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PLNA",
        "network": "IR",
        "latitude": 42.4762,
        "longitude": 23.4255,
        "channel": "???",
        "isc_distance_deg": 151.2,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:02.40Z",
        "isc_residual_s": "0.9",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VTS",
        "network": "IR",
        "latitude": 42.618,
        "longitude": 23.235,
        "channel": "???",
        "isc_distance_deg": 151.2,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:02.20Z",
        "isc_residual_s": "0.7",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "EZN",
        "network": "IR",
        "latitude": 39.8258,
        "longitude": 26.3253,
        "channel": "???",
        "isc_distance_deg": 151.22,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.76Z",
        "isc_residual_s": "-0.7",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "EZN",
        "network": "IR",
        "latitude": 39.8258,
        "longitude": 26.3253,
        "channel": "S?Z",
        "isc_distance_deg": 151.22,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:53.80Z",
        "isc_residual_s": "-0.7",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BOVS",
        "network": "IR",
        "latitude": 43.6351,
        "longitude": 21.7057,
        "channel": "???",
        "isc_distance_deg": 151.23,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:53.93Z",
        "isc_residual_s": "-0.5",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TRAN",
        "network": "IR",
        "latitude": 42.8346,
        "longitude": 22.6511,
        "channel": "???",
        "isc_distance_deg": 151.35,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:02.40Z",
        "isc_residual_s": "0.7",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LESA",
        "network": "IR",
        "latitude": 47.4243,
        "longitude": 12.677,
        "channel": "???",
        "isc_distance_deg": 151.35,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:53.97Z",
        "isc_residual_s": "-0.6",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BFO",
        "network": "IR",
        "latitude": 48.3301,
        "longitude": 8.3296,
        "channel": "S?Z",
        "isc_distance_deg": 151.36,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:54.60Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BFO",
        "network": "IR",
        "latitude": 48.3301,
        "longitude": 8.3296,
        "channel": "???",
        "isc_distance_deg": 151.36,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:54.25Z",
        "isc_residual_s": "-0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BFO",
        "network": "IR",
        "latitude": 48.3301,
        "longitude": 8.3296,
        "channel": "???",
        "isc_distance_deg": 151.36,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:28:02.44Z",
        "isc_residual_s": "0.9",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SOKA",
        "network": "IR",
        "latitude": 46.678,
        "longitude": 15.0327,
        "channel": "???",
        "isc_distance_deg": 151.37,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:53.96Z",
        "isc_residual_s": "-0.7",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KBA",
        "network": "IR",
        "latitude": 47.0784,
        "longitude": 13.3447,
        "channel": "???",
        "isc_distance_deg": 151.5,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:53.94Z",
        "isc_residual_s": "-1.0",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TEKS",
        "network": "IR",
        "latitude": 44.5492,
        "longitude": 19.5287,
        "channel": "???",
        "isc_distance_deg": 151.52,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:54.90Z",
        "isc_residual_s": "0.0",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ROSF",
        "network": "IR",
        "latitude": 48.3364,
        "longitude": -3.2736,
        "channel": "S?Z",
        "isc_distance_deg": 151.55,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:55.50Z",
        "isc_residual_s": "0.8",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ECH",
        "network": "IR",
        "latitude": 48.2158,
        "longitude": 7.1583,
        "channel": "S?Z",
        "isc_distance_deg": 151.63,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:54.90Z",
        "isc_residual_s": "-0.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MMB",
        "network": "IR",
        "latitude": 41.547,
        "longitude": 23.7495,
        "channel": "???",
        "isc_distance_deg": 151.66,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:02.70Z",
        "isc_residual_s": "0.3",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OBKA",
        "network": "IR",
        "latitude": 46.5092,
        "longitude": 14.5489,
        "channel": "???",
        "isc_distance_deg": 151.67,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:54.25Z",
        "isc_residual_s": "-0.9",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UBR",
        "network": "IR",
        "latitude": 47.6807,
        "longitude": 10.108,
        "channel": "???",
        "isc_distance_deg": 151.68,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:54.74Z",
        "isc_residual_s": "-0.3",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UBR",
        "network": "IR",
        "latitude": 47.6807,
        "longitude": 10.108,
        "channel": "???",
        "isc_distance_deg": 151.68,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:28:03.00Z",
        "isc_residual_s": "0.8",
        "reporter": "BGR",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WATA",
        "network": "IR",
        "latitude": 47.3357,
        "longitude": 11.5763,
        "channel": "???",
        "isc_distance_deg": 151.7,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:54.68Z",
        "isc_residual_s": "-0.5",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WATA",
        "network": "IR",
        "latitude": 47.3357,
        "longitude": 11.5763,
        "channel": "???",
        "isc_distance_deg": 151.7,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:02.51Z",
        "isc_residual_s": "0.1",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RETA",
        "network": "IR",
        "latitude": 47.4871,
        "longitude": 10.7623,
        "channel": "???",
        "isc_distance_deg": 151.74,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:54.88Z",
        "isc_residual_s": "-0.3",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "RETA",
        "network": "IR",
        "latitude": 47.4871,
        "longitude": 10.7623,
        "channel": "???",
        "isc_distance_deg": 151.74,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:02.40Z",
        "isc_residual_s": "0.0",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WTTA",
        "network": "IR",
        "latitude": 47.2638,
        "longitude": 11.6363,
        "channel": "???",
        "isc_distance_deg": 151.76,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:54.94Z",
        "isc_residual_s": "-0.4",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "WTTA",
        "network": "IR",
        "latitude": 47.2638,
        "longitude": 11.6363,
        "channel": "???",
        "isc_distance_deg": 151.76,
        "isc_phase": "PKiKP",
        "reported_phase": "PKiKP",
        "arrival_utc": "2022-11-12T07:28:02.90Z",
        "isc_residual_s": "0.3",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KKB",
        "network": "IR",
        "latitude": 41.8417,
        "longitude": 23.1287,
        "channel": "???",
        "isc_distance_deg": 151.79,
        "isc_phase": "PKiKP",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:03.20Z",
        "isc_residual_s": "0.6",
        "reporter": "SOF",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MOTA",
        "network": "IR",
        "latitude": 47.3448,
        "longitude": 11.1037,
        "channel": "???",
        "isc_distance_deg": 151.8,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:54.96Z",
        "isc_residual_s": "-0.4",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MOTA",
        "network": "IR",
        "latitude": 47.3448,
        "longitude": 11.1037,
        "channel": "???",
        "isc_distance_deg": 151.8,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:02.62Z",
        "isc_residual_s": "0.0",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MYKA",
        "network": "IR",
        "latitude": 46.6299,
        "longitude": 13.6416,
        "channel": "???",
        "isc_distance_deg": 151.83,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:54.25Z",
        "isc_residual_s": "-1.1",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SQTA",
        "network": "IR",
        "latitude": 47.2205,
        "longitude": 11.2087,
        "channel": "???",
        "isc_distance_deg": 151.89,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:54.96Z",
        "isc_residual_s": "-0.5",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SQTA",
        "network": "IR",
        "latitude": 47.2205,
        "longitude": 11.2087,
        "channel": "???",
        "isc_distance_deg": 151.89,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:02.75Z",
        "isc_residual_s": "-0.0",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ARG",
        "network": "IR",
        "latitude": 36.2161,
        "longitude": 28.1261,
        "channel": "S?Z",
        "isc_distance_deg": 151.91,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:56.60Z",
        "isc_residual_s": "0.9",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BDRM",
        "network": "IR",
        "latitude": 37.0653,
        "longitude": 27.4442,
        "channel": "???",
        "isc_distance_deg": 151.98,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:54.77Z",
        "isc_residual_s": "-1.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CRES",
        "network": "IR",
        "latitude": 45.826,
        "longitude": 15.4569,
        "channel": "???",
        "isc_distance_deg": 152.0,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:55.70Z",
        "isc_residual_s": "0.2",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ABTA",
        "network": "IR",
        "latitude": 46.7474,
        "longitude": 12.5123,
        "channel": "???",
        "isc_distance_deg": 152.03,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:54.65Z",
        "isc_residual_s": "-1.0",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LJU",
        "network": "IR",
        "latitude": 46.0438,
        "longitude": 14.5274,
        "channel": "B?Z",
        "isc_distance_deg": 152.1,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:55.38Z",
        "isc_residual_s": "-0.3",
        "reporter": "LJU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LJU",
        "network": "IR",
        "latitude": 46.0438,
        "longitude": 14.5274,
        "channel": "B?Z",
        "isc_distance_deg": 152.1,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:28:03.03Z",
        "isc_residual_s": "-0.0",
        "reporter": "LJU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DAVA",
        "network": "IR",
        "latitude": 47.2867,
        "longitude": 9.8803,
        "channel": "???",
        "isc_distance_deg": 152.11,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:55.55Z",
        "isc_residual_s": "-0.3",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DAVA",
        "network": "IR",
        "latitude": 47.2867,
        "longitude": 9.8803,
        "channel": "???",
        "isc_distance_deg": 152.11,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:03.48Z",
        "isc_residual_s": "0.2",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BBLS",
        "network": "IR",
        "latitude": 43.866,
        "longitude": 19.405,
        "channel": "???",
        "isc_distance_deg": 152.12,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:56.20Z",
        "isc_residual_s": "0.4",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HAPS",
        "network": "IR",
        "latitude": 44.0862,
        "longitude": 18.9507,
        "channel": "???",
        "isc_distance_deg": 152.14,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:56.10Z",
        "isc_residual_s": "0.2",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FETA",
        "network": "IR",
        "latitude": 47.0211,
        "longitude": 10.7291,
        "channel": "???",
        "isc_distance_deg": 152.19,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:55.87Z",
        "isc_residual_s": "-0.1",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FETA",
        "network": "IR",
        "latitude": 47.0211,
        "longitude": 10.7291,
        "channel": "???",
        "isc_distance_deg": 152.19,
        "isc_phase": "PKiKP",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:03.72Z",
        "isc_residual_s": "0.3",
        "reporter": "VIE",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MRAK",
        "network": "IR",
        "latitude": 45.013,
        "longitude": 16.9,
        "channel": "???",
        "isc_distance_deg": 152.19,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:56.10Z",
        "isc_residual_s": "0.2",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VISS",
        "network": "IR",
        "latitude": 45.8033,
        "longitude": 14.8393,
        "channel": "B?Z",
        "isc_distance_deg": 152.22,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:28:04.02Z",
        "isc_residual_s": "0.7",
        "reporter": "LJU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BLY",
        "network": "IR",
        "latitude": 44.7488,
        "longitude": 17.1839,
        "channel": "???",
        "isc_distance_deg": 152.31,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:55.87Z",
        "isc_residual_s": "-0.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BOJS",
        "network": "IR",
        "latitude": 45.504,
        "longitude": 15.252,
        "channel": "???",
        "isc_distance_deg": 152.35,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:56.50Z",
        "isc_residual_s": "0.5",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BOJS",
        "network": "IR",
        "latitude": 45.504,
        "longitude": 15.252,
        "channel": "B?Z",
        "isc_distance_deg": 152.35,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:27:56.13Z",
        "isc_residual_s": "0.1",
        "reporter": "LJU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BOJS",
        "network": "IR",
        "latitude": 45.504,
        "longitude": 15.252,
        "channel": "B?Z",
        "isc_distance_deg": 152.35,
        "isc_phase": "PKiKP",
        "reported_phase": "PKPbc",
        "arrival_utc": "2022-11-12T07:28:03.74Z",
        "isc_residual_s": "0.2",
        "reporter": "LJU",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "CEY",
        "network": "IR",
        "latitude": 45.7388,
        "longitude": 14.4267,
        "channel": "???",
        "isc_distance_deg": 152.4,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:56.40Z",
        "isc_residual_s": "0.3",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SKO",
        "network": "IR",
        "latitude": 41.9721,
        "longitude": 21.4396,
        "channel": "??Z",
        "isc_distance_deg": 152.59,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:55.57Z",
        "isc_residual_s": "-0.9",
        "reporter": "SKO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TRI",
        "network": "IR",
        "latitude": 45.7089,
        "longitude": 13.7642,
        "channel": "S?Z",
        "isc_distance_deg": 152.63,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:55.70Z",
        "isc_residual_s": "-0.7",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SKDS",
        "network": "IR",
        "latitude": 45.5464,
        "longitude": 14.0143,
        "channel": "???",
        "isc_distance_deg": 152.7,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:56.30Z",
        "isc_residual_s": "-0.3",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UPM",
        "network": "IR",
        "latitude": 43.2031,
        "longitude": 18.9051,
        "channel": "???",
        "isc_distance_deg": 152.87,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:57.50Z",
        "isc_residual_s": "0.4",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KLINJ",
        "network": "IR",
        "latitude": 43.1694,
        "longitude": 18.5853,
        "channel": "???",
        "isc_distance_deg": 153.03,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:00.90Z",
        "isc_residual_s": "3.7",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "NEVS",
        "network": "IR",
        "latitude": 43.299,
        "longitude": 18.103,
        "channel": "???",
        "isc_distance_deg": 153.14,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:57.30Z",
        "isc_residual_s": "-0.0",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "BRY",
        "network": "IR",
        "latitude": 42.9016,
        "longitude": 18.5389,
        "channel": "???",
        "isc_distance_deg": 153.27,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:58.10Z",
        "isc_residual_s": "0.5",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "APE",
        "network": "IR",
        "latitude": 37.0689,
        "longitude": 25.5306,
        "channel": "???",
        "isc_distance_deg": 153.28,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:56.14Z",
        "isc_residual_s": "-1.6",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "APE",
        "network": "IR",
        "latitude": 37.0689,
        "longitude": 25.5306,
        "channel": "S?Z",
        "isc_distance_deg": 153.28,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:56.10Z",
        "isc_residual_s": "-1.6",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PDG",
        "network": "IR",
        "latitude": 42.4302,
        "longitude": 19.2553,
        "channel": "???",
        "isc_distance_deg": 153.31,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:57.48Z",
        "isc_residual_s": "0.0",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PDG",
        "network": "IR",
        "latitude": 42.4302,
        "longitude": 19.2553,
        "channel": "???",
        "isc_distance_deg": 153.31,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:57.10Z",
        "isc_residual_s": "-0.4",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TREB",
        "network": "IR",
        "latitude": 42.717,
        "longitude": 18.35,
        "channel": "???",
        "isc_distance_deg": 153.5,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:57.60Z",
        "isc_residual_s": "-0.2",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DRME",
        "network": "IR",
        "latitude": 42.1867,
        "longitude": 19.1843,
        "channel": "???",
        "isc_distance_deg": 153.54,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:57.80Z",
        "isc_residual_s": "-0.0",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OHR",
        "network": "IR",
        "latitude": 41.1114,
        "longitude": 20.7989,
        "channel": "??Z",
        "isc_distance_deg": 153.55,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:57.38Z",
        "isc_residual_s": "-0.6",
        "reporter": "SKO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "HCY",
        "network": "IR",
        "latitude": 42.4473,
        "longitude": 18.4938,
        "channel": "???",
        "isc_distance_deg": 153.65,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:27:57.70Z",
        "isc_residual_s": "-0.3",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TIR",
        "network": "IR",
        "latitude": 41.3477,
        "longitude": 19.865,
        "channel": "S?Z",
        "isc_distance_deg": 153.85,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:27:59.30Z",
        "isc_residual_s": "1.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "IDI",
        "network": "IR",
        "latitude": 35.288,
        "longitude": 24.89,
        "channel": "HHZ",
        "isc_distance_deg": 154.66,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:58.11Z",
        "isc_residual_s": "-1.6",
        "reporter": "SFS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SSB",
        "network": "IR",
        "latitude": 45.2792,
        "longitude": 4.5417,
        "channel": "S?Z",
        "isc_distance_deg": 154.8,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:28:00.20Z",
        "isc_residual_s": "0.7",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "VLC",
        "network": "IR",
        "latitude": 44.1594,
        "longitude": 10.3864,
        "channel": "HHZ",
        "isc_distance_deg": 154.97,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:58.11Z",
        "isc_residual_s": "-1.7",
        "reporter": "SFS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SACV",
        "network": "IR",
        "latitude": 14.9702,
        "longitude": -23.6085,
        "channel": "B?Z",
        "isc_distance_deg": 155.26,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:01.28Z",
        "isc_residual_s": "0.3",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AQU",
        "network": "IR",
        "latitude": 42.3539,
        "longitude": 13.4019,
        "channel": "HHZ",
        "isc_distance_deg": 155.76,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:27:59.65Z",
        "isc_residual_s": "-1.3",
        "reporter": "SFS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AQU",
        "network": "IR",
        "latitude": 42.3539,
        "longitude": 13.4019,
        "channel": "S?Z",
        "isc_distance_deg": 155.76,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:28:01.20Z",
        "isc_residual_s": "0.3",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AQU",
        "network": "IR",
        "latitude": 42.3539,
        "longitude": 13.4019,
        "channel": "???",
        "isc_distance_deg": 155.76,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:00.60Z",
        "isc_residual_s": "-0.3",
        "reporter": "RHSSO",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PGAV",
        "network": "IR",
        "latitude": 41.9653,
        "longitude": -8.2698,
        "channel": "???",
        "isc_distance_deg": 156.61,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:02.60Z",
        "isc_residual_s": "0.5",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PGF",
        "network": "IR",
        "latitude": 42.5483,
        "longitude": 8.9994,
        "channel": "S?Z",
        "isc_distance_deg": 156.82,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:28:02.40Z",
        "isc_residual_s": "0.1",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PCAB",
        "network": "IR",
        "latitude": 41.7102,
        "longitude": -8.027,
        "channel": "???",
        "isc_distance_deg": 156.92,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:02.53Z",
        "isc_residual_s": "0.1",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TIP",
        "network": "IR",
        "latitude": 39.1786,
        "longitude": 16.7583,
        "channel": "???",
        "isc_distance_deg": 157.03,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:02.73Z",
        "isc_residual_s": "0.1",
        "reporter": "BUC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ECAL",
        "network": "IR",
        "latitude": 41.9413,
        "longitude": -6.7371,
        "channel": "HHZ",
        "isc_distance_deg": 157.07,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:01.32Z",
        "isc_residual_s": "-1.3",
        "reporter": "SFS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PBRG",
        "network": "IR",
        "latitude": 41.8048,
        "longitude": -6.7413,
        "channel": "???",
        "isc_distance_deg": 157.2,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:03.18Z",
        "isc_residual_s": "0.4",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PTO",
        "network": "IR",
        "latitude": 41.1386,
        "longitude": -8.6022,
        "channel": "???",
        "isc_distance_deg": 157.25,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:03.10Z",
        "isc_residual_s": "0.4",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "POLO",
        "network": "IR",
        "latitude": 41.3738,
        "longitude": -7.7942,
        "channel": "???",
        "isc_distance_deg": 157.29,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:03.77Z",
        "isc_residual_s": "0.8",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PVRL",
        "network": "IR",
        "latitude": 41.2756,
        "longitude": -7.7161,
        "channel": "???",
        "isc_distance_deg": 157.41,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:03.83Z",
        "isc_residual_s": "0.8",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MVO",
        "network": "IR",
        "latitude": 41.1644,
        "longitude": -7.0289,
        "channel": "???",
        "isc_distance_deg": 157.71,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:03.80Z",
        "isc_residual_s": "0.4",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "EJON",
        "network": "IR",
        "latitude": 42.4487,
        "longitude": 2.8886,
        "channel": "HHZ",
        "isc_distance_deg": 157.71,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:02.58Z",
        "isc_residual_s": "-0.8",
        "reporter": "SFS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PVIS",
        "network": "IR",
        "latitude": 40.7157,
        "longitude": -7.8945,
        "channel": "???",
        "isc_distance_deg": 157.86,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:03.69Z",
        "isc_residual_s": "0.1",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "COI",
        "network": "IR",
        "latitude": 40.208,
        "longitude": -8.4124,
        "channel": "???",
        "isc_distance_deg": 158.15,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:04.40Z",
        "isc_residual_s": "0.5",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PCAS",
        "network": "IR",
        "latitude": 40.0528,
        "longitude": -8.4983,
        "channel": "???",
        "isc_distance_deg": 158.26,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:04.10Z",
        "isc_residual_s": "0.1",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MTE",
        "network": "IR",
        "latitude": 40.3997,
        "longitude": -7.5442,
        "channel": "???",
        "isc_distance_deg": 158.26,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:04.78Z",
        "isc_residual_s": "0.7",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PSBE",
        "network": "IR",
        "latitude": 39.5142,
        "longitude": -8.7955,
        "channel": "???",
        "isc_distance_deg": 158.63,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:04.75Z",
        "isc_residual_s": "0.2",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PSARD",
        "network": "IR",
        "latitude": 39.6003,
        "longitude": -8.1623,
        "channel": "???",
        "isc_distance_deg": 158.78,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:04.67Z",
        "isc_residual_s": "-0.0",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PCBR",
        "network": "IR",
        "latitude": 39.8403,
        "longitude": -7.4775,
        "channel": "???",
        "isc_distance_deg": 158.79,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:05.43Z",
        "isc_residual_s": "0.8",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PMOZ",
        "network": "IR",
        "latitude": 32.823,
        "longitude": -17.197,
        "channel": "???",
        "isc_distance_deg": 158.85,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:06.81Z",
        "isc_residual_s": "1.7",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PMOZ",
        "network": "IR",
        "latitude": 32.823,
        "longitude": -17.197,
        "channel": "???",
        "isc_distance_deg": 158.85,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:07.76Z",
        "isc_residual_s": "2.7",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PMAFR",
        "network": "IR",
        "latitude": 38.9554,
        "longitude": -9.2827,
        "channel": "???",
        "isc_distance_deg": 158.93,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:04.85Z",
        "isc_residual_s": "-0.0",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LIS",
        "network": "IR",
        "latitude": 38.718,
        "longitude": -9.1505,
        "channel": "B?Z",
        "isc_distance_deg": 159.19,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:05.40Z",
        "isc_residual_s": "0.3",
        "reporter": "IGIL",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LIS",
        "network": "IR",
        "latitude": 38.718,
        "longitude": -9.1505,
        "channel": "???",
        "isc_distance_deg": 159.19,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:06.08Z",
        "isc_residual_s": "1.0",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PMRV",
        "network": "IR",
        "latitude": 39.4283,
        "longitude": -7.3917,
        "channel": "???",
        "isc_distance_deg": 159.19,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:05.78Z",
        "isc_residual_s": "0.6",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PMTG",
        "network": "IR",
        "latitude": 39.069,
        "longitude": -8.2253,
        "channel": "???",
        "isc_distance_deg": 159.23,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:05.66Z",
        "isc_residual_s": "0.5",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PMPST",
        "network": "IR",
        "latitude": 33.0788,
        "longitude": -16.3334,
        "channel": "???",
        "isc_distance_deg": 159.32,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:06.65Z",
        "isc_residual_s": "1.2",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "UCM",
        "network": "IR",
        "latitude": 40.3077,
        "longitude": -3.2442,
        "channel": "HHZ",
        "isc_distance_deg": 159.42,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:06.49Z",
        "isc_residual_s": "1.1",
        "reporter": "SFS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PARRA",
        "network": "IR",
        "latitude": 38.8153,
        "longitude": -8.0402,
        "channel": "???",
        "isc_distance_deg": 159.52,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:05.85Z",
        "isc_residual_s": "0.3",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PESTR",
        "network": "IR",
        "latitude": 38.8672,
        "longitude": -7.5902,
        "channel": "???",
        "isc_distance_deg": 159.63,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:06.14Z",
        "isc_residual_s": "0.5",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PESTR",
        "network": "IR",
        "latitude": 38.8672,
        "longitude": -7.5902,
        "channel": "???",
        "isc_distance_deg": 159.63,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:06.86Z",
        "isc_residual_s": "1.2",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MOE",
        "network": "IR",
        "latitude": 38.5228,
        "longitude": -8.3506,
        "channel": "???",
        "isc_distance_deg": 159.66,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:06.25Z",
        "isc_residual_s": "0.5",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "EVO",
        "network": "IR",
        "latitude": 38.532,
        "longitude": -8.013,
        "channel": "???",
        "isc_distance_deg": 159.78,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:06.87Z",
        "isc_residual_s": "1.0",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ESBB",
        "network": "IR",
        "latitude": 39.6755,
        "longitude": -3.9617,
        "channel": "???",
        "isc_distance_deg": 159.89,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:06.87Z",
        "isc_residual_s": "0.9",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ESDC",
        "network": "IR",
        "latitude": 39.6743,
        "longitude": -3.9631,
        "channel": "???",
        "isc_distance_deg": 159.89,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:06.45Z",
        "isc_residual_s": "0.5",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PAB",
        "network": "IR",
        "latitude": 39.5458,
        "longitude": -4.3483,
        "channel": "S?Z",
        "isc_distance_deg": 159.93,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:28:07.10Z",
        "isc_residual_s": "1.0",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PAB",
        "network": "IR",
        "latitude": 39.5458,
        "longitude": -4.3483,
        "channel": "???",
        "isc_distance_deg": 159.93,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:07.09Z",
        "isc_residual_s": "1.0",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GRNL",
        "network": "IR",
        "latitude": 38.1083,
        "longitude": -8.569,
        "channel": "???",
        "isc_distance_deg": 159.94,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:06.98Z",
        "isc_residual_s": "1.0",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MAHO",
        "network": "IR",
        "latitude": 39.8959,
        "longitude": 4.2665,
        "channel": "HHZ",
        "isc_distance_deg": 160.17,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:06.49Z",
        "isc_residual_s": "0.3",
        "reporter": "SFS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PBEJ",
        "network": "IR",
        "latitude": 38.0264,
        "longitude": -7.8664,
        "channel": "???",
        "isc_distance_deg": 160.28,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:07.44Z",
        "isc_residual_s": "1.0",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MESJ",
        "network": "IR",
        "latitude": 37.8395,
        "longitude": -8.2199,
        "channel": "B?Z",
        "isc_distance_deg": 160.31,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:06.72Z",
        "isc_residual_s": "0.3",
        "reporter": "IGIL",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MESJ",
        "network": "IR",
        "latitude": 37.8395,
        "longitude": -8.2199,
        "channel": "???",
        "isc_distance_deg": 160.31,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:07.06Z",
        "isc_residual_s": "0.6",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PTEO",
        "network": "IR",
        "latitude": 37.5456,
        "longitude": -8.7242,
        "channel": "???",
        "isc_distance_deg": 160.37,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:07.58Z",
        "isc_residual_s": "1.1",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ETOS",
        "network": "IR",
        "latitude": 39.7678,
        "longitude": 2.8144,
        "channel": "HHZ",
        "isc_distance_deg": 160.39,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:06.49Z",
        "isc_residual_s": "0.0",
        "reporter": "SFS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PBAR",
        "network": "IR",
        "latitude": 38.175,
        "longitude": -7.039,
        "channel": "???",
        "isc_distance_deg": 160.44,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:07.52Z",
        "isc_residual_s": "1.0",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PCVE",
        "network": "IR",
        "latitude": 37.632,
        "longitude": -8.039,
        "channel": "???",
        "isc_distance_deg": 160.56,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:07.88Z",
        "isc_residual_s": "1.2",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MORF",
        "network": "IR",
        "latitude": 37.3044,
        "longitude": -8.6526,
        "channel": "B?Z",
        "isc_distance_deg": 160.6,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:07.00Z",
        "isc_residual_s": "0.2",
        "reporter": "IGIL",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MORF",
        "network": "IR",
        "latitude": 37.3044,
        "longitude": -8.6526,
        "channel": "???",
        "isc_distance_deg": 160.6,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:08.03Z",
        "isc_residual_s": "1.2",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PFVI",
        "network": "IR",
        "latitude": 37.1328,
        "longitude": -8.8268,
        "channel": "???",
        "isc_distance_deg": 160.68,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:08.23Z",
        "isc_residual_s": "1.4",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PFVI",
        "network": "IR",
        "latitude": 37.1328,
        "longitude": -8.8268,
        "channel": "???",
        "isc_distance_deg": 160.68,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:08.57Z",
        "isc_residual_s": "1.7",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PVAQ",
        "network": "IR",
        "latitude": 37.4037,
        "longitude": -7.7173,
        "channel": "???",
        "isc_distance_deg": 160.88,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:08.21Z",
        "isc_residual_s": "1.2",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PVAQ",
        "network": "IR",
        "latitude": 37.4037,
        "longitude": -7.7173,
        "channel": "???",
        "isc_distance_deg": 160.88,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:08.58Z",
        "isc_residual_s": "1.5",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PBDV",
        "network": "IR",
        "latitude": 37.243,
        "longitude": -7.9312,
        "channel": "???",
        "isc_distance_deg": 160.94,
        "isc_phase": "PKPdf",
        "reported_phase": "PKPdf",
        "arrival_utc": "2022-11-12T07:28:09.07Z",
        "isc_residual_s": "1.9",
        "reporter": "INMG",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "EDA",
        "network": "IR",
        "latitude": 3.7789,
        "longitude": 10.1534,
        "channel": "B?Z",
        "isc_distance_deg": 161.83,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:08.24Z",
        "isc_residual_s": "-0.5",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KEST",
        "network": "IR",
        "latitude": 35.7318,
        "longitude": 9.346,
        "channel": "???",
        "isc_distance_deg": 163.08,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:09.47Z",
        "isc_residual_s": "0.1",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KEST",
        "network": "IR",
        "latitude": 35.7318,
        "longitude": 9.346,
        "channel": "???",
        "isc_distance_deg": 163.08,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:10.02Z",
        "isc_residual_s": "0.6",
        "reporter": "OMAN",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "KEST",
        "network": "IR",
        "latitude": 35.7318,
        "longitude": 9.346,
        "channel": "???",
        "isc_distance_deg": 163.08,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:09.92Z",
        "isc_residual_s": "0.5",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "PALE",
        "network": "IR",
        "latitude": 35.217,
        "longitude": -3.939,
        "channel": "???",
        "isc_distance_deg": 164.13,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:11.01Z",
        "isc_residual_s": "0.7",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "AVE",
        "network": "IR",
        "latitude": 33.2981,
        "longitude": -7.4133,
        "channel": "???",
        "isc_distance_deg": 164.52,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:12.05Z",
        "isc_residual_s": "1.4",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SICH",
        "network": "IR",
        "latitude": 34.104,
        "longitude": -5.317,
        "channel": "???",
        "isc_distance_deg": 164.7,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:12.96Z",
        "isc_residual_s": "2.2",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ZHG",
        "network": "IR",
        "latitude": 33.446,
        "longitude": -6.642,
        "channel": "???",
        "isc_distance_deg": 164.75,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:12.96Z",
        "isc_residual_s": "2.0",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "SRHM",
        "network": "IR",
        "latitude": 32.471,
        "longitude": -7.898,
        "channel": "???",
        "isc_distance_deg": 164.95,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:12.02Z",
        "isc_residual_s": "0.9",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DBIC",
        "network": "IR",
        "latitude": 6.6701,
        "longitude": -4.8563,
        "channel": "???",
        "isc_distance_deg": 165.16,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:11.55Z",
        "isc_residual_s": "-0.2",
        "reporter": "",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DBIC",
        "network": "IR",
        "latitude": 6.6701,
        "longitude": -4.8563,
        "channel": "B?Z",
        "isc_distance_deg": 165.16,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:11.66Z",
        "isc_residual_s": "-0.1",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "DBIC",
        "network": "IR",
        "latitude": 6.6701,
        "longitude": -4.8563,
        "channel": "???",
        "isc_distance_deg": 165.16,
        "isc_phase": "PKPdf",
        "reported_phase": "PKP",
        "arrival_utc": "2022-11-12T07:28:11.65Z",
        "isc_residual_s": "-0.1",
        "reporter": "OSUNB",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "LCRM",
        "network": "IR",
        "latitude": 33.686,
        "longitude": -4.858,
        "channel": "???",
        "isc_distance_deg": 165.25,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:13.96Z",
        "isc_residual_s": "2.5",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "JBK",
        "network": "IR",
        "latitude": 34.335,
        "longitude": -2.567,
        "channel": "???",
        "isc_distance_deg": 165.32,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:13.96Z",
        "isc_residual_s": "2.4",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ACRG",
        "network": "IR",
        "latitude": 5.6415,
        "longitude": -0.2072,
        "channel": "B?Z",
        "isc_distance_deg": 165.48,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:11.97Z",
        "isc_residual_s": "-0.1",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OUK",
        "network": "IR",
        "latitude": 31.207,
        "longitude": -7.866,
        "channel": "???",
        "isc_distance_deg": 165.94,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:13.96Z",
        "isc_residual_s": "1.6",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TTIG",
        "network": "IR",
        "latitude": 30.547,
        "longitude": -8.441,
        "channel": "???",
        "isc_distance_deg": 166.1,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:13.97Z",
        "isc_residual_s": "1.8",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "MDT",
        "network": "IR",
        "latitude": 32.814,
        "longitude": -4.607,
        "channel": "???",
        "isc_distance_deg": 166.12,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:13.65Z",
        "isc_residual_s": "1.5",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TISM",
        "network": "IR",
        "latitude": 31.721,
        "longitude": -6.44,
        "channel": "???",
        "isc_distance_deg": 166.28,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:13.99Z",
        "isc_residual_s": "1.9",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TIO",
        "network": "IR",
        "latitude": 30.9267,
        "longitude": -7.2617,
        "channel": "???",
        "isc_distance_deg": 166.49,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:13.84Z",
        "isc_residual_s": "1.3",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TIO",
        "network": "IR",
        "latitude": 30.9267,
        "longitude": -7.2617,
        "channel": "B?Z",
        "isc_distance_deg": 166.49,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:13.48Z",
        "isc_residual_s": "1.0",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "OUZM",
        "network": "IR",
        "latitude": 30.909,
        "longitude": -6.76,
        "channel": "???",
        "isc_distance_deg": 166.77,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:14.07Z",
        "isc_residual_s": "1.4",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "GOLM",
        "network": "IR",
        "latitude": 31.704,
        "longitude": -4.952,
        "channel": "???",
        "isc_distance_deg": 166.97,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:14.96Z",
        "isc_residual_s": "2.1",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TZRR",
        "network": "IR",
        "latitude": 30.785,
        "longitude": -5.569,
        "channel": "???",
        "isc_distance_deg": 167.47,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:14.96Z",
        "isc_residual_s": "1.8",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "FIGM",
        "network": "IR",
        "latitude": 32.133,
        "longitude": -1.262,
        "channel": "???",
        "isc_distance_deg": 167.72,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:14.96Z",
        "isc_residual_s": "1.6",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "ZGR",
        "network": "IR",
        "latitude": 30.119,
        "longitude": -5.62,
        "channel": "???",
        "isc_distance_deg": 167.99,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:14.97Z",
        "isc_residual_s": "1.4",
        "reporter": "CNRM",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TAM",
        "network": "IR",
        "latitude": 22.7917,
        "longitude": 5.5233,
        "channel": "S?Z",
        "isc_distance_deg": 175.62,
        "isc_phase": "PKPdf",
        "reported_phase": "PKIKP",
        "arrival_utc": "2022-11-12T07:28:18.00Z",
        "isc_residual_s": "0.7",
        "reporter": "MOS",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    },
    {
        "station": "TAM",
        "network": "IR",
        "latitude": 22.7917,
        "longitude": 5.5233,
        "channel": "B?Z",
        "isc_distance_deg": 175.62,
        "isc_phase": "PKPdf",
        "reported_phase": "P",
        "arrival_utc": "2022-11-12T07:28:17.75Z",
        "isc_residual_s": "0.4",
        "reporter": "ISC",
        "origin_utc": "2022-11-12T07:09:14.55Z",
        "source_latitude": -20.1146,
        "source_longitude": -178.2086,
        "depth_km": 601.2
    }
]


## 4. Select one report per station and phase

Accept only reports with **|ISC RES| < 1 s**, excluding exactly -1 and +1 s. As discussed in the article, a small catalog residual is not proof of a correct pick: the filter selects observations that already agree with the catalog calculation. This is a selection-sensitivity experiment, not an independent waveform quality assessment.

Reject station coordinate conflicts across either phase. For duplicate reports, choose the actual arrival closest to the median arrival time, then break ties by reporter, network, and channel. Do not select the smallest catalog residual. Normalize ISC `PKPdf` to `PKIKP`; retain agency labels for comparison.

The article reports 402 selected PKIKP picks, 148 PKiKP picks, and 64 stations with both phases. The supported sample is determined separately after predicting every radius.


In [ ]:
def select_catalog_picks(records):
    """Filter catalog reports and choose one arrival per station/phase.
    
    Parameters
    ----------
    records : list of dict
        Embedded ISC reports. Each report includes the station, phase,
        coordinates (degrees), arrival/origin UTC strings, source depth
        (km), and catalog residual (seconds).
    
    Returns
    -------
    source : dict
        Unique fixed origin time, latitude, longitude, and depth.
    picks : list of dict
        Selected reports, sorted by phase then station. Adds pick_id,
        observed_time_s (arrival minus origin), and duplicate diagnostics.
    audit_rows : list of dict
        All reports, with normalized phase, numeric residual, and filter flag.
    conflicts : list of dict
        Passing reports excluded because station coordinates conflict.
    
    Uses global RESIDUAL_LIMIT_S as a strict absolute residual limit.
    Raises ValueError for mixed sources or RuntimeError for no selected picks.
    """
    source_keys = ("origin_utc", "source_latitude", "source_longitude", "depth_km")
    origins = {tuple(record[key] for key in source_keys) for record in records}
    if len(origins) != 1:
        raise ValueError("Mixed ISC origins: inspect embedded records.")
    source = dict(zip(source_keys, next(iter(origins))))
    origin_time = UTCDateTime(source["origin_utc"])

    audit_rows = []
    by_station = defaultdict(list)
    for record in records:
        row = dict(record)
        row["phase"] = "PKIKP" if row["isc_phase"] == "PKPdf" else row["isc_phase"]
        residual = numeric_or_nan(row["isc_residual_s"])
        row["isc_residual_s"] = residual
        row["passes_RES"] = bool(np.isfinite(residual) and abs(residual) < RESIDUAL_LIMIT_S)
        audit_rows.append(row)
        if row["passes_RES"]:
            by_station[row["station"]].append(row)

    picks = []
    conflicts = []
    for station in sorted(by_station):
        reports = by_station[station]
        latitudes = np.array([row["latitude"] for row in reports])
        longitudes = np.array([row["longitude"] for row in reports])
        longitude_offsets = (longitudes - longitudes[0] + 180.0) % 360.0 - 180.0
        if np.ptp(latitudes) > 0.01 or np.ptp(longitude_offsets) > 0.01:
            conflicts.extend(reports)
            continue

        for phase in sorted({row["phase"] for row in reports}):
            phase_reports = [dict(row) for row in reports if row["phase"] == phase]
            arrival_times = np.array([float(UTCDateTime(row["arrival_utc"])) for row in phase_reports])
            median_time = np.median(arrival_times)
            for row, arrival_time in zip(phase_reports, arrival_times):
                row["distance_from_median_s"] = float(abs(arrival_time - median_time))
            chosen = min(phase_reports, key=lambda row: (
                row["distance_from_median_s"], row["reporter"], row["network"], row["channel"]
            ))
            chosen["report_time_span_s"] = float(np.ptp(arrival_times))
            chosen["observed_time_s"] = float(UTCDateTime(chosen["arrival_utc"]) - origin_time)
            chosen["pick_id"] = f"{station}|{phase}"
            picks.append(chosen)

    picks.sort(key=lambda row: (row["phase"], row["station"]))
    if not picks:
        raise RuntimeError("No selected arrivals.")
    return source, picks, audit_rows, conflicts


SOURCE, picks, audit_rows, conflicts = select_catalog_picks(RAW_RECORDS)
write_csv(OUTPUT_DIR / "all_reports_filter_audit.csv", audit_rows)
write_csv(OUTPUT_DIR / "coordinate_conflicts.csv", conflicts, list(audit_rows[0]))
write_csv(OUTPUT_DIR / "selected_catalog_picks.csv", picks)
print("Fixed ISC source:", SOURCE)
print("Reports / passing filter / selected:", len(audit_rows), sum(row["passes_RES"] for row in audit_rows), len(picks))
print("Selected phase counts:", dict(Counter(row["phase"] for row in picks)))
print_table(picks, ["station", "phase", "reporter", "arrival_utc", "isc_residual_s"])


## 5. Continuous inner-core radius models

Keep the mantle, core-mantle boundary (CMB), and center fixed. Remap outer-core depths between the CMB and new inner-core boundary (ICB), and inner-core depths between the new ICB and center. Velocities and densities remain attached to the moving nodes. Both copies of the ICB move together; no depth nodes are deleted.

The article's earlier model builder deleted nodes when the boundary crossed them. This introduced artificial travel-time jumps. The continuous remapping corrects that numerical artifact, but also changes core velocity gradients. It is one explicit model-family assumption, not a unique physical representation of changing radius.


In [ ]:
def radius_model_path(radius_km):
    """Build or reuse an ak135-based TauP model for one radius.
    
    Parameters
    ----------
    radius_km : float
        Trial inner-core radius in kilometers, strictly between zero and
        the outer-core radius (EARTH_RADIUS_KM - CMB_DEPTH_KM).
    
    Returns
    -------
    Path
        Compiled .npz TauP model in MODEL_DIR. Builds it if absent.
    
    Reads AK135_PATH. Remaps core depths while keeping node velocities;
    this changes core velocity gradients as well as the boundary radius.
    """
    radius_km = float(radius_km)
    if not 0 < radius_km < EARTH_RADIUS_KM - CMB_DEPTH_KM:
        raise ValueError("Invalid inner-core radius.")

    original_bytes = AK135_PATH.read_bytes()
    fingerprint = original_bytes + MODEL_METHOD.encode() + obspy.__version__.encode()
    digest = hashlib.sha256(fingerprint).hexdigest()[:12]
    stem = f"continuous_{digest}_{radius_km:.8f}".replace(".", "p")
    model_path = MODEL_DIR / (stem + ".npz")
    if model_path.exists():
        return model_path

    new_icb_depth = EARTH_RADIUS_KM - radius_km
    lines = original_bytes.decode().splitlines()
    output_lines = lines[:2]
    for line in lines[2:]:
        values = [float(value) for value in line.split()]
        depth = values[0]
        if CMB_DEPTH_KM < depth <= REFERENCE_ICB_DEPTH_KM:
            fraction = (depth - CMB_DEPTH_KM) / (REFERENCE_ICB_DEPTH_KM - CMB_DEPTH_KM)
            depth = CMB_DEPTH_KM + fraction * (new_icb_depth - CMB_DEPTH_KM)
        elif depth > REFERENCE_ICB_DEPTH_KM:
            fraction = (depth - REFERENCE_ICB_DEPTH_KM) / (EARTH_RADIUS_KM - REFERENCE_ICB_DEPTH_KM)
            depth = new_icb_depth + fraction * (EARTH_RADIUS_KM - new_icb_depth)
        values[0] = depth
        output_lines.append(" ".join(f"{value:.10f}" for value in values))

    velocity_path = MODEL_DIR / (stem + ".tvel")
    velocity_path.write_text("\n".join(output_lines) + "\n")
    build_taup_model(str(velocity_path), output_folder=str(MODEL_DIR), verbose=False)
    if not model_path.exists():
        raise RuntimeError("TauP model build failed.")
    return model_path


## 6. Continuity and reference-model checks

The article linked three artificial derivative spikes to node crossings at radii **1115.68, 1166.39, and 1267.43 km**. Compare slopes immediately before, across, and after each crossing. Smooth behavior must come from the model construction, not smoothing the plotted curves.

At 1217.5 km, compare the rebuilt model with unmodified ak135 at several distances for both phases. Agreement checks the reference construction; it does not establish that the model describes Earth's actual core.


In [ ]:
continuity_rows = []
for center_radius in (1115.68, 1166.39, 1267.43):
    offsets = np.array([-0.1, -0.01, 0.01, 0.1])
    phase_times = []
    for offset in offsets:
        model = TauPyModel(str(radius_model_path(center_radius + offset)))
        arrivals = model.get_travel_times(SOURCE["depth_km"], 140.0, phase_list=list(PHASES))
        phase_times.append([min(arrival.time for arrival in arrivals if arrival.name == phase) for phase in PHASES])
    slopes = np.diff(np.asarray(phase_times), axis=0) / np.diff(offsets)[:, None]
    for column, phase in enumerate(PHASES):
        continuity_rows.append({
            "radius_km": center_radius, "phase": phase,
            "left_slope": slopes[0, column], "crossing_slope": slopes[1, column],
            "right_slope": slopes[2, column],
        })
    print(f"Checked crossing at {center_radius} km", flush=True)
write_csv(OUTPUT_DIR / "continuity_check.csv", continuity_rows)
print_table(continuity_rows)

reference_model = TauPyModel("ak135")
rebuilt_model = TauPyModel(str(radius_model_path(REFERENCE_RADIUS_KM)))
for distance in (100.0, 140.0, 155.0, 175.0):
    for phase in PHASES:
        original_arrivals = reference_model.get_travel_times(SOURCE["depth_km"], distance, phase_list=[phase])
        rebuilt_arrivals = rebuilt_model.get_travel_times(SOURCE["depth_km"], distance, phase_list=[phase])
        assert len(original_arrivals) == len(rebuilt_arrivals), (distance, phase, "branch count changed")
        assert np.allclose(
            [arrival.time for arrival in original_arrivals],
            [arrival.time for arrival in rebuilt_arrivals], rtol=0, atol=1e-5,
        ), (distance, phase, "reference mismatch")
print("Reference model check passed.")


## 7. Predicted arrivals and geometric corrections

Distances use WGS84 surface distance divided by 6371 km. Apply ellipticity correction to the earliest exact-name branch and record all matching branch counts. No station-elevation correction is applied.

The article's residuals are observed minus these calculated times. They need not equal ISC RES: a pick can pass the catalog's one-second filter and have a larger residual in this calculation. Saved predictions are associated with the source, selected observations, radius grid, model definition, and relevant package versions.


In [ ]:
geod = Geod(ellps="WGS84")
for pick in picks:
    azimuth, _, distance_m = geod.inv(
        SOURCE["source_longitude"], SOURCE["source_latitude"],
        pick["longitude"], pick["latitude"],
    )
    pick["distance_deg"] = distance_m / (EARTH_RADIUS_KM * 1000.0) * 180.0 / np.pi
    pick["azimuth"] = azimuth % 360.0

specification = {
    "source": SOURCE, "picks": picks, "radii": RADII_KM.tolist(),
    "limit": RESIDUAL_LIMIT_S, "obspy": obspy.__version__,
    "ellipticipy": version("ellipticipy"), "method": MODEL_METHOD,
    "model_source_sha256": hashlib.sha256(AK135_PATH.read_bytes()).hexdigest(),
}
run_key = hashlib.sha256(json.dumps(specification, sort_keys=True).encode()).hexdigest()[:16]
RUN_DIR = OUTPUT_DIR / run_key
RUN_DIR.mkdir(exist_ok=True)
(RUN_DIR / "inputs.json").write_text(json.dumps(specification, indent=2))
CACHE_PATH = RUN_DIR / "predictions.json"


def predict_one_radius(radius_km):
    """Predict one arrival per selected station/phase at one radius.
    
    Parameters
    ----------
    radius_km : float
        Trial inner-core radius in kilometers.
    
    Returns
    -------
    list of dict
        One record per global picks entry: pick_id, branches (count),
        predicted_time_s (seconds after origin, or None if unavailable),
        and ellipticity_s when an arrival exists.
    
    Uses global SOURCE (depth and latitude) and picks (phase, angular
        distance in degrees, and azimuth in degrees). Chooses the earliest
        exact-name branch and applies ellipticity correction.
    """
    model = TauPyModel(str(radius_model_path(radius_km)))
    predictions = []
    for number, pick in enumerate(picks, start=1):
        arrivals = model.get_ray_paths(SOURCE["depth_km"], pick["distance_deg"], phase_list=[pick["phase"]])
        arrivals = [arrival for arrival in arrivals if arrival.name == pick["phase"]]
        row = {"pick_id": pick["pick_id"], "branches": len(arrivals), "predicted_time_s": None}
        if arrivals:
            arrival = min(arrivals, key=lambda item: item.time)
            correction = float(ellipticity_correction(
                arrival, azimuth=pick["azimuth"], source_latitude=SOURCE["source_latitude"],
            ))
            row["predicted_time_s"] = float(arrival.time + correction)
            row["ellipticity_s"] = correction
        predictions.append(row)
        if number % 50 == 0:
            print(f"  {number}/{len(picks)} arrivals", flush=True)
    return predictions


saved_predictions = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
started = time.monotonic()
for number, radius in enumerate(RADII_KM, start=1):
    radius_key = f"{radius:.3f}"
    if radius_key not in saved_predictions:
        print(f"Starting radius {number}/{len(RADII_KM)}: {radius:g} km", flush=True)
        saved_predictions[radius_key] = predict_one_radius(radius)
        temporary_path = CACHE_PATH.with_suffix(".tmp")
        temporary_path.write_text(json.dumps(saved_predictions))
        temporary_path.replace(CACHE_PATH)
    elapsed_minutes = (time.monotonic() - started) / 60.0
    print(f"Completed {number}/{len(RADII_KM)}; elapsed {elapsed_minutes:.1f} min", flush=True)


## 8. Residuals and support across the radius grid

For phase $\phi$ at station $i$, define

$$e_{i,\phi}(r) = a_{i,\phi} - t_0 - T_{i,\phi}(r),$$

where $a$ is the catalog arrival time, $t_0$ the fixed origin time, and $T(r)$ the predicted travel time, including ellipticity correction. Positive residuals mean the observation is later than predicted.

Use only observations with finite predictions at **every tested radius** in each fit. Otherwise a changing sample could produce a misleading minimum. Retain excluded observations in the audit output. The article's completed run retained 392 PKIKP picks, all 148 PKiKP picks, and all 64 pairs.


In [ ]:
def prediction_arrays(picks, saved, radii):
    """Align cached arrivals with the selected-pick array columns.
    
    Parameters
    ----------
    picks : list of dict
        Selected observations with unique pick_id and observed_time_s.
    saved : dict
        Prediction cache keyed by radius formatted to three decimals.
        Each value is the complete list from predict_one_radius.
    radii : 1-D numpy.ndarray
        Trial radii in kilometers, in desired output row order.
    
    Returns
    -------
    times : 2-D numpy.ndarray
        Predicted travel times (seconds); missing arrivals are NaN.
    branches : 2-D numpy.ndarray of int
        Number of exact-name branches for each radius and pick.
    residuals : 2-D numpy.ndarray
        Observed minus predicted time, in seconds.
    rows : list of dict
        Combined observation/prediction records for CSV output.
    pick_index : dict
        Maps pick_id to its array column index.
    
    All three arrays have shape (len(radii), len(picks)). Cache entries
    must contain every selected pick exactly once per radius.
    """
    pick_index = {pick["pick_id"]: column for column, pick in enumerate(picks)}
    if len(pick_index) != len(picks):
        raise ValueError("Duplicate selected pick IDs.")
    times = np.full((len(radii), len(picks)), np.nan)
    branches = np.zeros(times.shape, dtype=int)
    rows = []
    for radius_index, radius in enumerate(radii):
        seen = set()
        for prediction in saved[f"{radius:.3f}"]:
            pick_id = prediction["pick_id"]
            if pick_id not in pick_index or pick_id in seen:
                raise ValueError(f"Unexpected or duplicate cache pick: {pick_id}")
            seen.add(pick_id)
            column = pick_index[pick_id]
            predicted_time = numeric_or_nan(prediction["predicted_time_s"])
            times[radius_index, column] = predicted_time
            branches[radius_index, column] = prediction["branches"]
            row = {"radius_km": float(radius), **prediction, **picks[column]}
            row["predicted_time_s"] = predicted_time
            row["residual_s"] = picks[column]["observed_time_s"] - predicted_time
            rows.append(row)
        if seen != set(pick_index):
            raise ValueError(f"Incomplete cache at radius {radius:g} km.")
    observed = np.array([pick["observed_time_s"] for pick in picks])
    residuals = observed[None, :] - times
    return times, branches, residuals, rows, pick_index


predicted_times, branch_counts, residuals, prediction_rows, pick_index = prediction_arrays(
    picks, saved_predictions, RADII_KM,
)
write_csv(RUN_DIR / "all_radius_station_residuals.csv", prediction_rows)
common_support = np.isfinite(residuals).all(axis=0)
excluded_picks = [pick for column, pick in enumerate(picks) if not common_support[column]]
write_csv(RUN_DIR / "unsupported_across_grid.csv", excluded_picks, list(picks[0]))
print("Excluded for missing predictions at some radius:", len(excluded_picks))


## 9. Compare absolute and paired radius estimates

For each curve, minimize the equal-weight root mean square residual:

$$\mathrm{RMS}(r) = \sqrt{\frac{1}{N}\sum_{i=1}^{N} e_i(r)^2}.$$

The combined absolute curve weights every pick equally, so the more numerous phase contributes more. Same-station differential residuals cancel origin time exactly. A boundary minimum is unresolved within the tested interval.

**Comparison values reported in the article's October 6 run** (not hardcoded calculation results):

| Fit | Observations | Best radius (km) | Minimum RMS (s) |
| --- | ---: | ---: | ---: |
| PKIKP absolute | 392 | 1214 | 0.525 |
| PKiKP absolute | 148 | 1214 | 0.563 |
| Combined absolute | 540 | 1214 | 0.535 |
| Same-station phase differences | 64 pairs | 1226 | 0.560 |

Use these as a reproducibility comparison after running the notebook. A discrepancy should prompt inspection of observations, phase support, model construction, and package versions. The NumPy rewrite has not yet been validated with a complete TauP rerun.

The article's earlier model favored 1214 km using 52 pairs. Both the model and supported sample changed; the shift to 1226 km cannot be assigned to either factor alone without a matched 52-pair comparison. These fits share observations and filtering, so they are not independent measurements.


In [ ]:
def radius_profiles(residuals, picks, common_support, radii):
    """Calculate equal-weight RMS curves and their grid minima.
    
    Parameters
    ----------
    residuals : 2-D numpy.ndarray, shape (len(radii), len(picks))
        Observed minus predicted arrival times in seconds.
    picks : list of dict
        Observation metadata in the residual array's column order.
    common_support : 1-D boolean numpy.ndarray, length len(picks)
        True for observations supported at every trial radius.
    radii : 1-D numpy.ndarray
        Trial radii in kilometers, in the residual array's row order.
    
    Returns
    -------
    curves : dict of 1-D numpy.ndarray
        RMS in seconds at every radius, keyed by fit name.
    summaries : list of dict
        Observation count, best grid radius (km), RMS (s), boundary flag.
    paired_stations : list of str
        Alphabetically ordered stations with both supported phases.
    paired_residuals : 2-D numpy.ndarray
        PKiKP residual minus PKIKP residual, in seconds; rows are radii,
        columns follow paired_stations.
    
    A curve with zero observations is omitted. A boundary minimum is
    unresolved within the grid; this function does not estimate uncertainty.
    """
    curves = {}
    summaries = []

    def add_curve(name, values):
        """Add one RMS profile to the enclosing result collections.
        
        name : str
            Descriptive fit name, used as the curve key and CSV label.
        values : 2-D numpy.ndarray
            Residuals in seconds, with rows matching the enclosing radii
            and columns representing equally weighted observations.
        Returns None. Empty-column arrays are skipped.
        """
        if values.shape[1] == 0:
            print(f"{name}: no common-support observations; no estimate.")
            return
        rms = np.sqrt(np.mean(values ** 2, axis=1))
        best_index = int(np.argmin(rms))
        curves[name] = rms
        summaries.append({
            "case": name, "count": values.shape[1],
            "best_radius_km": float(radii[best_index]), "rms_s": float(rms[best_index]),
            "boundary": best_index in (0, len(radii) - 1),
        })

    # Use a fixed alphabetical observation order for reproducible summation.
    columns = sorted(np.flatnonzero(common_support), key=lambda column: picks[column]["pick_id"])
    for phase in PHASES:
        phase_columns = [column for column in columns if picks[column]["phase"] == phase]
        add_curve(phase, residuals[:, phase_columns])
    add_curve("Combined_absolute", residuals[:, columns])

    lookup = {picks[column]["pick_id"]: column for column in columns}
    paired_stations = sorted({
        pick["station"] for pick in picks
        if pick["station"] + "|PKIKP" in lookup and pick["station"] + "|PKiKP" in lookup
    })
    paired_residuals = np.empty((len(radii), len(paired_stations)))
    for column, station in enumerate(paired_stations):
        paired_residuals[:, column] = (
            residuals[:, lookup[station + "|PKiKP"]] - residuals[:, lookup[station + "|PKIKP"]]
        )
    add_curve("Paired_differential", paired_residuals)
    return curves, summaries, paired_stations, paired_residuals


curves, radius_summary, paired_stations, paired_residuals = radius_profiles(
    residuals, picks, common_support, RADII_KM,
)
write_matrix(RUN_DIR / "same_station_differential_residuals.csv", paired_residuals, paired_stations)
profile_rows = [{"radius_km": float(radius), **{name: values[i] for name, values in curves.items()}}
                for i, radius in enumerate(RADII_KM)]
write_csv(RUN_DIR / "radius_profiles.csv", profile_rows)
write_csv(RUN_DIR / "radius_summary.csv", radius_summary,
          ["case", "count", "best_radius_km", "rms_s", "boundary"])
print_table(radius_summary)
if any(row["boundary"] for row in radius_summary):
    print("WARNING: a boundary minimum is unresolved within this radius grid.")
if np.any(branch_counts > 1):
    print("WARNING: multiple branches present; inspect all_radius_station_residuals.csv.")

supported_ids = {pick["pick_id"] for column, pick in enumerate(picks) if common_support[column]}
for result in radius_summary:
    rows = [row for row in prediction_rows
            if row["radius_km"] == result["best_radius_km"] and row["pick_id"] in supported_ids]
    if result["case"] in PHASES:
        rows = [row for row in rows if row["phase"] == result["case"]]
    write_csv(RUN_DIR / (result["case"] + "_best_station_residuals.csv"), rows)
reference_rows = [row for row in prediction_rows if row["radius_km"] == REFERENCE_RADIUS_KM]
write_csv(RUN_DIR / "ak135_station_residuals.csv", reference_rows)
print_table(reference_rows, ["station", "phase", "isc_residual_s", "residual_s"])


## 10. Article Figure 1: RMS curves and reference residuals

The upper panel compares four radius fits. A lower RMS means closer agreement under the chosen assumptions, not necessarily a more accurate radius. Inspect the width of each minimum as well as its location.

The lower panel shows residuals relative to ak135 for all selected picks, including observations excluded from common support across the full radius grid. The article used 550 selected picks for this panel and 540 for the combined absolute fit.


In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(14, 11), layout="constrained")
for name, rms in curves.items():
    axes[0].plot(RADII_KM, rms, label=name)
axes[0].set(xlabel="Inner-core radius (km)", ylabel="RMS (s)",
            title=f"Fixed ISC source; strict |ISC RES| < {RESIDUAL_LIMIT_S:g} s")
axes[0].axvline(REFERENCE_RADIUS_KM, color="gray", linestyle=":", label="ak135 1217.5 km")
axes[0].legend()
axes[0].grid(alpha=0.3)
for phase in PHASES:
    rows = [row for row in reference_rows if row["phase"] == phase]
    axes[1].scatter([row["distance_deg"] for row in rows], [row["residual_s"] for row in rows],
                    label=phase, alpha=0.65)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(xlabel="Distance (degrees)", ylabel="Observed - ak135 predicted time (s)",
            title="Our ak135 residuals using catalog origin")
axes[1].legend()
axes[1].grid(alpha=0.3)
figure.savefig(RUN_DIR / "radius_profiles_and_reference_residuals.png", dpi=170)
plt.show()


## 11. Radius sensitivity and branch checks

The local sensitivity is $g_i(r)=dT_i/dr$, measured in seconds per kilometer. For a small radius change, $\delta T_i \approx g_i\,\delta r$.

Use three-point finite differences on the actual nonuniform radius grid, with one-sided differences at the endpoints. Mask derivatives if any stencil point has multiple branches or missing times. No smoothing is applied. This mask cannot detect every branch switch.

The differential derivative follows the article's **PKIKP minus PKiKP** convention. The radius-search differential residual above uses the opposite order. Reversing the sign changes neither RMS nor the best-fitting radius.


In [ ]:
def radius_derivative(times, branches, radii):
    """Differentiate travel times with respect to inner-core radius.
    
    Parameters
    ----------
    times : 2-D numpy.ndarray, shape (len(radii), number_of_stations)
        Travel times or paired travel-time differences, in seconds.
    branches : 2-D numpy.ndarray, same shape as times
        Branch counts for a single phase; for a paired difference use
        1 only where BOTH phases have exactly one branch, otherwise 0.
    radii : 1-D numpy.ndarray
        Strictly increasing radii in km; at least three points required.
        Spacing may be nonuniform, including the extra 1217.5 km point.
    
    Returns
    -------
    raw : 2-D numpy.ndarray
        Three-point numerical derivative in seconds/km.
    masked : 2-D numpy.ndarray
        Same derivative, with NaN wherever any point in its stencil has
        non-finite time or a branch value other than 1.
    
    Endpoints use one-sided stencils. Interior points use their two
    neighbors. No smoothing or complete branch-tracking is performed.
    """
    raw = np.gradient(times, radii, axis=0, edge_order=2)
    safe = np.zeros(times.shape, dtype=bool)
    for row in range(len(radii)):
        if row == 0:
            stencil = [0, 1, 2]
        elif row == len(radii) - 1:
            stencil = [len(radii) - 3, len(radii) - 2, len(radii) - 1]
        else:
            stencil = [row - 1, row, row + 1]
        safe[row] = np.all(branches[stencil] == 1, axis=0) & np.all(np.isfinite(times[stencil]), axis=0)
    return raw, np.where(safe, raw, np.nan)


phase_columns = {}
for phase in PHASES:
    phase_columns[phase] = sorted(
        [column for column, pick in enumerate(picks) if pick["phase"] == phase and common_support[column]],
        key=lambda column: picks[column]["station"],
    )
write_csv(RUN_DIR / "excluded_missing_predictions.csv",
          [{"phase": pick["phase"], "station": pick["station"]} for pick in excluded_picks],
          ["phase", "station"])

pkikp_columns = phase_columns["PKIKP"]
raw_pkikp, clean_pkikp = radius_derivative(
    predicted_times[:, pkikp_columns], branch_counts[:, pkikp_columns], RADII_KM,
)
transmitted_columns = [pick_index[station + "|PKIKP"] for station in paired_stations]
reflected_columns = [pick_index[station + "|PKiKP"] for station in paired_stations]
time_difference = predicted_times[:, transmitted_columns] - predicted_times[:, reflected_columns]
both_single = ((branch_counts[:, transmitted_columns] == 1) & (branch_counts[:, reflected_columns] == 1)).astype(int)
raw_difference, clean_difference = radius_derivative(time_difference, both_single, RADII_KM)

derivatives = {
    "PKIKP": {"stations": [picks[column]["station"] for column in pkikp_columns],
              "raw": raw_pkikp, "clean": clean_pkikp, "label": r"$dT_{PKIKP}/dr$"},
    "PKIKP_minus_PKiKP": {"stations": paired_stations, "raw": raw_difference,
                          "clean": clean_difference, "label": r"$d(T_{PKIKP}-T_{PKiKP})/dr$"},
}
for name, result in derivatives.items():
    write_matrix(RUN_DIR / (name + "_derivative_raw_s_per_km.csv"), result["raw"], result["stations"])
    write_matrix(RUN_DIR / (name + "_derivative_masked_s_per_km.csv"), result["clean"], result["stations"])
    print(name, "stations:", len(result["stations"]), "masked values:", np.isnan(result["clean"]).sum())


## 12. Article Figures 2 and 3: sensitivity along different paths

**Figure 2:** Individual derivative curves, station medians, and the middle 80% of station values. Shading describes variation among paths, not uncertainty in radius. The article reports smooth corrected curves without the earlier three artificial spikes.

**Figure 3:** Derivatives with stations ordered by distance. Each row is a station, not an equal distance interval. The panels use separate color scales. PKIKP-only paths and paired paths are different samples; their median sensitivities are not a comparison of identical paths.

The article reports PKIKP slopes from -0.089563 to -0.035057 s/km at 1217.5 km, and paired slopes from -0.014901 to -0.000948 s/km. Pairs with slopes close to zero carry little radius information for a given timing error. Many stations with similar paths do not necessarily provide many independent constraints.


In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(15, 12), sharex=True, layout="constrained")
for axis, result in zip(axes, derivatives.values()):
    values = result["clean"]
    axis.plot(RADII_KM, values, color="steelblue", alpha=0.14, linewidth=0.8)
    quantiles = np.full((3, len(RADII_KM)), np.nan)
    for row in range(len(RADII_KM)):
        finite_values = values[row, np.isfinite(values[row])]
        if finite_values.size:
            quantiles[:, row] = np.quantile(finite_values, [0.1, 0.5, 0.9])
    if values.shape[1]:
        axis.fill_between(RADII_KM, quantiles[0], quantiles[2], color="orange", alpha=0.2,
                          label="10-90% station range")
        axis.plot(RADII_KM, quantiles[1], color="darkorange", linewidth=2, label="Station median")
    axis.axhline(0, color="black", linewidth=0.7)
    axis.axvline(REFERENCE_RADIUS_KM, color="gray", linestyle=":", label="ak135 radius")
    axis.set(title=result["label"] + " - individual stations", ylabel="Sensitivity (s/km)")
    axis.grid(alpha=0.25)
    axis.legend()
axes[-1].set_xlabel("Inner-core radius (km)")
figure.savefig(RUN_DIR / "derivative_curves.png", dpi=180)
plt.show()

station_distances = {pick["station"]: pick["distance_deg"] for pick in picks if pick["phase"] == "PKIKP"}
figure, axes = plt.subplots(2, 1, figsize=(15, 12), layout="constrained")
for axis, result in zip(axes, derivatives.values()):
    stations = result["stations"]
    order = sorted(range(len(stations)), key=lambda column: station_distances[stations[column]])
    if not order or not np.isfinite(result["clean"]).any():
        axis.text(0.5, 0.5, "No unambiguous derivatives available", ha="center", transform=axis.transAxes)
        continue
    values = result["clean"][:, order].T
    scale = float(np.nanmax(np.abs(values))) or 1.0
    mesh = axis.pcolormesh(RADII_KM, np.arange(len(order)), np.ma.masked_invalid(values),
                          shading="nearest", cmap="RdBu_r", vmin=-scale, vmax=scale)
    ticks = np.unique(np.linspace(0, len(order) - 1, min(12, len(order))).astype(int))
    tick_labels = [f"{stations[order[i]]} ({station_distances[stations[order[i]]]:.1f}°)" for i in ticks]
    axis.set_yticks(ticks, tick_labels)
    axis.set(title=result["label"], xlabel="Inner-core radius (km)", ylabel="Stations ordered by distance")
    axis.axvline(REFERENCE_RADIUS_KM, color="black", linestyle=":", linewidth=1)
    figure.colorbar(mesh, ax=axis, label="s/km")
figure.savefig(RUN_DIR / "derivative_heatmaps.png", dpi=180)
plt.show()


## 13. Sensitivity near the ak135 reference radius

The article reports median slopes of **-0.047651 s/km** for PKIKP and **-0.011607 s/km** for PKIKP minus PKiKP at 1217.5 km. A local +10 km radius change corresponds to approximately -0.477 s and -0.116 s, respectively.

At the reported median paired sensitivity, a 0.5 s error in one phase-time difference corresponds locally to about $0.5/0.011607 \approx 43$ km. This is a single-pair illustration, not an uncertainty estimate for the 64-pair fit. Shared model errors and correlated paths do not average away like independent random errors.


In [ ]:
reference_index = int(np.argmin(np.abs(RADII_KM - REFERENCE_RADIUS_KM)))
sensitivity_summary = []
for name, result in derivatives.items():
    values = result["clean"][reference_index]
    rows = []
    for column, station in enumerate(result["stations"]):
        rows.append({
            "station": station, "derivative_s_per_km": values[column],
            "raw_derivative_s_per_km": result["raw"][reference_index, column],
            "distance_deg": station_distances[station],
            "linear_change_for_plus10km_s": 10.0 * values[column],
        })
    write_csv(RUN_DIR / (name + "_sensitivity_at_reference.csv"), rows,
              ["station", "derivative_s_per_km", "raw_derivative_s_per_km", "distance_deg", "linear_change_for_plus10km_s"])
    finite_values = values[np.isfinite(values)]
    sensitivity_summary.append({
        "quantity": name, "radius_km": RADII_KM[reference_index],
        "unmasked_stations": len(finite_values),
        "minimum_s_per_km": np.min(finite_values) if finite_values.size else np.nan,
        "median_s_per_km": np.median(finite_values) if finite_values.size else np.nan,
        "maximum_s_per_km": np.max(finite_values) if finite_values.size else np.nan,
    })
print_table(sensitivity_summary)
print(f"DONE. Plots, CSV files, and prediction cache: {RUN_DIR}")


## Interpretation

### Origin time and radius can trade off

For two stations, differencing leaves sensitivity $g_i-g_j$. If their slopes are similar, removing origin time also removes much of the radius signal. With an equal-weight fitted origin-time shift, the retained sensitivities are $g_i-\bar g$. A nearly constant slope at one station is not the problem; similar slopes across stations are what make the two effects difficult to separate.

This notebook's same-station phase differences have sensitivity $g_{i,PKIKP}-g_{i,PKiKP}$. They remove origin time, but can still respond weakly to radius. The article's earlier interstation experiment used a different source and distance range; these plots do not quantify that experiment's sensitivity directly.

### What the Fiji result establishes

The scan provides conditional best-fitting radii for one fixed source, one catalog selection, and one continuous velocity-model family. It does not establish a precise physical inner-core radius or temporal core change. The radius grid, station ranges, and curve minima are not confidence intervals.

### Related research cited in the article

1. Engdahl, Flinn, and Massé (1974), *Differential PKiKP Travel Times and the Radius of the Inner Core*. https://doi.org/10.1111/j.1365-246X.1974.tb05467.x. Uses PKiKP minus PcP, a different phase pair.
2. Ivan, Wang, and Hofstetter (2018), *Non quasi-Hemispherical Seismological Pattern of the Earth's Uppermost Inner Core*. https://doi.org/10.1038/s41598-018-20657-x. Uses waveform comparisons for PKIKP/PKiKP; this catalog-pick test does not repeat that method.
3. Kennett, Engdahl, and Buland (1995), *Constraints on seismic velocities in the Earth from traveltimes*. https://doi.org/10.1111/j.1365-246X.1995.tb03540.x. Reference for ak135 and velocity assumptions.
4. Crotwell, Owens, and Ritsema (1999), *The TauP Toolkit: Flexible Seismic Travel-Time and Raypath Utilities*. https://doi.org/10.1785/gssrl.70.2.154.
5. Russell, Rudge, Irving, and Cottaar (2022), *A re-examination of ellipticity corrections for seismic phases*. https://doi.org/10.1093/gji/ggac315.

References and comparison values above are reproduced from the accompanying article; they are context for this experiment, not independent validation of its fitted radii.
